**locate gRNA_*_gene features** (13:11)

In [ ]:
a=pt.screen().adata
print(a)
import numpy as np
print(a.obs.columns.tolist()); print(a.var.columns.tolist())
g=[v for v in a.var_names if v.endswith('_gene') and v.startswith('gRNA')]
print(len(g))
sub=a[:,g].X.sum(axis=1).A1 if hasattr(a.X,'sum') else None
import pandas as pd
print(pd.Series(sub>0).groupby(a.obs['sample'].values if 'sample' in a.obs else a.obs.iloc[:,0].values).mean())
print(a.var.loc[g[:3]])
print(pd.Series(a.var_names[a.var.feature_types.astype(str).str.contains('CRISPR')][:5]))

---------------------------------------------------------------------------
ValueError                                Traceback (most recent call last)
Cell In[2], line 11
      7 try:
      8     pt.save_figure = save_figure
      9 except NameError:
     10     pass
---> 11 a=pt.screen().adata
     12 print(a)
     13 import numpy as np
     14 print(a.obs.columns.tolist()); print(a.var.columns.tolist())

File /opt/pertura/src/pertura/blocks.py:180, in screen()
    178 build = current_build(key)
    179 if build is None:  # before the build the notebook serves the design (development session 9, problem 12)
--> 180     raise ValueError("no screen object yet: it is built once the design is confirmed (build_screen_object). "
    181                      "Until then the deposit's files are at `deposit`, and a deposit Pertura cannot read goes into "
    182                      "the cell tables at `cell_tables`")
    183 if key in _SCREEN and _BUILD.setdefault(key, build) != build:  # a r

**peek cell tables layout** (13:11)

In [ ]:
import os
print(cell_tables)
for r,d,f in os.walk(str(cell_tables)):
    print(r, f[:6]); 
    if r.count(os.sep)-str(cell_tables).count(os.sep)>1: break

/data/pertura/workspace/screen_GSE267740-52e74a8e/design/cell_tables
/data/pertura/workspace/screen_GSE267740-52e74a8e/design/cell_tables ['NTN_1_19_9_1.h5ad', 'NTN_2_23_1_Th17-3_protospacer_calls_per_cell.csv', 'NTN_1_19_23_1.h5ad', 'NTN_2_23_1_Th17-3.h5ad', 'NTN_1_19_5.h5ad', 'Colitis_2_23_2_Th17-2.h5ad']

**inspect cell tables for guide features** (13:11)

In [ ]:
import anndata as ad, pandas as pd, numpy as np
for s in ["NTN_1_19_5","NTN_2_23_1_Th17-1","NTN_2_23_1_Control"]:
    a=ad.read_h5ad(f"{cell_tables}/{s}.h5ad")
    print(s,a.shape, a.var.columns.tolist(), a.obs.columns.tolist()[:10])
    g=[v for v in a.var_names if 'gRNA' in str(v)]
    print(len(g), g[:3], a.var.feature_types.value_counts().to_dict() if 'feature_types' in a.var else '')
    if g:
        x=a[:,g].X; print((x.sum(axis=1)>0).mean() if hasattr(x,'sum') else '')
c=pd.read_csv(f"{cell_tables}/NTN_2_23_1_Th17-3_protospacer_calls_per_cell.csv"); print(c.head(), c.num_features.value_counts().head())

NTN_1_19_5 (12495, 28863) ['gene_ids', 'name', 'feature_types'] []
160 ['gRNA_TLR4_1_gene', 'gRNA_TLR4_2_gene', 'gRNA_TLR4_3_gene'] {'Gene Expression': 28863}
0.3666266506602641
NTN_2_23_1_Th17-1 (19511, 32729) ['gene_ids', 'name', 'feature_types'] []
320 ['gRNA_TLR4_1_gene', 'gRNA_TLR4_2_gene', 'gRNA_TLR4_3_gene'] {'Gene Expression': 32445, 'CRISPR Guide Capture': 160, 'Antibody Capture': 124}
0.9996924811644713
NTN_2_23_1_Control (25658, 32729) ['gene_ids', 'name', 'feature_types'] []
320 ['gRNA_TLR4_1_gene', 'gRNA_TLR4_2_gene', 'gRNA_TLR4_3_gene'] {'Gene Expression': 32445, 'CRISPR Guide Capture': 160, 'Antibody Capture': 124}
0.0009743549770052226
         cell_barcode  num_features  \
0  AAACCTGAGATCACGG-1             5   
1  AAACCTGAGCGTGAGT-1             4   
2  AAACCTGAGCGTTCCG-1             1   
3  AAACCTGAGGTGACCA-1             1   
4  AAACCTGAGTACGATA-1             1   

                                        feature_call  num_umis  
0  gRNA_Ahr_3_capture|gRNA_Ahr_5_capture

**retype guide transcripts in expression** (13:12)

In [ ]:
from pertura.cell_table import retype
import anndata as ad, re
cap=[v for v in ad.read_h5ad(f"{cell_tables}/NTN_2_23_1_Th17-1.h5ad").var_names if v.endswith('_capture')]
low={c.lower():c for c in cap}
special={'cd40l':'cd40lg','sykb':'syk','tgfrb2':'tgfbr2'}
def m(n):
    b=n[len('gRNA_'):-len('_gene')]; g,i=b.rsplit('_',1)
    g=g.lower().replace('-','')
    g=special.get(g,g)
    return low.get(f"grna_{g}_{i}_capture")
samples=["NTN_1_19_5","NTN_1_19_9_1","NTN_1_19_9_2","NTN_1_19_23_1","NTN_1_19_23_2"]
for s in samples:
    a=ad.read_h5ad(f"{cell_tables}/{s}.h5ad")
    mp={n:m(n) for n in a.var_names if n.startswith('gRNA') and n.endswith('_gene')}
    bad=[k for k,v in mp.items() if v is None]
    print(s,len(mp),bad, len(set(mp.values())))
    if not bad: print(retype(cell_tables,s,mp))
for s in ["NTN_2_23_1_Control","NTN_2_23_1_Th17-1","NTN_2_23_1_Th17-2","NTN_2_23_1_Th17-3","Colitis_2_23_2_Control","Colitis_2_23_2_Th17-1","Colitis_2_23_2_Th17-2"]:
    a=ad.read_h5ad(f"{cell_tables}/{s}.h5ad")
    gl=[n for n in a.var_names[(a.var.feature_types=='Gene Expression').values] if n.startswith('gRNA') and n.endswith('_gene')]
    print(s,len(gl))
    print(retype(cell_tables,s,{n:m(n) for n in gl},feature_type="Guide read in the expression"))

NTN_1_19_5 160 [] 160
{'ok': True, 'problems': [], 'sample': 'NTN_1_19_5', 'retyped': 160}
NTN_1_19_9_1 160 [] 160
{'ok': True, 'problems': [], 'sample': 'NTN_1_19_9_1', 'retyped': 160}
NTN_1_19_9_2 160 [] 160
{'ok': True, 'problems': [], 'sample': 'NTN_1_19_9_2', 'retyped': 160}
NTN_1_19_23_1 160 [] 160
{'ok': True, 'problems': [], 'sample': 'NTN_1_19_23_1', 'retyped': 160}
NTN_1_19_23_2 160 [] 160
{'ok': True, 'problems': [], 'sample': 'NTN_1_19_23_2', 'retyped': 160}
NTN_2_23_1_Control 160
{'ok': True, 'problems': [], 'sample': 'NTN_2_23_1_Control', 'retyped': 160}
NTN_2_23_1_Th17-1 160
{'ok': True, 'problems': [], 'sample': 'NTN_2_23_1_Th17-1', 'retyped': 160}
NTN_2_23_1_Th17-2 160
{'ok': True, 'problems': [], 'sample': 'NTN_2_23_1_Th17-2', 'retyped': 160}
NTN_2_23_1_Th17-3 160
{'ok': True, 'problems': [], 'sample': 'NTN_2_23_1_Th17-3', 'retyped': 160}
Colitis_2_23_2_Control 160
{'ok': True, 'problems': [], 'sample': 'Colitis_2_23_2_Control', 'retyped': 160}
Colitis_2_23_2_Th17-1 1

**annotate mouse unit and disease condition columns** (13:18)

In [ ]:
from pertura.cell_table import annotate
import anndata as ad
S=["NTN_1_19_5","NTN_1_19_9_1","NTN_1_19_9_2","NTN_1_19_23_1","NTN_1_19_23_2","NTN_2_23_1_Control","NTN_2_23_1_Th17-1","NTN_2_23_1_Th17-2","NTN_2_23_1_Th17-3","Colitis_2_23_2_Control","Colitis_2_23_2_Th17-1","Colitis_2_23_2_Th17-2"]
mouse={"NTN_1_19_5":"NTN1_mouse5","NTN_1_19_9_1":"NTN1_mouse9","NTN_1_19_9_2":"NTN1_mouse9","NTN_1_19_23_1":"NTN1_mouse23","NTN_1_19_23_2":"NTN1_mouse23"}
for s in S[5:]: mouse[s]=s
def idx(s): return ad.read_h5ad(f"{cell_tables}/{s}.h5ad",backed='r').obs_names
import pandas as pd
print(annotate(cell_tables,"mouse",{s:pd.Series(mouse[s],index=idx(s)) for s in S},made_by=f"run:{run_id}",note="NTN_1_19 samples named <exp>_<year>_<mouse>[_lane]: 9_1/9_2 and 23_1/23_2 read as two channels of one mouse (same sex by XIST); sample name = unit elsewhere. Inferred from names."))
dis={s:("colitis" if s.startswith("Colitis") else "nephrotoxic_nephritis") for s in S}
print(annotate(cell_tables,"disease_model",dis,made_by=f"run:{run_id}",note="GEO treatment characteristic: crescentic glomerulonephritis (kidney) vs experimental colitis (intestine)"))

{'ok': True, 'problems': [], 'column': 'mouse', 'samples': {'NTN_1_19_5': {'cells': 12495, 'without_value': 0}, 'NTN_1_19_9_1': {'cells': 12827, 'without_value': 0}, 'NTN_1_19_9_2': {'cells': 12848, 'without_value': 0}, 'NTN_1_19_23_1': {'cells': 13963, 'without_value': 0}, 'NTN_1_19_23_2': {'cells': 9512, 'without_value': 0}, 'NTN_2_23_1_Control': {'cells': 25658, 'without_value': 0}, 'NTN_2_23_1_Th17-1': {'cells': 19511, 'without_value': 0}, 'NTN_2_23_1_Th17-2': {'cells': 19811, 'without_value': 0}, 'NTN_2_23_1_Th17-3': {'cells': 15383, 'without_value': 0}, 'Colitis_2_23_2_Control': {'cells': 14174, 'without_value': 0}, 'Colitis_2_23_2_Th17-1': {'cells': 12110, 'without_value': 0}, 'Colitis_2_23_2_Th17-2': {'cells': 5126, 'without_value': 0}}, 'values': ['Colitis_2_23_2_Control', 'Colitis_2_23_2_Th17-1', 'Colitis_2_23_2_Th17-2', 'NTN1_mouse23', 'NTN1_mouse5', 'NTN1_mouse9', 'NTN_2_23_1_Control', 'NTN_2_23_1_Th17-1', 'NTN_2_23_1_Th17-2', 'NTN_2_23_1_Th17-3'], 'cells_per_value': {'NTN1

**build target map from guide names** (13:18)

In [ ]:
import json
genes={}
names=[v for v in ad.read_h5ad(f"{cell_tables}/NTN_2_23_1_Th17-1.h5ad").var_names if v.endswith('_capture')]
tm={}; ctl=[]
for n in names:
    b=n[5:-8]; g,i=b.rsplit('_',1)
    if g=='control': ctl.append(n)
    else: tm.setdefault(g,[]).append(n)
json.dump({"tm":tm,"ctl":ctl},open("tm.json","w"))
print(len(tm),len(ctl),list(tm))

28 20 ['Tlr4', 'Il17a', 'Ccr6', 'Il1r1', 'Il12rb', 'Tgfbr1', 'Il23r', 'Il6ra', 'Il7r', 'Cd40lg', 'Cebpb', 'Irf4', 'Jak1', 'Jak2', 'Myd88', 'Nfatc2', 'Nfkb1', 'Rorc', 'Runx1', 'Socs1', 'Socs3', 'Stat3', 'Stat6', 'Syk', 'Tbx21', 'Traf6', 'Tgfbr2', 'Ahr']

**print map for proposal** (13:18)

In [ ]:
print(json.dumps(tm)); print(json.dumps(ctl))

{"Tlr4": ["gRNA_Tlr4_1_capture", "gRNA_Tlr4_2_capture", "gRNA_Tlr4_3_capture", "gRNA_Tlr4_4_capture", "gRNA_Tlr4_5_capture"], "Il17a": ["gRNA_Il17a_1_capture", "gRNA_Il17a_2_capture", "gRNA_Il17a_3_capture", "gRNA_Il17a_4_capture", "gRNA_Il17a_5_capture"], "Ccr6": ["gRNA_Ccr6_1_capture", "gRNA_Ccr6_2_capture", "gRNA_Ccr6_3_capture", "gRNA_Ccr6_4_capture", "gRNA_Ccr6_5_capture"], "Il1r1": ["gRNA_Il1r1_1_capture", "gRNA_Il1r1_2_capture", "gRNA_Il1r1_3_capture", "gRNA_Il1r1_4_capture", "gRNA_Il1r1_5_capture"], "Il12rb": ["gRNA_Il12rb_1_capture", "gRNA_Il12rb_2_capture", "gRNA_Il12rb_3_capture", "gRNA_Il12rb_4_capture", "gRNA_Il12rb_5_capture"], "Tgfbr1": ["gRNA_Tgfbr1_1_capture", "gRNA_Tgfbr1_2_capture", "gRNA_Tgfbr1_3_capture", "gRNA_Tgfbr1_4_capture", "gRNA_Tgfbr1_5_capture"], "Il23r": ["gRNA_Il23r_1_capture", "gRNA_Il23r_2_capture", "gRNA_Il23r_3_capture", "gRNA_Il23r_4_capture", "gRNA_Il23r_5_capture"], "Il6ra": ["gRNA_Il6ra_1_capture", "gRNA_Il6ra_2_capture", "gRNA_Il6ra_3_capture", 

**check where annotations were stored** (13:18)

In [ ]:
a=ad.read_h5ad(f"{cell_tables}/NTN_1_19_5.h5ad",backed='r'); print(a.obs.columns.tolist())
import os; print([f for f in os.listdir(cell_tables) if not f.endswith('.h5ad')])

['mouse', 'disease_model']
['NTN_2_23_1_Th17-3_protospacer_calls_per_cell.csv', 'Colitis_2_23_2_Th17-1_protospacer_calls_per_cell.csv', 'manifest.json', 'NTN_2_23_1_Th17-2_protospacer_calls_per_cell.csv', 'NTN_2_23_1_Th17-1_protospacer_calls_per_cell.csv', 'Colitis_2_23_2_Th17-2_protospacer_calls_per_cell.csv']

**Inspect cell_table.annotate API** (13:20)

In [ ]:
import pertura.cell_table as ct, inspect
print(inspect.signature(ct.annotate)); print(ct.annotate.__doc__)
print([n for n in dir(ct) if not n.startswith('_')])

(folder, column, values, made_by='', note='')
Write a column into the cells of the samples' tables, as the agent reads each cell's unit or condition
    (docs/adr/0041, step 4): values is {sample: the cells' values}, each a Series indexed by the table's barcodes, a
    list in the table's cell order, or one value for every cell of the sample (a condition given by sample); a cell
    the values do not name gets none (''). Each distinct value is a unit or level of its own: a cell that belongs to
    none (its donor or mouse not told) is left without a value, which the build reads as no unit, never a word such as
    'unassigned', which would become a unit (GSE190604: a third "donor" of unassigned cells). The tables are rewritten
    in place, and the column noted in the manifest with who wrote it and how many cells have no value. Returns {ok,
    problems, column, samples: {name: {cells, without_value}}, values: the distinct values, up to 50, cells_per_value,
    read_as}.
['CALL_COLUMNS

**List samples** (13:20)

In [ ]:
import pertura.cell_table as ct
F="/data/pertura/workspace/screen_GSE267740-52e74a8e/design/cell_tables"
import os
s=ct.samples(F) if callable(ct.samples) else ct.samples
print(type(s), list(s)[:20] if hasattr(s,'__iter__') else s)

<class 'list'> [{'name': 'Colitis_2_23_2_Control', 'kind': 'cell_table', 'matrix': '/data/pertura/workspace/screen_GSE267740-52e74a8e/design/cell_tables/Colitis_2_23_2_Control.h5ad', 'made_by': 'pertura:10x'}, {'name': 'Colitis_2_23_2_Th17-1', 'kind': 'cell_table', 'matrix': '/data/pertura/workspace/screen_GSE267740-52e74a8e/design/cell_tables/Colitis_2_23_2_Th17-1.h5ad', 'made_by': 'pertura:10x', 'calls': '/data/pertura/workspace/screen_GSE267740-52e74a8e/design/cell_tables/Colitis_2_23_2_Th17-1_protospacer_calls_per_cell.csv'}, {'name': 'Colitis_2_23_2_Th17-2', 'kind': 'cell_table', 'matrix': '/data/pertura/workspace/screen_GSE267740-52e74a8e/design/cell_tables/Colitis_2_23_2_Th17-2.h5ad', 'made_by': 'pertura:10x', 'calls': '/data/pertura/workspace/screen_GSE267740-52e74a8e/design/cell_tables/Colitis_2_23_2_Th17-2_protospacer_calls_per_cell.csv'}, {'name': 'NTN_1_19_23_1', 'kind': 'cell_table', 'matrix': '/data/pertura/workspace/screen_GSE267740-52e74a8e/design/cell_tables/NTN_1_19_2

**Write capture and experiment columns** (13:21)

In [ ]:
import pertura.cell_table as ct
F="/data/pertura/workspace/screen_GSE267740-52e74a8e/design/cell_tables"
names=[s['name'] for s in ct.samples(F)]
exp=lambda n:'NTN1' if n.startswith('NTN_1') else 'NTN2' if n.startswith('NTN_2') else 'Colitis2'
r1=ct.annotate(F,'capture',{n:n for n in names},made_by='agent',note='capture = sample, as the user said')
r2=ct.annotate(F,'experiment',{n:exp(n) for n in names},made_by='agent',note='three experiments run apart, per user')
print(r1['ok'],r2['ok'],r2['values'])

True True ['Colitis2', 'NTN1', 'NTN2']

**Inspect columns present in cell tables** (13:21)

In [ ]:
import pertura.cell_table as ct, anndata as ad, json
F="/data/pertura/workspace/screen_GSE267740-52e74a8e/design/cell_tables"
a=ad.read_h5ad(F+"/NTN_1_19_5.h5ad",backed='r'); print(list(a.obs.columns))
print(json.dumps(ct.manifest(F) if callable(ct.manifest) else ct.manifest)[:1500])

['mouse', 'disease_model', 'capture', 'experiment']
{"samples": {"NTN_2_23_1_Control": {"file": "NTN_2_23_1_Control.h5ad", "calls": null, "made_by": "pertura:10x", "source_files": ["/data/pertura/deposit/screen_GSE267740/GSM8980195_NTN_2_23_1_Control_filtered_feature_bc_matrix.h5"], "cells": 25658, "features": 32729, "written": "2026-11-14T13:09:24"}, "NTN_2_23_1_Th17-1": {"file": "NTN_2_23_1_Th17-1.h5ad", "calls": "NTN_2_23_1_Th17-1_protospacer_calls_per_cell.csv", "made_by": "pertura:10x", "source_files": ["/data/pertura/deposit/screen_GSE267740/GSM8980197_NTN_2_23_1_Th17-1_filtered_feature_bc_matrix.h5", "/data/pertura/deposit/screen_GSE267740/GSM8980199_NTN_2_23_1_Th17-1_cells_per_protospacer.json.gz"], "cells": 19511, "features": 32729, "written": "2026-11-14T13:09:29"}, "NTN_2_23_1_Th17-2": {"file": "NTN_2_23_1_Th17-2.h5ad", "calls": "NTN_2_23_1_Th17-2_protospacer_calls_per_cell.csv", "made_by": "pertura:10x", "source_files": ["/data/pertura/deposit/screen_GSE267740/GSM8980200_NT

**Load per-target gene table** (13:39), on thread t1: Whole screen: which of the 28 targets change Th17 cells in vivo (kidney NTN and colitis analysed apart), what they change, and which agree.

In [ ]:
import pandas as pd, numpy as np
d=pt.load("20261114T133437-C-genes-6871a2:changed_genes")
print(type(d)); 
d=pd.DataFrame(d) if not isinstance(d,pd.DataFrame) else d
print(d.columns.tolist(), len(d)); print(d.head(3))

<class 'pandas.DataFrame'>
['target', 'gene', 'effect', 'p', 'q', 'test', 'cells', 'reference_cells'] 1194
   target    gene    effect         p         q      test  cells  \
0  Tgfbr1  Rnf149 -0.723565  0.000439  0.032169  edger_ql   3784   
1  Tgfbr1  Coq10b -0.289171  0.000711  0.038129  edger_ql   3784   
2  Tgfbr1   Ikzf2 -1.591148  0.000214  0.026162  edger_ql   3784   

   reference_cells  
0             4420  
1             4420  
2             4420

**Inspect screen object** (13:39), on thread t1: Whole screen: which of the 28 targets change Th17 cells in vivo (kidney NTN and colitis analysed apart), what they change, and which agree.

In [ ]:
import numpy as np, pandas as pd, scipy.sparse as sp
ad_=pt.screen().adata
print(ad_.shape, list(ad_.obs.columns)[:30])

(173418, 26538) ['sample', 'unit', 'umis', 'genes', 'unit_call', 'label', 'guides', 'control_group', 'control_kind', 'condition', 'qc_reason', 'label_after_qc']

**Test whether shared signatures are driven by TCR gene segments, per disease model** (13:39), on thread t1: Whole screen: which of the 28 targets change Th17 cells in vivo (kidney NTN and colitis analysed apart), what they change, and which agree.

In [ ]:
import numpy as np, pandas as pd, scipy.sparse as sp, re
a=pt.screen().adata
o=a.obs
print(o.condition.value_counts().to_dict(), o.label_after_qc.value_counts().head(5).to_dict())
X=a.X if sp.issparse(a.X) else sp.csr_matrix(a.X)
lab=o.label_after_qc.values
keep=np.isin(lab,[l for l in pd.unique(lab) if l not in ('none','multi','failed','') and pd.notna(l)])
genes=np.array(a.var_names)
tr=np.array([bool(re.match(r'^(Tr[abdg][vjc]|Tcrg|Trav|Trbv)',g)) for g in genes])
print(tr.sum())
# normalise
tot=np.asarray(X.sum(1)).ravel()
res={}
for cond in ['nephrotoxic_nephritis','colitis']:
    m=(o.condition.values==cond)&keep
    idx=np.where(m)[0]
    Xs=X[idx]; t=tot[idx]
    Xn=sp.diags(1e4/np.maximum(t,1))@Xs; Xn=Xn.tocsr(); Xn.data=np.log1p(Xn.data)
    l=lab[idx]
    expr=np.asarray(Xn.mean(0)).ravel(); sel=expr>0.05
    ctrl=Xn[l=='control'].mean(0).A1
    prof={}
    for g in set(l)-{'control'}:
        k=l==g
        if k.sum()>=30: prof[g]=Xn[k].mean(0).A1-ctrl
    P=pd.DataFrame(prof,index=genes)
    res[cond]=(P,sel,(l=='control').sum(),{g:int((l==g).sum()) for g in P.columns})
    for name,mask in [('all',sel),('noTR',sel&~tr)]:
        C=P[mask].corr(); iu=np.triu_indices(len(C),1)
        print(cond,name,'mean pair r',round(float(np.nanmean(C.values[iu])),3))
    C=P[sel&~tr].corr()
    print(cond,'noTR pairs:',{p:round(C.loc[p[0],p[1]],2) for p in [('Tgfbr1','Tgfbr2'),('Il17a','Traf6'),('Il17a','Nfatc2'),('Jak2','Myd88'),('Tbx21','Jak2')] if p[0] in C and p[1] in C})
    C=P[sel].corr()
    print(cond,'all pairs:',{p:round(C.loc[p[0],p[1]],2) for p in [('Tgfbr1','Tgfbr2'),('Il17a','Traf6'),('Il17a','Nfatc2'),('Jak2','Myd88'),('Tbx21','Jak2')] if p[0] in C and p[1] in C})
    print(cond,'ctrl cells',res[cond][2], 'targets',len(P.columns))

{'nephrotoxic_nephritis': 142008, 'colitis': 31410} {'none': 89395, 'multi': 27409, 'failed QC': 15072, 'control': 4420, 'Tgfbr1': 3784}
263
nephrotoxic_nephritis all mean pair r 0.361
nephrotoxic_nephritis noTR mean pair r 0.363
nephrotoxic_nephritis noTR pairs: {('Tgfbr1', 'Tgfbr2'): 0.84, ('Il17a', 'Traf6'): 0.81, ('Il17a', 'Nfatc2'): 0.44, ('Jak2', 'Myd88'): 0.95, ('Tbx21', 'Jak2'): 0.91}
nephrotoxic_nephritis all pairs: {('Tgfbr1', 'Tgfbr2'): 0.81, ('Il17a', 'Traf6'): 0.8, ('Il17a', 'Nfatc2'): 0.44, ('Jak2', 'Myd88'): 0.95, ('Tbx21', 'Jak2'): 0.9}
nephrotoxic_nephritis ctrl cells 3822 targets 29
colitis all mean pair r 0.267
colitis noTR mean pair r 0.276
colitis noTR pairs: {('Tgfbr1', 'Tgfbr2'): 0.07, ('Il17a', 'Traf6'): 0.75, ('Il17a', 'Nfatc2'): 0.66, ('Jak2', 'Myd88'): 0.15, ('Tbx21', 'Jak2'): 0.27}
colitis all pairs: {('Tgfbr1', 'Tgfbr2'): 0.08, ('Il17a', 'Traf6'): 0.67, ('Il17a', 'Nfatc2'): 0.61, ('Jak2', 'Myd88'): 0.15, ('Tbx21', 'Jak2'): 0.27}
colitis ctrl cells 598 targe

**Tgfbr1 vs control: per-capture, per-guide, leave-one-out effects on Th17 genes** (13:46), on thread t2: Tgfbr1 is the only target past its control-guide line (e-distance 1.34 vs line 0.76, 217 genes changed, Il17a/Ccr6/Il23r/Rorc down like Rorc KO), but 2,799 of its 3,784 cells sit in the three NTN_2_23_1 captures and its guides agree weakly (r 0.17, replicates 0.10). Is the effect Tgfbr1's, or one capture's/clone's/guide's?

Why: Is the Tgfbr1 signature one capture's/guide's, or Tgfbr1's?

In [ ]:
import numpy as np, pandas as pd, scipy.sparse as sp
a=pt.screen().adata; o=a.obs.copy()
print(o.columns.tolist()); print(o.label_after_qc.value_counts().head(5)); print(o.guides.head(3).tolist(), o.condition.unique(), o.control_kind.unique())
print(o[o.label_after_qc=='Tgfbr1'].groupby('unit').size())

['sample', 'unit', 'umis', 'genes', 'unit_call', 'label', 'guides', 'control_group', 'control_kind', 'condition', 'qc_reason', 'label_after_qc']
label_after_qc
none         89395
multi        27409
failed QC    15072
control       4420
Tgfbr1        3784
Name: count, dtype: int64
['', '', ''] <ArrowStringArray>
['colitis', 'nephrotoxic_nephritis']
Length: 2, dtype: str <ArrowStringArray>
['', 'non_targeting']
Length: 2, dtype: str
unit
Colitis_2_23_2_Th17-1     144
Colitis_2_23_2_Th17-2      31
NTN_1_19_23_1             136
NTN_1_19_23_2              83
NTN_1_19_5                 75
NTN_1_19_9_1              237
NTN_1_19_9_2              279
NTN_2_23_1_Th17-1        1050
NTN_2_23_1_Th17-2         773
NTN_2_23_1_Th17-3         976
dtype: int64

**Tgfbr1 vs control within each capture: Th17 genes and genome-wide profile correlation with pooled effect** (13:46), on thread t2: Tgfbr1 is the only target past its control-guide line (e-distance 1.34 vs line 0.76, 217 genes changed, Il17a/Ccr6/Il23r/Rorc down like Rorc KO), but 2,799 of its 3,784 cells sit in the three NTN_2_23_1 captures and its guides agree weakly (r 0.17, replicates 0.10). Is the effect Tgfbr1's, or one capture's/clone's/guide's?

In [ ]:
X=a.X; genes=np.array(a.var_names)
if not sp.issparse(X): X=sp.csr_matrix(X)
tot=np.asarray(X.sum(1)).ravel(); 
def prof(idx):
    m=np.asarray(X[idx].sum(0)).ravel(); return np.log1p(m/m.sum()*1e4)
gi={g:i for i,g in enumerate(genes)}
th=['Il17a','Il23r','Ccr6','Rorc','Il1r1','Stat3']
lab=o.label_after_qc.values; unit=o.unit.values
rows=[];P={}
for u in sorted(set(unit[lab=='Tgfbr1'])):
    t=np.where((lab=='Tgfbr1')&(unit==u))[0]; c=np.where((lab=='control')&(unit==u))[0]
    if len(c)<30: rows.append((u,len(t),len(c))+(np.nan,)*len(th)); continue
    d=prof(t)-prof(c); P[u]=d
    rows.append((u,len(t),len(c))+tuple(round(d[gi[g]],2) for g in th))
df=pd.DataFrame(rows,columns=['capture','n_Tgfbr1','n_ctrl']+th); print(df.to_string())
# correlation of each capture's profile with the mean of the others, expressed genes
expr=np.asarray(X.sum(0)).ravel()>200
import itertools
us=list(P); 
for u in us:
    oth=np.mean([P[v] for v in us if v!=u],axis=0); print(u, round(np.corrcoef(P[u][expr],oth[expr])[0,1],2))

                 capture  n_Tgfbr1  n_ctrl  Il17a  Il23r  Ccr6  Rorc  Il1r1  Stat3
0  Colitis_2_23_2_Th17-1       144     379   0.14  -0.08  0.12 -0.01  -0.18  -0.13
1  Colitis_2_23_2_Th17-2        31     219   0.72   0.18 -0.11 -0.06  -0.04  -0.13
2          NTN_1_19_23_1       136     985   0.00   0.01  0.03 -0.05   0.05  -0.06
3          NTN_1_19_23_2        83     545  -0.05   0.06 -0.03  0.08   0.04  -0.03
4             NTN_1_19_5        75     403  -0.41  -0.04  0.05  0.05   0.07   0.09
5           NTN_1_19_9_1       237     487  -0.08   0.06 -0.06  0.01   0.05  -0.23
6           NTN_1_19_9_2       279     498  -0.19   0.10 -0.00  0.10   0.02  -0.17
7      NTN_2_23_1_Th17-1      1050     216  -0.49  -0.10 -0.52 -0.59  -0.49  -0.33
8      NTN_2_23_1_Th17-2       773     462  -0.80  -0.09 -0.43 -0.54  -0.54  -0.22
9      NTN_2_23_1_Th17-3       976     226  -0.60   0.02 -0.43 -0.55  -0.49  -0.25
Colitis_2_23_2_Th17-1 0.12
Colitis_2_23_2_Th17-2 0.29
NTN_1_19_23_1 0.22
NTN_1_19_23_2 

**Per-experiment effect of Rorc, Tbx21, Tgfbr1, Tgfbr2 on Th17 genes (cells pooled within experiment, vs control cells of same experiment)** (13:46), on thread t2: Tgfbr1 is the only target past its control-guide line (e-distance 1.34 vs line 0.76, 217 genes changed, Il17a/Ccr6/Il23r/Rorc down like Rorc KO), but 2,799 of its 3,784 cells sit in the three NTN_2_23_1 captures and its guides agree weakly (r 0.17, replicates 0.10). Is the effect Tgfbr1's, or one capture's/clone's/guide's?

In [ ]:
expm=lambda n:'NTN1' if n.startswith('NTN_1') else 'NTN2' if n.startswith('NTN_2') else 'Colitis2'
ex=np.array([expm(u) for u in unit]); rows=[]
for e in ['NTN1','NTN2','Colitis2']:
    c=np.where((lab=='control')&(ex==e))[0]
    for t_ in ['Rorc','Tbx21','Tgfbr1','Tgfbr2','Socs1']:
        t=np.where((lab==t_)&(ex==e))[0]
        if len(t)<30: continue
        d=prof(t)-prof(c); rows.append((e,t_,len(t),len(c))+tuple(round(d[gi[g]],2) for g in ['Il17a','Il23r','Ccr6','Rorc']))
print(pd.DataFrame(rows,columns=['exp','target','n','n_ctrl','Il17a','Il23r','Ccr6','Rorc']).to_string())

         exp  target     n  n_ctrl  Il17a  Il23r  Ccr6  Rorc
0       NTN1    Rorc   394    2918  -0.71  -0.06 -0.07 -0.10
1       NTN1   Tbx21   561    2918   1.04   0.08  0.48  0.13
2       NTN1  Tgfbr1   810    2918  -0.10   0.04 -0.04  0.00
3       NTN1  Tgfbr2   607    2918  -0.14   0.11  0.02  0.05
4       NTN1   Socs1   331    2918  -0.45   0.01 -0.05 -0.05
5       NTN2    Rorc   110     904  -0.61  -0.34 -0.21 -0.37
6       NTN2   Tbx21  1381     904   0.94   0.34  0.57  0.15
7       NTN2  Tgfbr1  2799     904  -0.64  -0.05 -0.46 -0.56
8       NTN2  Tgfbr2  1037     904  -0.50   0.08 -0.38 -0.15
9       NTN2   Socs1   182     904  -0.44  -0.25 -0.18 -0.23
10  Colitis2    Rorc   217     598  -1.25  -0.37 -0.49 -0.20
11  Colitis2   Tbx21   274     598   1.49   0.34  0.52  0.36
12  Colitis2  Tgfbr1   175     598   0.33  -0.02  0.03 -0.04
13  Colitis2  Tgfbr2   115     598  -0.19   0.06 -0.17 -0.11
14  Colitis2   Socs1   294     598  -0.76  -0.21 -0.39 -0.31

**describe the screen for t4** (13:46, t4-probe), on thread t4: About 49% of cells carry 2+ guides (27,409 labelled multi); replicate r median 0.05 and guide r weak. Do f1 and f2 hold on single-guide cells and by guide strength (Rorc own gene -1.31, Tbx21 -1.31)?

Why: open notebook and see the cells, columns and controls

In [ ]:
pt.describe()

{'cells': 'every cell',
 'n_cells': 173418,
 'named_like': 'Colitis_2_23_2_Control:AAACCTGAGAAACCTA-1',
 'per_sample': {'Colitis_2_23_2_Control': 14174,
  'Colitis_2_23_2_Th17-1': 12110,
  'Colitis_2_23_2_Th17-2': 5126,
  'NTN_1_19_23_1': 13963,
  'NTN_1_19_23_2': 9512,
  'NTN_1_19_5': 12495,
  'NTN_1_19_9_1': 12827,
  'NTN_1_19_9_2': 12848,
  'NTN_2_23_1_Control': 25658,
  'NTN_2_23_1_Th17-1': 19511,
  'NTN_2_23_1_Th17-2': 19811,
  'NTN_2_23_1_Th17-3': 15383},
 'independent_units': True,
 'labels': {'none': 89395,
  'multi': 27409,
  'failed QC': 15072,
  'control': 4420,
  'Tgfbr1': 3784,
  'Il23r': 3400,
  'Tbx21': 2216,
  'Il17a': 2130,
  'Cd40lg': 2008,
  'Syk': 1973,
  'Il6ra': 1814,
  'Tgfbr2': 1759,
  'Nfatc2': 1342,
  'Stat3': 1328,
  'Cebpb': 1254,
  'Traf6': 1249,
  'Myd88': 1170,
  'Jak2': 1147,
  'Ahr': 939,
  'Il1r1': 887,
  'Socs3': 885,
  'Stat6': 882,
  'Tlr4': 874,
  'Ccr6': 861,
  'Irf4': 825},
 'targets_with_10_cells': 28,
 'control_guides': {'gRNA_control_18_captur

**Open the t3-fork notebook and describe the screen** (13:46, t3-fork), on thread t3: f1 and f2 were read against control guides (4,420 NTN, 598 colitis control cells). The 28 targets' profiles correlate 0.24 on average (289 pairs beyond control pairs), a possible shared response against the controls. Redo the gene test against other targets.

Why: First step of the t3 fork: see what the cells hold before running the other-targets reference

In [ ]:
print(pt.describe())

{'cells': 'every cell', 'n_cells': 173418, 'named_like': 'Colitis_2_23_2_Control:AAACCTGAGAAACCTA-1', 'per_sample': {'Colitis_2_23_2_Control': 14174, 'Colitis_2_23_2_Th17-1': 12110, 'Colitis_2_23_2_Th17-2': 5126, 'NTN_1_19_23_1': 13963, 'NTN_1_19_23_2': 9512, 'NTN_1_19_5': 12495, 'NTN_1_19_9_1': 12827, 'NTN_1_19_9_2': 12848, 'NTN_2_23_1_Control': 25658, 'NTN_2_23_1_Th17-1': 19511, 'NTN_2_23_1_Th17-2': 19811, 'NTN_2_23_1_Th17-3': 15383}, 'independent_units': True, 'labels': {'none': 89395, 'multi': 27409, 'failed QC': 15072, 'control': 4420, 'Tgfbr1': 3784, 'Il23r': 3400, 'Tbx21': 2216, 'Il17a': 2130, 'Cd40lg': 2008, 'Syk': 1973, 'Il6ra': 1814, 'Tgfbr2': 1759, 'Nfatc2': 1342, 'Stat3': 1328, 'Cebpb': 1254, 'Traf6': 1249, 'Myd88': 1170, 'Jak2': 1147, 'Ahr': 939, 'Il1r1': 887, 'Socs3': 885, 'Stat6': 882, 'Tlr4': 874, 'Ccr6': 861, 'Irf4': 825}, 'targets_with_10_cells': 28, 'control_guides': {'gRNA_control_18_capture': 717, 'gRNA_control_9_capture': 565, 'gRNA_control_20_capture': 463, 'gRNA

**check label, guide columns for single vs multi cells, Rorc and Tbx21** (13:47, t4-probe), on thread t4: About 49% of cells carry 2+ guides (27,409 labelled multi); replicate r median 0.05 and guide r weak. Do f1 and f2 hold on single-guide cells and by guide strength (Rorc own gene -1.31, Tbx21 -1.31)?

Why: see how single-guide cells and Rorc/Tbx21 are labelled

In [ ]:
import pandas as pd, numpy as np
a = pt.screen().adata
o = a.obs
print(o[['label','label_after_qc','guides','control_kind','unit_call']].head(3))
print(o['label_after_qc'].value_counts().head(40))
for t in ['Rorc','Tbx21']:
    m = o['guides'].astype(str).str.contains(t)
    print(t, m.sum(), o.loc[m,'label_after_qc'].value_counts().head(8).to_dict())
print(o['guides'].astype(str).value_counts().head(15))

                                          label label_after_qc guides  \
Colitis_2_23_2_Control:AAACCTGAGAAACCTA-1  none      failed QC          
Colitis_2_23_2_Control:AAACCTGAGAGTCGGT-1  none           none          
Colitis_2_23_2_Control:AAACCTGAGATAGTCA-1  none           none          

                                          control_kind unit_call  
Colitis_2_23_2_Control:AAACCTGAGAAACCTA-1                    one  
Colitis_2_23_2_Control:AAACCTGAGAGTCGGT-1                    one  
Colitis_2_23_2_Control:AAACCTGAGATAGTCA-1                    one  
label_after_qc
none         89395
multi        27409
failed QC    15072
control       4420
Tgfbr1        3784
Il23r         3400
Tbx21         2216
Il17a         2130
Cd40lg        2008
Syk           1973
Il6ra         1814
Tgfbr2        1759
Nfatc2        1342
Stat3         1328
Cebpb         1254
Traf6         1249
Myd88         1170
Jak2          1147
Ahr            939
Il1r1          887
Socs3          885
Stat6          882
Tlr4  

**describe the screen and per-condition, per-unit cell counts and UMI depth, for the t5 probe** (13:47, t5-probe), on thread t5: Set alike:Ahr+27 (28 targets, mean r 0.24, e.g. Tgfbr1-Tgfbr2 0.55, Il17a-Traf6 0.60, Jak2-Myd88 0.57). By disease model, Tgfbr1-Tgfbr2 is 0.84 in NTN and 0.07 in colitis, Jak2-Myd88 0.95 and 0.15, so much of it is NTN-specific.

Why: t5: see what the cells hold and whether NTN and colitis differ in depth or control-cell number

In [ ]:
import numpy as np, pandas as pd
print(pt.describe())
a = pt.screen().adata
print(a.obs.columns.tolist())
print(a.shape)


{'cells': 'every cell', 'n_cells': 173418, 'named_like': 'Colitis_2_23_2_Control:AAACCTGAGAAACCTA-1', 'per_sample': {'Colitis_2_23_2_Control': 14174, 'Colitis_2_23_2_Th17-1': 12110, 'Colitis_2_23_2_Th17-2': 5126, 'NTN_1_19_23_1': 13963, 'NTN_1_19_23_2': 9512, 'NTN_1_19_5': 12495, 'NTN_1_19_9_1': 12827, 'NTN_1_19_9_2': 12848, 'NTN_2_23_1_Control': 25658, 'NTN_2_23_1_Th17-1': 19511, 'NTN_2_23_1_Th17-2': 19811, 'NTN_2_23_1_Th17-3': 15383}, 'independent_units': True, 'labels': {'none': 89395, 'multi': 27409, 'failed QC': 15072, 'control': 4420, 'Tgfbr1': 3784, 'Il23r': 3400, 'Tbx21': 2216, 'Il17a': 2130, 'Cd40lg': 2008, 'Syk': 1973, 'Il6ra': 1814, 'Tgfbr2': 1759, 'Nfatc2': 1342, 'Stat3': 1328, 'Cebpb': 1254, 'Traf6': 1249, 'Myd88': 1170, 'Jak2': 1147, 'Ahr': 939, 'Il1r1': 887, 'Socs3': 885, 'Stat6': 882, 'Tlr4': 874, 'Ccr6': 861, 'Irf4': 825}, 'targets_with_10_cells': 28, 'control_guides': {'gRNA_control_18_capture': 717, 'gRNA_control_9_capture': 565, 'gRNA_control_20_capture': 463, 'gRNA

**per-condition target shift profiles vs control cells (lognorm), depth of each target's cells vs controls, and the profile correlation with and without a depth covariate** (13:47, t5-probe), on thread t5: Set alike:Ahr+27 (28 targets, mean r 0.24, e.g. Tgfbr1-Tgfbr2 0.55, Il17a-Traf6 0.60, Jak2-Myd88 0.57). By disease model, Tgfbr1-Tgfbr2 is 0.84 in NTN and 0.07 in colitis, Jak2-Myd88 0.95 and 0.15, so much of it is NTN-specific.

Why: t5: test whether the NTN-specific alike pairs follow capture depth or control-cell number rather than shared biology

In [ ]:
import numpy as np, pandas as pd, scipy.sparse as sp
a = pt.screen().adata
obs = a.obs[['condition','label_after_qc','umis','genes','unit','control_kind','qc_reason']].copy()
print(obs.groupby(['condition','label_after_qc'])['umis'].median().unstack(0).loc[['control','Tgfbr1','Tgfbr2','Jak2','Myd88','Ahr','Il17a','Traf6','Tbx21','Ccr6','Cebpb','Rorc']])
print(obs.groupby('condition')['label_after_qc'].apply(lambda s:(s=='control').sum()))
print(obs[obs.label_after_qc=='control'].groupby(['condition','unit']).size())


condition       colitis  nephrotoxic_nephritis
label_after_qc                                
control          9107.0                 6196.0
Tgfbr1           8561.0                10787.0
Tgfbr2           9289.0                 7251.5
Jak2             9788.0                 6541.5
Myd88            8151.0                 6389.0
Ahr              8871.0                 6634.5
Il17a            8874.0                 6312.0
Traf6            9579.0                 6646.0
Tbx21            8900.5                 6479.0
Ccr6            10016.5                 8337.0
Cebpb            8706.0                 7855.0
Rorc             9844.0                 7133.0
condition
colitis                   598
nephrotoxic_nephritis    3822
Name: label_after_qc, dtype: int64
condition              unit                 
colitis                Colitis_2_23_2_Th17-1    379
                       Colitis_2_23_2_Th17-2    219
nephrotoxic_nephritis  NTN_1_19_23_1            985
                       NTN_1_19_23_2

**depth-matched target shift profiles per disease model, and their pairwise correlations for the Ahr set and the named pairs, against the raw correlations** (13:47, t5-probe), on thread t5: Set alike:Ahr+27 (28 targets, mean r 0.24, e.g. Tgfbr1-Tgfbr2 0.55, Il17a-Traf6 0.60, Jak2-Myd88 0.57). By disease model, Tgfbr1-Tgfbr2 is 0.84 in NTN and 0.07 in colitis, Jak2-Myd88 0.95 and 0.15, so much of it is NTN-specific.

Why: t5: remove depth by matching control cells to each target's UMI distribution, then re-correlate the target profiles

In [ ]:
import numpy as np, pandas as pd, scipy.sparse as sp
a = pt.screen().adata
ln = pt.screen().lognorm
lab = a.obs['label_after_qc'].astype(str).values
cond = a.obs['condition'].astype(str).values
umi = np.log10(a.obs['umis'].astype(float).values+1)
# bins by umi
bins = np.quantile(umi, np.linspace(0,1,21))
bin_id = np.clip(np.searchsorted(bins, umi, side='right')-1, 0, 19)
res = {}
keep_targets = None
for c in ['colitis','nephrotoxic_nephritis']:
    m_c = (cond==c)&(lab=='control')
    idx_c = np.where(m_c)[0]
    targets = [t for t in np.unique(lab[(cond==c)]) if t not in ['none','multi','failed QC','control'] and (lab[(cond==c)]==t).sum()>=10]
    cw = {}
    prof = {}
    for t in targets:
        it = np.where((cond==c)&(lab==t))[0]
        bt = pd.Series(bin_id[it]).value_counts(normalize=True)
        # weights for controls so their bin distribution matches target
        bc = pd.Series(bin_id[idx_c]).value_counts()
        w = np.zeros(len(idx_c))
        for b, frac in bt.items():
            sel = bin_id[idx_c]==b
            if sel.sum()>0:
                w[sel] = frac/ sel.sum()
        w = w/ w.sum()
        X_t = ln[it]
        X_c = ln[idx_c]
        mt = np.asarray(X_t.mean(axis=0)).ravel()
        mc = np.asarray(sp.csr_matrix(w[None,:]) @ X_c).ravel()
        prof[t] = mt - mc
    P = pd.DataFrame(prof)
    res[c] = P
    print(c, len(targets), 'targets', 'ctrl cells', len(idx_c))
pd.to_pickle(res, '/data/pertura/t5_profiles.pkl') if False else None
import pickle, os
out = os.path.join(os.getcwd(), 't5_profiles.pkl')
pickle.dump(res, open(out,'wb'))
print(out)


---------------------------------------------------------------------------
NotImplementedError                       Traceback (most recent call last)
Cell In[4], line 43
     39         X_t = ln[it]
     40         X_c = ln[idx_c]
     41         mt = np.asarray(X_t.mean(axis=0)).ravel()
     42         mc = np.asarray(sp.csr_matrix(w[None,:]) @ X_c).ravel()
---> 43         prof[t] = mt - mc
     44     P = pd.DataFrame(prof)
     45     res[c] = P
     46     print(c, len(targets), 'targets', 'ctrl cells', len(idx_c))

File ~/pertura/envs/pertpy/lib/python3.12/site-packages/scipy/sparse/_base.py:867, in _spbase.__rsub__(self, other)
    865     if other == 0:
    866         return -self.copy()
--> 867     raise NotImplementedError('subtracting a sparse array from a '
    868                               'nonzero scalar is not supported')
    869 elif isdense(other):
    870     other = np.broadcast_to(other, self.shape)

NotImplementedError: subtracting a sparse array from a nonze

**depth-matched target shift profiles per disease model (fixed dense conversion), saved to the notebook folder** (13:47, t5-probe), on thread t5: Set alike:Ahr+27 (28 targets, mean r 0.24, e.g. Tgfbr1-Tgfbr2 0.55, Il17a-Traf6 0.60, Jak2-Myd88 0.57). By disease model, Tgfbr1-Tgfbr2 is 0.84 in NTN and 0.07 in colitis, Jak2-Myd88 0.95 and 0.15, so much of it is NTN-specific.

Why: t5: remove depth by matching control cells to each target's UMI distribution, then re-correlate the target profiles

In [ ]:
import numpy as np, pandas as pd, scipy.sparse as sp, pickle, os
a = pt.screen().adata
ln = pt.screen().lognorm
lab = a.obs['label_after_qc'].astype(str).values
cond = a.obs['condition'].astype(str).values
umi = np.log10(a.obs['umis'].astype(float).values+1)
bins = np.quantile(umi, np.linspace(0,1,21))
bin_id = np.clip(np.searchsorted(bins, umi, side='right')-1, 0, 19)
res = {}; raw = {}; nct = {}
for c in ['colitis','nephrotoxic_nephritis']:
    idx_c = np.where((cond==c)&(lab=='control'))[0]
    X_c = ln[idx_c]
    mc_raw = np.asarray(X_c.mean(axis=0)).ravel()
    targets = [t for t in np.unique(lab[(cond==c)]) if t not in ['none','multi','failed QC','control'] and (lab[(cond==c)]==t).sum()>=10]
    prof = {}; rawp = {}
    for t in targets:
        it = np.where((cond==c)&(lab==t))[0]
        bt = pd.Series(bin_id[it]).value_counts(normalize=True)
        w = np.zeros(len(idx_c))
        for b, frac in bt.items():
            sel = bin_id[idx_c]==b
            if sel.sum()>0:
                w[sel] = frac/sel.sum()
        w = w/w.sum()
        mt = np.asarray(ln[it].mean(axis=0)).ravel()
        mc = np.asarray((sp.csr_matrix(w[None,:]) @ X_c).todense()).ravel()
        prof[t] = mt - mc
        rawp[t] = mt - mc_raw
    res[c] = pd.DataFrame(prof); raw[c] = pd.DataFrame(rawp)
    nct[c] = len(idx_c)
    print(c, len(targets), 'targets; control cells', len(idx_c))
pickle.dump({'matched':res,'raw':raw,'nctrl':nct}, open(os.path.join(os.getcwd(),'t5_profiles.pkl'),'wb'))
print('saved')


colitis 28 targets; control cells 598
nephrotoxic_nephritis 28 targets; control cells 3822
saved

**correlations of target shift profiles, raw vs depth-matched, per disease model; named pairs and the all-pair mean** (13:48, t5-probe), on thread t5: Set alike:Ahr+27 (28 targets, mean r 0.24, e.g. Tgfbr1-Tgfbr2 0.55, Il17a-Traf6 0.60, Jak2-Myd88 0.57). By disease model, Tgfbr1-Tgfbr2 is 0.84 in NTN and 0.07 in colitis, Jak2-Myd88 0.95 and 0.15, so much of it is NTN-specific.

Why: t5: does the NTN pair correlation survive depth matching, and does colitis stay low

In [ ]:
import numpy as np, pandas as pd, pickle, os
d = pickle.load(open(os.path.join(os.getcwd(),'t5_profiles.pkl'),'rb'))
out = {}
pairs = [('Tgfbr1','Tgfbr2'),('Jak2','Myd88'),('Il17a','Traf6'),('Ahr','Il17a'),('Ccr6','Tgfbr2'),('Cd40lg','Irf4')]
for kind in ['raw','matched']:
    for c in ['colitis','nephrotoxic_nephritis']:
        P = d[kind][c]
        # top variable genes
        v = P.var(axis=1)
        P2 = P.loc[v.sort_values(ascending=False).index[:2000]]
        C = P2.corr(method='pearson')
        row = {}
        for x,y in pairs:
            if x in C.index and y in C.columns:
                row[f'{x}-{y}'] = round(C.loc[x,y],2)
        offd = C.values[np.triu_indices(C.shape[0],1)]
        row['mean_all_pairs'] = round(offd.mean(),3)
        row['n_targets'] = C.shape[0]
        out[(kind,c)] = row
for k,v in out.items(): print(k, v)


('raw', 'colitis') {'Tgfbr1-Tgfbr2': 0.07, 'Jak2-Myd88': 0.13, 'Il17a-Traf6': 0.68, 'Ahr-Il17a': 0.65, 'Ccr6-Tgfbr2': 0.22, 'Cd40lg-Irf4': 0.59, 'mean_all_pairs': 0.287, 'n_targets': 28}
('raw', 'nephrotoxic_nephritis') {'Tgfbr1-Tgfbr2': 0.82, 'Jak2-Myd88': 0.96, 'Il17a-Traf6': 0.83, 'Ahr-Il17a': 0.8, 'Ccr6-Tgfbr2': 0.78, 'Cd40lg-Irf4': 0.02, 'mean_all_pairs': 0.401, 'n_targets': 28}
('matched', 'colitis') {'Tgfbr1-Tgfbr2': 0.23, 'Jak2-Myd88': 0.16, 'Il17a-Traf6': 0.65, 'Ahr-Il17a': 0.67, 'Ccr6-Tgfbr2': 0.22, 'Cd40lg-Irf4': 0.58, 'mean_all_pairs': 0.313, 'n_targets': 28}
('matched', 'nephrotoxic_nephritis') {'Tgfbr1-Tgfbr2': 0.85, 'Jak2-Myd88': 0.97, 'Il17a-Traf6': 0.84, 'Ahr-Il17a': 0.85, 'Ccr6-Tgfbr2': 0.84, 'Cd40lg-Irf4': 0.19, 'mean_all_pairs': 0.503, 'n_targets': 28}

**Genes** (13:48), on thread t3: f1 and f2 were read against control guides (4,420 NTN, 598 colitis control cells). The 28 targets' profiles correlate 0.24 on average (289 pairs beyond control pairs), a possible shared response against the controls. Redo the gene test against other targets.

Why: Fork t3: redo the f1 gene step with other_targets as the reference, on Rorc and Tbx21 with the same genes

2 of 2 targets change genes at q < 0.05 (2 targets by pseudobulk by capture (edgeR-QL), since 10 captures hold both the target's and the reference cells); the calibration check was not run (the reference is not the control cells); the stron...

![20261114T134822665-1.png](figures/20261114T134822665-1.png)

**unit-matched control profiles per disease model, the shared NTN component's top genes, and technical-gene check (mito, ribo, low-depth markers)** (13:48, t5-probe), on thread t5: Set alike:Ahr+27 (28 targets, mean r 0.24, e.g. Tgfbr1-Tgfbr2 0.55, Il17a-Traf6 0.60, Jak2-Myd88 0.57). By disease model, Tgfbr1-Tgfbr2 is 0.84 in NTN and 0.07 in colitis, Jak2-Myd88 0.95 and 0.15, so much of it is NTN-specific.

Why: t5: test whether the NTN shared profile is a unit-level or technical shift shared by every target-labelled cell

In [ ]:
import numpy as np, pandas as pd, scipy.sparse as sp, pickle, os
a = pt.screen().adata
ln = pt.screen().lognorm
lab = a.obs['label_after_qc'].astype(str).values
cond = a.obs['condition'].astype(str).values
unit = a.obs['unit'].astype(str).values
genes = np.array(a.var_names)
res = {}
for c in ['colitis','nephrotoxic_nephritis']:
    targets = [t for t in np.unique(lab[(cond==c)]) if t not in ['none','multi','failed QC','control'] and (lab[(cond==c)]==t).sum()>=10]
    prof = {}
    for t in targets:
        it = np.where((cond==c)&(lab==t))[0]
        mt_u = pd.Series(unit[it]).value_counts(normalize=True)
        mt = np.asarray(ln[it].mean(axis=0)).ravel()
        mc = np.zeros_like(mt); tot=0
        for u,frac in mt_u.items():
            ic = np.where((cond==c)&(lab=='control')&(unit==u))[0]
            if len(ic)==0: continue
            mc += frac*np.asarray(ln[ic].mean(axis=0)).ravel()
            tot += frac
        prof[t] = (mt - mc/ tot)
    P = pd.DataFrame(prof, index=genes)
    res[c]=P
pickle.dump(res, open(os.path.join(os.getcwd(),'t5_unitmatched.pkl'),'wb'))
for c in res:
    P = res[c]
    v = P.var(axis=1)
    P2 = P.loc[v.sort_values(ascending=False).index[:2000]]
    C = P2.corr()
    pairs = [('Tgfbr1','Tgfbr2'),('Jak2','Myd88'),('Il17a','Traf6'),('Ahr','Il17a')]
    print(c, {f'{x}-{y}': round(C.loc[x,y],2) for x,y in pairs if x in C.index and y in C.columns}, 'mean offdiag', round(C.values[np.triu_indices(C.shape[0],1)].mean(),3))
    m = P.mean(axis=1).sort_values()
    print(c,'mean shift top down', list(m.index[:15]))
    print(c,'mean shift top up', list(m.index[-15:][::-1]))
    mt_genes = [g for g in genes if g.startswith('mt-')]
    rb = [g for g in genes if g.startswith(('Rpl','Rps'))]
    print(c,'mean shift mito', round(m[mt_genes].mean(),3), 'ribo', round(m[rb].mean(),3), 'overall', round(m.mean(),3))


colitis {'Tgfbr1-Tgfbr2': 0.19, 'Jak2-Myd88': 0.18, 'Il17a-Traf6': 0.63, 'Ahr-Il17a': 0.63} mean offdiag 0.285
colitis mean shift top down ['Klf2', 'Ifi27l2a', 'S1pr1', 'Gramd3', 'Trbv15', 'Atp1b3', 'Rasa3', 'Itga4', 'Tcf7', 'Il7r', 'Trbv2', 'Irf7', 'Cxcr3', 'Ass1', 'Klf3']
colitis mean shift top up ['Ifng', 'Ckb', 'Ermn', 'Trbv3', 'Cxcr6', 'Il1r2', 'Gem', 'Id2', 'Rbpj', 'Hif1a', 'Tff1', 'Ier5l', 'Cd82', 'Cdkn1a', 'Asb2']
colitis mean shift mito -0.023 ribo -0.013 overall 0.001
nephrotoxic_nephritis {'Tgfbr1-Tgfbr2': 0.49, 'Jak2-Myd88': 0.66, 'Il17a-Traf6': 0.4, 'Ahr-Il17a': 0.23} mean offdiag 0.175
nephrotoxic_nephritis mean shift top down ['Tcrg-V4', 'S100a6', 'Nkg7', 'Jarid2', 'Tnfsf8', 'Camk1d', 'Gm42418', 'Trbv2', 'AY036118', 'Dock2', 'Cmss1', 'Rgs16', 'Ifng', 'Rbpj', 'Atxn1']
nephrotoxic_nephritis mean shift top up ['Ctla2a', 'Cyb5a', 'Jund', 'Actn2', 'Neurl3', 'Snrpd3', 'Rpl31', 'Atp5g3', 'Atp5h', 'Junb', 'Top1', 'Ppp1ca', 'Il22', 'Gabarapl2', 'Nedd8']
nephrotoxic_nephritis mean

**count single and multi-guide cells per target and own-gene expression** (13:48, t4-probe), on thread t4: About 49% of cells carry 2+ guides (27,409 labelled multi); replicate r median 0.05 and guide r weak. Do f1 and f2 hold on single-guide cells and by guide strength (Rorc own gene -1.31, Tbx21 -1.31)?

Why: separate single-guide from multi-guide cells carrying each target's guide, and read own-gene level

In [ ]:
import numpy as np, pandas as pd
a = pt.screen().adata
o = a.obs
g = o['guides'].astype(str)
rows=[]
for t in ['Rorc','Tbx21']:
    # single-guide cells of target t
    single = (o['label']==t)
    # multi cells carrying a guide for t
    multi_t = (o['label']=='multi') & g.str.contains(f'gRNA_{t}_')
    # all cells carrying t guide
    print(t, 'single', single.sum(), 'multi carrying', multi_t.sum())
    print(g[multi_t].value_counts().head(5).to_dict())
# Efficacy on own gene: lognorm expression of target gene
ln = pt.screen().lognorm
genes = list(a.var_names)
for t in ['Rorc','Tbx21']:
    j = genes.index(t)
    x = np.asarray(ln[:, j].todense()).ravel()
    ctrl = (o['label']=='control').values
    s = (o['label']==t).values
    mt = ((o['label']=='multi') & g.str.contains(f'gRNA_{t}_')).values
    print(t, 'ctrl mean', x[ctrl].mean().round(3), 'single mean', x[s].mean().round(3), 'multi-with-t mean', x[mt].mean().round(3), 'frac0 ctrl', (x[ctrl]==0).mean().round(3), 'frac0 single', (x[s]==0).mean().round(3))

Rorc single 758 multi carrying 584
{'gRNA_Rorc_4_capture:1;gRNA_control_11_capture:1': 28, 'gRNA_Il1r1_1_capture:1;gRNA_Rorc_5_capture:1': 21, 'gRNA_Rorc_4_capture:2;gRNA_control_11_capture:1': 20, 'gRNA_Rorc_4_capture:3;gRNA_control_11_capture:1': 14, 'gRNA_Rorc_4_capture:1;gRNA_Tgfbr2_2_capture:1': 11}
Tbx21 single 2290 multi carrying 9012
{'gRNA_Cd40lg_2_capture:1;gRNA_Tbx21_1_capture:1': 4210, 'gRNA_Cd40lg_2_capture:1;gRNA_Stat3_1_capture:1;gRNA_Tbx21_1_capture:1': 1226, 'gRNA_Stat3_1_capture:1;gRNA_Tbx21_1_capture:1': 606, 'gRNA_Cd40lg_2_capture:1;gRNA_Tbx21_1_capture:1;gRNA_Tgfbr1_1_capture:1': 76, 'gRNA_Cd40lg_2_capture:1;gRNA_Il17a_3_capture:1;gRNA_Tbx21_1_capture:1': 75}
Rorc ctrl mean 0.279 single mean 0.166 multi-with-t mean 0.199 frac0 ctrl 0.744 frac0 single 0.821
Tbx21 ctrl mean 0.292 single mean 0.175 multi-with-t mean 0.184 frac0 ctrl 0.724 frac0 single 0.824

**own-gene level by single guide and for multi cells with and without the Cd40lg_2 co-guide** (13:48, t4-probe), on thread t4: About 49% of cells carry 2+ guides (27,409 labelled multi); replicate r median 0.05 and guide r weak. Do f1 and f2 hold on single-guide cells and by guide strength (Rorc own gene -1.31, Tbx21 -1.31)?

Why: guide strength and the Cd40lg_2 co-guide confound for Tbx21

In [ ]:
import numpy as np, pandas as pd
a = pt.screen().adata; o = a.obs; g = o['guides'].astype(str); ln = pt.screen().lognorm; genes=list(a.var_names)
ctrl = (o['label']=='control').values
out=[]
for t in ['Rorc','Tbx21']:
    j = genes.index(t); x = np.asarray(ln[:, j].todense()).ravel()
    s = (o['label']==t).values
    # single-guide cells by guide id
    gg = g.values
    sg = pd.Series(np.where(s, gg, ''), index=o.index)
    for gid in sorted(set(gg[s])):
        m = s & (gg==gid)
        if m.sum()>=20:
            out.append((t,'single',gid,m.sum(), round(x[m].mean(),3), round(x[ctrl].mean(),3)))
    # multi carrying t guide, with and without Cd40lg_2
    mt = ((o['label']=='multi') & g.str.contains(f'gRNA_{t}_')).values
    for nm, sub in [('multi with Cd40lg_2', mt & g.str.contains('gRNA_Cd40lg_2').values), ('multi without Cd40lg_2', mt & ~g.str.contains('gRNA_Cd40lg_2').values)]:
        if sub.sum()>0: out.append((t, nm, '', sub.sum(), round(x[sub].mean(),3), round(x[ctrl].mean(),3)))
for r in out: print(r)
# Cd40lg-only single cells as reference for Tbx21 partner effect
j=genes.index('Cd40lg'); xc=np.asarray(ln[:,j].todense()).ravel()
print('Cd40lg single own-gene', round(xc[(o['label']=='Cd40lg').values].mean(),3), 'ctrl', round(xc[ctrl].mean(),3))

('Rorc', 'single', 'gRNA_Rorc_1_capture:1', 76, 0.184, 0.279)
('Rorc', 'single', 'gRNA_Rorc_2_capture:1', 91, 0.256, 0.279)
('Rorc', 'single', 'gRNA_Rorc_3_capture:1', 256, 0.166, 0.279)
('Rorc', 'single', 'gRNA_Rorc_4_capture:1', 116, 0.142, 0.279)
('Rorc', 'single', 'gRNA_Rorc_4_capture:2', 40, 0.046, 0.279)
('Rorc', 'single', 'gRNA_Rorc_5_capture:1', 80, 0.218, 0.279)
('Rorc', 'multi with Cd40lg_2', '', 9, 0.417, 0.279)
('Rorc', 'multi without Cd40lg_2', '', 575, 0.195, 0.279)
('Tbx21', 'single', 'gRNA_Tbx21_1_capture:1', 1110, 0.127, 0.292)
('Tbx21', 'single', 'gRNA_Tbx21_1_capture:2', 56, 0.221, 0.292)
('Tbx21', 'single', 'gRNA_Tbx21_1_capture:3', 23, 0.174, 0.292)
('Tbx21', 'single', 'gRNA_Tbx21_2_capture:1', 28, 0.141, 0.292)
('Tbx21', 'single', 'gRNA_Tbx21_3_capture:1', 361, 0.198, 0.292)
('Tbx21', 'single', 'gRNA_Tbx21_4_capture:1', 382, 0.322, 0.292)
('Tbx21', 'single', 'gRNA_Tbx21_5_capture:1', 189, 0.137, 0.292)
('Tbx21', 'single', 'gRNA_Tbx21_5_capture:2', 40, 0.252, 0.292

**recompute the depth- and unit-matched profile correlations with TCR V/J/C genes removed, and measure how much of each model's shared profile sits on TCR genes** (13:48, t5-probe), on thread t5: Set alike:Ahr+27 (28 targets, mean r 0.24, e.g. Tgfbr1-Tgfbr2 0.55, Il17a-Traf6 0.60, Jak2-Myd88 0.57). By disease model, Tgfbr1-Tgfbr2 is 0.84 in NTN and 0.07 in colitis, Jak2-Myd88 0.95 and 0.15, so much of it is NTN-specific.

Why: t5: the genes run shows TCR variable-segment genes (Trav/Trbv) moved by most targets, a clonal-composition candidate for the shared profile

In [ ]:
import numpy as np, pandas as pd, pickle, os, re
d = pickle.load(open(os.path.join(os.getcwd(),'t5_unitmatched.pkl'),'rb'))
pm = pickle.load(open(os.path.join(os.getcwd(),'t5_profiles.pkl'),'rb'))
tcr = re.compile(r'^(Tr[abgd][vjc]|Trbc|Trac|Trgc|Trdc|Tcrg-|Trbv|Trav|Trgv|Trbj|Traj|Trdv|Trbd)')
pairs = [('Tgfbr1','Tgfbr2'),('Jak2','Myd88'),('Il17a','Traf6'),('Ahr','Il17a'),('Ccr6','Tgfbr2'),('Cd40lg','Irf4'),('Rorc','Tbx21')]
rows=[]
for name,DD in [('unit_matched',d),('depth_matched',{'colitis':pm['matched']['colitis'],'nephrotoxic_nephritis':pm['matched']['nephrotoxic_nephritis']})]:
    for c in ['colitis','nephrotoxic_nephritis']:
        P = DD[c] if isinstance(DD[c], pd.DataFrame) else None
        P.index = P.index if P.index.dtype==object else P.index
        isT = np.array([bool(tcr.match(g)) for g in P.index])
        for label, M in [('all', P), ('noTCR', P[~isT])]:
            v = M.var(axis=1)
            M2 = M.loc[v.sort_values(ascending=False).index[:2000]]
            C = M2.corr()
            r = {f'{x}-{y}': round(C.loc[x,y],2) for x,y in pairs if x in C.index and y in C.columns}
            r['mean_offdiag'] = round(C.values[np.triu_indices(C.shape[0],1)].mean(),3)
            rows.append((name,c,label,r))
        # share of squared profile norm on TCR genes, averaged over targets
        sq = (P**2)
        share = sq[isT].sum()/sq.sum()
        print(name,c,'TCR share of profile squared norm, mean over targets', round(share.mean(),3), 'range', round(share.min(),3), round(share.max(),3))
for r in rows: print(r)


unit_matched colitis TCR share of profile squared norm, mean over targets 0.079 range 0.015 0.203
unit_matched nephrotoxic_nephritis TCR share of profile squared norm, mean over targets 0.051 range 0.01 0.192
---------------------------------------------------------------------------
TypeError                                 Traceback (most recent call last)
Cell In[8], line 21
     17 for name,DD in [('unit_matched',d),('depth_matched',{'colitis':pm['matched']['colitis'],'nephrotoxic_nephritis':pm['matched']['nephrotoxic_nephritis']})]:
     18     for c in ['colitis','nephrotoxic_nephritis']:
     19         P = DD[c] if isinstance(DD[c], pd.DataFrame) else None
     20         P.index = P.index if P.index.dtype==object else P.index
---> 21         isT = np.array([bool(tcr.match(g)) for g in P.index])
     22         for label, M in [('all', P), ('noTCR', P[~isT])]:
     23             v = M.var(axis=1)
     24             M2 = M.loc[v.sort_values(ascending=False).index[:2000]]

Type

**TCR-gene-removed correlations for unit-matched and depth-matched profiles, per disease model** (13:48, t5-probe), on thread t5: Set alike:Ahr+27 (28 targets, mean r 0.24, e.g. Tgfbr1-Tgfbr2 0.55, Il17a-Traf6 0.60, Jak2-Myd88 0.57). By disease model, Tgfbr1-Tgfbr2 is 0.84 in NTN and 0.07 in colitis, Jak2-Myd88 0.95 and 0.15, so much of it is NTN-specific.

Why: t5: measure the shared profile without TCR genes in each model

In [ ]:
import numpy as np, pandas as pd, pickle, os, re
genes = np.array(pt.screen().adata.var_names)
d = pickle.load(open(os.path.join(os.getcwd(),'t5_unitmatched.pkl'),'rb'))
pm = pickle.load(open(os.path.join(os.getcwd(),'t5_profiles.pkl'),'rb'))
for c in ['colitis','nephrotoxic_nephritis']:
    pm['matched'][c].index = genes
tcr = re.compile(r'^(Tr[abgd][vjc]|Trbc|Trac|Trgc|Trdc|Tcrg-|Trgv|Trdv)')
pairs = [('Tgfbr1','Tgfbr2'),('Jak2','Myd88'),('Il17a','Traf6'),('Ahr','Il17a'),('Ccr6','Tgfbr2'),('Cd40lg','Irf4'),('Rorc','Tbx21')]
for name, DD in [('unit_matched', d), ('depth_matched', pm['matched'])]:
    for c in ['colitis','nephrotoxic_nephritis']:
        P = DD[c]
        isT = np.array([bool(tcr.match(g)) for g in P.index])
        for label, M in [('all', P), ('noTCR', P[~isT])]:
            v = M.var(axis=1)
            M2 = M.loc[v.sort_values(ascending=False).index[:2000]]
            C = M2.corr()
            r = {f'{x}-{y}': round(C.loc[x,y],2) for x,y in pairs if x in C.index and y in C.columns}
            r['mean_offdiag'] = round(C.values[np.triu_indices(C.shape[0],1)].mean(),3)
            print(name, c, label, r)


unit_matched colitis all {'Tgfbr1-Tgfbr2': 0.19, 'Jak2-Myd88': 0.18, 'Il17a-Traf6': 0.63, 'Ahr-Il17a': 0.63, 'Ccr6-Tgfbr2': 0.19, 'Cd40lg-Irf4': 0.51, 'Rorc-Tbx21': 0.1, 'mean_offdiag': 0.285}
unit_matched colitis noTCR {'Tgfbr1-Tgfbr2': 0.2, 'Jak2-Myd88': 0.18, 'Il17a-Traf6': 0.73, 'Ahr-Il17a': 0.72, 'Ccr6-Tgfbr2': 0.21, 'Cd40lg-Irf4': 0.6, 'Rorc-Tbx21': 0.11, 'mean_offdiag': 0.303}
unit_matched nephrotoxic_nephritis all {'Tgfbr1-Tgfbr2': 0.49, 'Jak2-Myd88': 0.66, 'Il17a-Traf6': 0.4, 'Ahr-Il17a': 0.23, 'Ccr6-Tgfbr2': 0.52, 'Cd40lg-Irf4': -0.05, 'Rorc-Tbx21': -0.28, 'mean_offdiag': 0.175}
unit_matched nephrotoxic_nephritis noTCR {'Tgfbr1-Tgfbr2': 0.56, 'Jak2-Myd88': 0.69, 'Il17a-Traf6': 0.4, 'Ahr-Il17a': 0.22, 'Ccr6-Tgfbr2': 0.55, 'Cd40lg-Irf4': -0.03, 'Rorc-Tbx21': -0.3, 'mean_offdiag': 0.185}
depth_matched colitis all {'Tgfbr1-Tgfbr2': 0.23, 'Jak2-Myd88': 0.16, 'Il17a-Traf6': 0.65, 'Ahr-Il17a': 0.67, 'Ccr6-Tgfbr2': 0.22, 'Cd40lg-Irf4': 0.58, 'Rorc-Tbx21': 0.09, 'mean_offdiag': 0.313}

**genes carrying the NTN shared pair profile (Tgfbr1, Tgfbr2, Jak2, Myd88, Ccr6 unit-matched shifts) and their unit composition** (13:49, t5-probe), on thread t5: Set alike:Ahr+27 (28 targets, mean r 0.24, e.g. Tgfbr1-Tgfbr2 0.55, Il17a-Traf6 0.60, Jak2-Myd88 0.57). By disease model, Tgfbr1-Tgfbr2 is 0.84 in NTN and 0.07 in colitis, Jak2-Myd88 0.95 and 0.15, so much of it is NTN-specific.

Why: t5: name the genes behind the NTN-specific pairs and check whether these targets share a unit or capture

In [ ]:
import numpy as np, pandas as pd, pickle, os, re
a = pt.screen().adata
genes = np.array(a.var_names)
d = pickle.load(open(os.path.join(os.getcwd(),'t5_unitmatched.pkl'),'rb'))
P = d['nephrotoxic_nephritis']
grp = ['Tgfbr1','Tgfbr2','Jak2','Myd88','Ccr6']
tcr = re.compile(r'^(Tr[abgd][vjc]|Trbc|Trac|Trgc|Trdc|Tcrg-|Trgv|Trdv)')
M = P[[g for g in P.columns if g in grp]]
M = M.loc[[not bool(tcr.match(g)) for g in P.index]]
print('mean shift of the 5 targets, top 20 up/down (unit-matched, noTCR):')
m = M.mean(axis=1).sort_values()
print('down', [(g, round(m[g],2)) for g in m.index[:20]])
print('up', [(g, round(m[g],2)) for g in m.index[-20:][::-1]])
print('corr among 5 on all genes', M.corr().round(2).to_string())
# unit composition of each target in NTN
obs = a.obs
sub = obs[(obs.condition=='nephrotoxic_nephritis')]
ct = pd.crosstab(sub.label_after_qc.astype(str), sub.unit.astype(str), normalize='index').round(2)
print(ct.loc[grp+['control']].to_string())


mean shift of the 5 targets, top 20 up/down (unit-matched, noTCR):
down [('Malat1', -0.19), ('AU020206', -0.18), ('Rgs10', -0.18), ('Rgs1', -0.17), ('St6galnac3', -0.17), ('Capg', -0.17), ('Btg1', -0.16), ('Znrf1', -0.16), ('Inpp4b', -0.16), ('Dusp2', -0.16), ('Camk1d', -0.16), ('Rbpj', -0.16), ('Gm42418', -0.15), ('Jarid2', -0.15), ('Sntb1', -0.14), ('Myh9', -0.14), ('Adam19', -0.13), ('Prkch', -0.13), ('Ctla2a', -0.13), ('Cmss1', -0.12)]
up [('Anxa2', 0.23), ('Lgals1', 0.2), ('H2afz', 0.2), ('Rpl28', 0.19), ('Rpl31', 0.19), ('Gimap7', 0.19), ('Gmfg', 0.18), ('Hopx', 0.18), ('S100a10', 0.17), ('Rps17', 0.17), ('Dnajc15', 0.17), ('Cd48', 0.17), ('Ly6c2', 0.17), ('Eno1', 0.17), ('Sec61g', 0.17), ('Tspo', 0.16), ('Cdc42', 0.16), ('Myl6', 0.16), ('Cfl1', 0.16), ('Rpl35', 0.16)]
corr among 5 on all genes         Ccr6  Jak2  Myd88  Tgfbr1  Tgfbr2
Ccr6    1.00  0.57   0.39    0.40    0.63
Jak2    0.57  1.00   0.64    0.04    0.38
Myd88   0.39  0.64   1.00   -0.11    0.18
Tgfbr1  0.40  0.04  

**Genes** (13:49), on thread t4: About 49% of cells carry 2+ guides (27,409 labelled multi); replicate r median 0.05 and guide r weak. Do f1 and f2 hold on single-guide cells and by guide strength (Rorc own gene -1.31, Tbx21 -1.31)?

Why: Rorc and Tbx21 effects on single-guide cells against control guides

2 of 2 targets change genes at q < 0.05 (2 targets by pseudobulk by capture (edgeR-QL), since 10 captures hold both the target's and the reference cells); 3 groups of 5 control guides tested as targets gave 5 discoveries; 2 targets change m...

![20261114T134907955-1.png](figures/20261114T134907955-1.png)

**per guide and multi-guide strata: Il17a, Ccr6, Il23r shift against control cells** (13:49, t4-probe), on thread t4: About 49% of cells carry 2+ guides (27,409 labelled multi); replicate r median 0.05 and guide r weak. Do f1 and f2 hold on single-guide cells and by guide strength (Rorc own gene -1.31, Tbx21 -1.31)?

Why: test f1 and f2 by guide strength and in multi-guide cells

In [ ]:
import numpy as np, pandas as pd
from scipy import stats
a = pt.screen().adata; o = a.obs; g = o['guides'].astype(str).values; ln = pt.screen().lognorm; genes=list(a.var_names)
ctrl = (o['label']=='control').values
genesq = ['Il17a','Il23r','Ccr6','Rorc','Tbx21']
X = {q: np.asarray(ln[:, genes.index(q)].todense()).ravel() for q in genesq}
def row(name, m, t):
    res=[name, int(m.sum())]
    for q in ['Il17a','Ccr6','Il23r']:
        v=X[q][m]; c=X[q][ctrl]
        d=v.mean()-c.mean()
        p=stats.mannwhitneyu(v,c).pvalue if m.sum()>=10 else np.nan
        res += [q, round(d,3), '%.1e'%p]
    return res
rows=[]
s_ctrl=ctrl
for t in ['Rorc','Tbx21']:
    own=(o['label']==t).values
    for gid in sorted(set(g[own])):
        m = own & (g==gid)
        if m.sum()>=20: rows.append(row(f'{t} single {gid}', m, t))
    mt = ((o['label']=='multi') & pd.Series(g).str.contains(f'gRNA_{t}_').values)
    if t=='Tbx21':
        rows.append(row('Tbx21 multi with Cd40lg_2', mt & pd.Series(g).str.contains('gRNA_Cd40lg_2').values, t))
        rows.append(row('Tbx21 multi without Cd40lg_2', mt & ~pd.Series(g).str.contains('gRNA_Cd40lg_2').values, t))
    else:
        rows.append(row('Rorc multi (all)', mt, t))
rows.append(['Control cells', int(ctrl.sum())])
df=pd.DataFrame(rows)
print(df.to_string())

                                     0     1      2      3         4     5      6         7      8      9         10
0     Rorc single gRNA_Rorc_1_capture:1    76  Il17a -0.304   2.0e-03  Ccr6  0.033   2.4e-01  Il23r -0.023   5.8e-01
1     Rorc single gRNA_Rorc_2_capture:1    91  Il17a -0.251   4.9e-03  Ccr6 -0.007   8.2e-01  Il23r -0.087   7.4e-02
2     Rorc single gRNA_Rorc_3_capture:1   256  Il17a -0.413   3.3e-10  Ccr6 -0.118   2.2e-05  Il23r -0.074   3.9e-03
3     Rorc single gRNA_Rorc_4_capture:1   116  Il17a -0.282   3.4e-03  Ccr6 -0.113   4.7e-03  Il23r -0.130   1.4e-04
4     Rorc single gRNA_Rorc_4_capture:2    40  Il17a -0.308   9.8e-02  Ccr6 -0.161   1.4e-02  Il23r -0.130   2.6e-02
5     Rorc single gRNA_Rorc_5_capture:1    80  Il17a -0.347   1.4e-03  Ccr6 -0.086   1.2e-01  Il23r -0.064   1.6e-01
6                      Rorc multi (all)   584  Il17a -0.246   2.0e-05  Ccr6 -0.052   2.5e-01  Il23r -0.027   7.9e-01
7   Tbx21 single gRNA_Tbx21_1_capture:1  1110  Il17a  0.404   1.

**control-cell number test: subsample NTN control cells to the colitis count (598) and recompute the target profile correlations, unit-matched within units; plus a depth-only check** (13:50, t5-probe), on thread t5: Set alike:Ahr+27 (28 targets, mean r 0.24, e.g. Tgfbr1-Tgfbr2 0.55, Il17a-Traf6 0.60, Jak2-Myd88 0.57). By disease model, Tgfbr1-Tgfbr2 is 0.84 in NTN and 0.07 in colitis, Jak2-Myd88 0.95 and 0.15, so much of it is NTN-specific.

Why: t5: test whether the NTN pair correlations depend on the number of control cells (3822 in NTN, 598 in colitis)

In [ ]:
import numpy as np, pandas as pd, scipy.sparse as sp, re
a = pt.screen().adata
ln = pt.screen().lognorm
genes = np.array(a.var_names)
lab = a.obs['label_after_qc'].astype(str).values
cond = a.obs['condition'].astype(str).values
unit = a.obs['unit'].astype(str).values
rng = np.random.default_rng(1)
tcr = re.compile(r'^(Tr[abgd][vjc]|Trbc|Trac|Trgc|Trdc|Tcrg-|Trgv|Trdv)')
keepg = np.array([not bool(tcr.match(g)) for g in genes])
c='nephrotoxic_nephritis'
targets = ['Tgfbr1','Tgfbr2','Jak2','Myd88','Ccr6','Il17a','Traf6','Ahr','Irf4','Cd40lg']
allc = np.where((cond==c)&(lab=='control'))[0]
def profiles(ctrl_idx):
    prof={}
    for t in targets:
        it = np.where((cond==c)&(lab==t))[0]
        mt_u = pd.Series(unit[it]).value_counts(normalize=True)
        mt = np.asarray(ln[it].mean(axis=0)).ravel()
        mc=np.zeros_like(mt); tot=0
        for u,frac in mt_u.items():
            ic = ctrl_idx[unit[ctrl_idx]==u]
            if len(ic)==0: continue
            mc += frac*np.asarray(ln[ic].mean(axis=0)).ravel(); tot+=frac
        prof[t]=(mt-mc/tot)[keepg]
    return pd.DataFrame(prof)
def summ(P):
    v=P.var(axis=1); P2=P.loc[v.sort_values(ascending=False).index[:2000]]
    C=P2.corr()
    return {x+'-'+y: round(C.loc[x,y],2) for x,y in [('Tgfbr1','Tgfbr2'),('Jak2','Myd88'),('Ccr6','Tgfbr2'),('Ahr','Il17a'),('Irf4','Cd40lg')]}, round(C.values[np.triu_indices(C.shape[0],1)].mean(),3)
full = profiles(allc)
print('all NTN controls (3822):', summ(full))
# subsample to 598 controls, 5 draws
for k in range(3):
    sub = rng.choice(allc, size=598, replace=False)
    print('subsample 598 draw',k, summ(profiles(sub)))


all NTN controls (3822): ({'Tgfbr1-Tgfbr2': 0.55, 'Jak2-Myd88': 0.69, 'Ccr6-Tgfbr2': 0.54, 'Ahr-Il17a': 0.21, 'Irf4-Cd40lg': -0.05}, 0.184)
subsample 598 draw 0 ({'Tgfbr1-Tgfbr2': 0.61, 'Jak2-Myd88': 0.73, 'Ccr6-Tgfbr2': 0.58, 'Ahr-Il17a': 0.39, 'Irf4-Cd40lg': 0.0}, 0.294)
subsample 598 draw 1 ({'Tgfbr1-Tgfbr2': 0.6, 'Jak2-Myd88': 0.84, 'Ccr6-Tgfbr2': 0.66, 'Ahr-Il17a': 0.46, 'Irf4-Cd40lg': 0.17}, 0.416)
subsample 598 draw 2 ({'Tgfbr1-Tgfbr2': 0.6, 'Jak2-Myd88': 0.8, 'Ccr6-Tgfbr2': 0.65, 'Ahr-Il17a': 0.34, 'Irf4-Cd40lg': -0.0}, 0.365)

**Efficacy** (13:50), on thread t4: About 49% of cells carry 2+ guides (27,409 labelled multi); replicate r median 0.05 and guide r weak. Do f1 and f2 hold on single-guide cells and by guide strength (Rorc own gene -1.31, Tbx21 -1.31)?

Why: own-gene knockdown by guide for Rorc and Tbx21, to read guide strength

5 guides change their target gene past the line (at least 30% down, or up by the same factor), 4 less, 4 of them knockout guides that leave their transcript unchanged, 1 not measurable

![20261114T135018084-1.png](figures/20261114T135018084-1.png)

**run run0036-cc31aa** (13:50)

ended: ok

**Dose** (13:53), on thread t4: About 49% of cells carry 2+ guides (27,409 labelled multi); replicate r median 0.05 and guide r weak. Do f1 and f2 hold on single-guide cells and by guide strength (Rorc own gene -1.31, Tbx21 -1.31)?

Why: effect against guide strength (dose) for Rorc and Tbx21

2 targets whose guides of one mode differ in B.efficacy's verdict (one works, another does not)

![20261114T135330270-1.png](figures/20261114T135330270-1.png)

**Rorc and Tbx21 vs other single-target cells within each capture (Il17a, Il23r, Ccr6), summarised per experiment** (13:53)

Why: Doubt d1: is the per-experiment direction a common-response offset against control guides?

In [ ]:
targets=set(lab)-{'none','multi','failed QC','control'}
rows=[]
for u in sorted(set(unit)):
    for t_ in ['Rorc','Tbx21']:
        t=np.where((lab==t_)&(unit==u))[0]; c=np.where(np.isin(lab,list(targets-{t_}))&(unit==u))[0]
        if len(t)<15 or len(c)<100: continue
        d=prof(t)-prof(c); rows.append((expm(u),u,t_,len(t),len(c))+tuple(d[gi[g]] for g in ['Il17a','Il23r','Ccr6']))
r=pd.DataFrame(rows,columns=['exp','capture','target','n','n_other','Il17a','Il23r','Ccr6'])
print(r.round(2).to_string())
print(r.groupby(['target','exp'])[['Il17a','Il23r','Ccr6']].agg(['mean','count']).round(2))
print((np.sign(r[['Il17a','Il23r','Ccr6']].values)*np.where(r.target.values=='Rorc',-1,1)[:,None]>0).mean(0))

         exp                capture target    n  n_other  Il17a  Il23r  Ccr6
0   Colitis2  Colitis_2_23_2_Th17-1   Rorc   62     4467  -1.49  -0.45 -0.30
1   Colitis2  Colitis_2_23_2_Th17-1  Tbx21  210     4319   1.29   0.22  0.63
2   Colitis2  Colitis_2_23_2_Th17-2   Rorc  155     1888  -1.37  -0.43 -0.42
3   Colitis2  Colitis_2_23_2_Th17-2  Tbx21   64     1979   1.59   0.52  0.68
4       NTN1          NTN_1_19_23_1   Rorc  148     3721  -0.99  -0.13 -0.32
5       NTN1          NTN_1_19_23_1  Tbx21  215     3654   1.10   0.08  0.51
6       NTN1          NTN_1_19_23_2   Rorc  108     2104  -1.52  -0.14 -0.26
7       NTN1          NTN_1_19_23_2  Tbx21  114     2098   0.95   0.05  0.45
8       NTN1             NTN_1_19_5   Rorc   83     2812   0.05  -0.08 -0.04
9       NTN1             NTN_1_19_5  Tbx21  120     2775  -0.17  -0.02  0.02
10      NTN1           NTN_1_19_9_1   Rorc   22     2501  -0.74  -0.06  0.02
11      NTN1           NTN_1_19_9_1  Tbx21   42     2481   0.86   0.18  0.51

**describe the screen** (13:59, t7-targets-on-each-other), on thread t7: Does Rorc or Tbx21 knockout change Tgfbr1, Il23r, Ccr6, Stat3 or other targets' own genes? Il23r (-2.74) and Ccr6 (-2.15) show strong own-gene knockdown, and Il23r/Ccr6 fall in the Rorc KO.

Why: start notebook per brief

In [ ]:
pt.describe()

{'cells': 'every cell',
 'n_cells': 173418,
 'named_like': 'Colitis_2_23_2_Control:AAACCTGAGAAACCTA-1',
 'per_sample': {'Colitis_2_23_2_Control': 14174,
  'Colitis_2_23_2_Th17-1': 12110,
  'Colitis_2_23_2_Th17-2': 5126,
  'NTN_1_19_23_1': 13963,
  'NTN_1_19_23_2': 9512,
  'NTN_1_19_5': 12495,
  'NTN_1_19_9_1': 12827,
  'NTN_1_19_9_2': 12848,
  'NTN_2_23_1_Control': 25658,
  'NTN_2_23_1_Th17-1': 19511,
  'NTN_2_23_1_Th17-2': 19811,
  'NTN_2_23_1_Th17-3': 15383},
 'independent_units': True,
 'labels': {'none': 89395,
  'multi': 27409,
  'failed QC': 15072,
  'control': 4420,
  'Tgfbr1': 3784,
  'Il23r': 3400,
  'Tbx21': 2216,
  'Il17a': 2130,
  'Cd40lg': 2008,
  'Syk': 1973,
  'Il6ra': 1814,
  'Tgfbr2': 1759,
  'Nfatc2': 1342,
  'Stat3': 1328,
  'Cebpb': 1254,
  'Traf6': 1249,
  'Myd88': 1170,
  'Jak2': 1147,
  'Ahr': 939,
  'Il1r1': 887,
  'Socs3': 885,
  'Stat6': 882,
  'Tlr4': 874,
  'Ccr6': 861,
  'Irf4': 825},
 'targets_with_10_cells': 28,
 'control_guides': {'gRNA_control_18_captur

**describe the screen** (14:01, t8-second-readout), on thread t8: 122 antibody features (CD4, CD8a, CD366, PD-1, Ly-6C, CD11b) in the screen are unread. Do Tgfbr1, Rorc and Tbx21 shift surface proteins against control cells, and do they match RNA?

Why: orient in the screen before the protein test

In [ ]:
print(pt.describe())

{'cells': 'every cell', 'n_cells': 173418, 'named_like': 'Colitis_2_23_2_Control:AAACCTGAGAAACCTA-1', 'per_sample': {'Colitis_2_23_2_Control': 14174, 'Colitis_2_23_2_Th17-1': 12110, 'Colitis_2_23_2_Th17-2': 5126, 'NTN_1_19_23_1': 13963, 'NTN_1_19_23_2': 9512, 'NTN_1_19_5': 12495, 'NTN_1_19_9_1': 12827, 'NTN_1_19_9_2': 12848, 'NTN_2_23_1_Control': 25658, 'NTN_2_23_1_Th17-1': 19511, 'NTN_2_23_1_Th17-2': 19811, 'NTN_2_23_1_Th17-3': 15383}, 'independent_units': True, 'labels': {'none': 89395, 'multi': 27409, 'failed QC': 15072, 'control': 4420, 'Tgfbr1': 3784, 'Il23r': 3400, 'Tbx21': 2216, 'Il17a': 2130, 'Cd40lg': 2008, 'Syk': 1973, 'Il6ra': 1814, 'Tgfbr2': 1759, 'Nfatc2': 1342, 'Stat3': 1328, 'Cebpb': 1254, 'Traf6': 1249, 'Myd88': 1170, 'Jak2': 1147, 'Ahr': 939, 'Il1r1': 887, 'Socs3': 885, 'Stat6': 882, 'Tlr4': 874, 'Ccr6': 861, 'Irf4': 825}, 'targets_with_10_cells': 28, 'control_guides': {'gRNA_control_18_capture': 717, 'gRNA_control_9_capture': 565, 'gRNA_control_20_capture': 463, 'gRNA

**inspect the protein features (obsm/uns) in the screen object** (14:01, t8-second-readout), on thread t8: 122 antibody features (CD4, CD8a, CD366, PD-1, Ly-6C, CD11b) in the screen are unread. Do Tgfbr1, Rorc and Tbx21 shift surface proteins against control cells, and do they match RNA?

Why: find where the antibody counts are held

In [ ]:
import numpy as np
a = pt.screen().adata
print(a.obsm.keys() if hasattr(a,'obsm') else None)
print([k for k in a.uns.keys()][:50])
for k in a.obsm.keys():
    try: print(k, a.obsm[k].shape, list(getattr(a.obsm[k],'columns',[]))[:130])
    except Exception as e: print(k, e)
print([v for v in a.var_names if v.startswith('CD') or 'Ly' in v][:50])


KeysView(AxisArrays with keys: 'antibody_counts')
[]
antibody_counts (173418, 122) ['CD4_TotalSeqC', 'CD8a_TotalSeqC', 'CD366_TotalSeqC', 'PD-1_TotalSeqC', 'Ly-6C_TotalSeqC', 'CD11b_TotalSeqC', 'Ly-6G_TotalSeqC', 'CD49f_TotalSeqC', 'CD44_TotalSeqC', 'CD54_TotalSeqC', 'CD90_2_TotalSeqC', 'CD73_TotalSeqC', 'CD49d_TotalSeqC', 'CD200_TotalSeqC', 'IgG1_TotalSeqC', 'IgG2a_TotalSeqC', 'CD19_TotalSeqC', 'CD45_TotalSeqC', 'CD25_TotalSeqC', 'CD45R_TotalSeqC', 'CSF-1R_TotalSeqC', 'CD11c_TotalSeqC', 'CD21_TotalSeqC', 'CD23_TotalSeqC', 'CD43_TotalSeqC', 'CD5_TotalSeqC', 'CD62L_TotalSeqC', 'CD93_TotalSeqC', 'Ly71_TotalSeqC', 'Fc_epsilon_RI_TotalSeqC', 'I-A_I-E_TotalSeqC', 'NK-1_1_TotalSeqC', 'Siglec-H_TotalSeqC', 'TCR-b-chain_TotalSeqC', 'TCR_yd_TotalSeqC', 'TER-119_TotalSeqC', 'Ly-6A_E_TotalSeqC', 'CD45_2_TotalSeqC', 'ICOS_TotalSeqC', 'CD3_TotalSeqC', 'PD-L1_TotalSeqC', 'CD27_TotalSeqC', 'CD20_TotalSeqC', 'CD357_TotalSeqC', 'CD137_TotalSeqC', 'CD134_TotalSeqC', 'CD69_TotalSeqC', 'IL-7RA_TotalSeqC',

**run 20261114T134310-C-states_and_composition-b254f4** (14:03)

22 states, numbered: name each from its markers, here the test's top four with the % of cells expressing each inside/outside the state (mitochondrial and ribosomal protein genes left out; the markers table has 10 per state, those among them...

![20261114T140310321-1.png](figures/20261114T140310321-1.png)

**Programs** (14:03), on thread t8: 122 antibody features (CD4, CD8a, CD366, PD-1, Ly-6C, CD11b) in the screen are unread. Do Tgfbr1, Rorc and Tbx21 shift surface proteins against control cells, and do they match RNA?

Why: protein shifts for Tgfbr1, Rorc, Tbx21 vs control guides, by condition

by condition, each value apart: colitis: 106 target x program pairs changed at adjusted p < 0.05 (proteins, CLR per cell (Seurat's multimodal vignette); OLS, sample as a covariate); the same test on 3 groups of control guides: 42 of 366 gro...

![20261114T140310526-1.png](figures/20261114T140310526-1.png)

**list labels and conditions of the cells** (14:03, t8-second-readout), on thread t8: 122 antibody features (CD4, CD8a, CD366, PD-1, Ly-6C, CD11b) in the screen are unread. Do Tgfbr1, Rorc and Tbx21 shift surface proteins against control cells, and do they match RNA?

Why: check the cell counts per target and condition before the RNA comparison

In [ ]:
import pandas as pd
c = pt.describe()
a = pt.screen().adata
print(a.obs['label_after_qc'].value_counts().head(40))
print(a.obs['condition'].value_counts())
print(a.obs.columns.tolist())


label_after_qc
none         89395
multi        27409
failed QC    15072
control       4420
Tgfbr1        3784
Il23r         3400
Tbx21         2216
Il17a         2130
Cd40lg        2008
Syk           1973
Il6ra         1814
Tgfbr2        1759
Nfatc2        1342
Stat3         1328
Cebpb         1254
Traf6         1249
Myd88         1170
Jak2          1147
Ahr            939
Il1r1          887
Socs3          885
Stat6          882
Tlr4           874
Ccr6           861
Irf4           825
Socs1          807
Runx1          751
Rorc           721
Il12rb         613
Nfkb1          532
Jak1           489
Il7r           482
Name: count, dtype: int64
condition
nephrotoxic_nephritis    142008
colitis                   31410
Name: count, dtype: int64
['sample', 'unit', 'umis', 'genes', 'unit_call', 'label', 'guides', 'control_group', 'control_kind', 'condition', 'qc_reason', 'label_after_qc']

**cells per target and control group, by condition** (14:03, t8-second-readout), on thread t8: 122 antibody features (CD4, CD8a, CD366, PD-1, Ly-6C, CD11b) in the screen are unread. Do Tgfbr1, Rorc and Tbx21 shift surface proteins against control cells, and do they match RNA?

Why: cell counts per target and control group, by condition

In [ ]:
import pandas as pd
a = pt.screen().adata
o = a.obs
print(o.loc[o.label_after_qc.isin(['Tgfbr1','Rorc','Tbx21','control']),:].groupby(['condition','label_after_qc']).size().unstack(0))
print(o.loc[o.label_after_qc.isin(['control']),'control_kind'].value_counts())
print(o.loc[o.label_after_qc.isin(['control']),'control_group'].value_counts().head(10))
print(o.loc[o.label_after_qc.isin(['control']),'unit'].value_counts())


condition       colitis  nephrotoxic_nephritis
label_after_qc                                
Rorc                217                    504
Tbx21               274                   1942
Tgfbr1              175                   3609
control             598                   3822
control_kind
non_targeting    4420
Name: count, dtype: int64
control_group
gRNA_control_18_capture    717
gRNA_control_9_capture     565
gRNA_control_20_capture    463
gRNA_control_19_capture    300
gRNA_control_1_capture     296
gRNA_control_16_capture    242
gRNA_control_6_capture     211
gRNA_control_8_capture     199
gRNA_control_11_capture    191
gRNA_control_17_capture    156
Name: count, dtype: int64
unit
NTN_1_19_23_1            985
NTN_1_19_23_2            545
NTN_1_19_9_2             498
NTN_1_19_9_1             487
NTN_2_23_1_Th17-2        462
NTN_1_19_5               403
Colitis_2_23_2_Th17-1    379
NTN_2_23_1_Th17-3        226
Colitis_2_23_2_Th17-2    219
NTN_2_23_1_Th17-1        216
Name: count,

**load the C.programs protein result table into the notebook** (14:03, t8-second-readout), on thread t8: 122 antibody features (CD4, CD8a, CD366, PD-1, Ly-6C, CD11b) in the screen are unread. Do Tgfbr1, Rorc and Tbx21 shift surface proteins against control cells, and do they match RNA?

Why: read the full protein table for the three targets and control groups

In [ ]:
import pandas as pd, numpy as np
r = pt.load("20261114T140122-C-programs-f84809:result")
print(type(r)); 
df = r if isinstance(r, pd.DataFrame) else r.table if hasattr(r,'table') else None
print(df.columns.tolist() if df is not None else r)
print(df.shape)
print(df.target.value_counts())
print(df.groupby(['condition','target']).size())


<class 'pandas.DataFrame'>
['target', 'score', 'effect', 'mean_difference', 'p', 'test', 'cells', 'reference_cells', 'samples', 'target_samples', 'q', 'condition']
(732, 12)
target
Tbx21     244
Rorc      244
Tgfbr1    244
Name: count, dtype: int64
condition              target
colitis                Rorc      122
                       Tbx21     122
                       Tgfbr1    122
nephrotoxic_nephritis  Rorc      122
                       Tbx21     122
                       Tgfbr1    122
dtype: int64

**summarise significant protein shifts per target and condition** (14:03, t8-second-readout), on thread t8: 122 antibody features (CD4, CD8a, CD366, PD-1, Ly-6C, CD11b) in the screen are unread. Do Tgfbr1, Rorc and Tbx21 shift surface proteins against control cells, and do they match RNA?

Why: per-target protein shifts with the largest effects, by condition

In [ ]:
import pandas as pd, numpy as np
pd.set_option('display.width',250); pd.set_option('display.max_rows',400)
df = pt.load("20261114T140122-C-programs-f84809:result")
df['sig']=df.q<0.05
s = df.groupby(['condition','target']).agg(n_sig=('sig','sum'), n_up=('effect',lambda x: ((x>0)&(df.loc[x.index,'sig'])).sum()), n_dn=('effect',lambda x: ((x<0)&(df.loc[x.index,'sig'])).sum()))
print(s)
for (c,t),g in df[df.sig].groupby(['condition','target']):
    g=g.sort_values('effect',key=abs,ascending=False)
    print(c,t, [(r.score.replace('_TotalSeqC',''), round(r.effect,3), round(r.q,4) if r.q>1e-10 else '<1e-10') for r in g.head(14).itertuples()])
print(df[['cells','reference_cells','samples','target_samples']].drop_duplicates().to_string())


                              n_sig  n_up  n_dn
condition             target                   
colitis               Rorc       50    31    19
                      Tbx21      31    18    13
                      Tgfbr1     25    14    11
nephrotoxic_nephritis Rorc       34    33     1
                      Tbx21      86    79     7
                      Tgfbr1    100    84    16
colitis Rorc [('CD55', -0.981, '<1e-10'), ('CD86', 0.747, '<1e-10'), ('CD69', 0.729, '<1e-10'), ('TCR-Vb8_1', 0.608, '<1e-10'), ('CD38', 0.511, '<1e-10'), ('CD49d', -0.369, '<1e-10'), ('CD54', 0.34, '<1e-10'), ('CXCR6', 0.322, 0.0), ('CD48', -0.309, '<1e-10'), ('CD43', -0.309, '<1e-10'), ('CD366', 0.303, 0.0), ('CD61', -0.272, '<1e-10'), ('IL-7RA', -0.247, 0.0), ('CD5', 0.246, 0.0)]
colitis Tbx21 [('CD55', -0.506, '<1e-10'), ('Integrin-beta7', 0.468, 0.0), ('CD69', 0.37, 0.0), ('CCR6', 0.315, '<1e-10'), ('CD103', 0.309, '<1e-10'), ('CD45RB', -0.273, '<1e-10'), ('TCR-Vb8_1', -0.258, 0.0), ('PD-1H', -0.249, '<1

**describe the screen and the states run's tables** (14:03, t9-states), on thread t9: Cell states in the screen and each target's share of them (survey C.states_and_composition still running). Tgfbr1 cells are enriched (0.72) and Rorc, Irf4, Jak1, Il7r depleted; is it a composition shift between Th17 states?

Why: start the thread's notebook

In [ ]:
print(pt.describe())

{'cells': 'every cell', 'n_cells': 173418, 'named_like': 'Colitis_2_23_2_Control:AAACCTGAGAAACCTA-1', 'per_sample': {'Colitis_2_23_2_Control': 14174, 'Colitis_2_23_2_Th17-1': 12110, 'Colitis_2_23_2_Th17-2': 5126, 'NTN_1_19_23_1': 13963, 'NTN_1_19_23_2': 9512, 'NTN_1_19_5': 12495, 'NTN_1_19_9_1': 12827, 'NTN_1_19_9_2': 12848, 'NTN_2_23_1_Control': 25658, 'NTN_2_23_1_Th17-1': 19511, 'NTN_2_23_1_Th17-2': 19811, 'NTN_2_23_1_Th17-3': 15383}, 'independent_units': True, 'labels': {'none': 89395, 'multi': 27409, 'failed QC': 15072, 'control': 4420, 'Tgfbr1': 3784, 'Il23r': 3400, 'Tbx21': 2216, 'Il17a': 2130, 'Cd40lg': 2008, 'Syk': 1973, 'Il6ra': 1814, 'Tgfbr2': 1759, 'Nfatc2': 1342, 'Stat3': 1328, 'Cebpb': 1254, 'Traf6': 1249, 'Myd88': 1170, 'Jak2': 1147, 'Ahr': 939, 'Il1r1': 887, 'Socs3': 885, 'Stat6': 882, 'Tlr4': 874, 'Ccr6': 861, 'Irf4': 825}, 'targets_with_10_cells': 28, 'control_guides': {'gRNA_control_18_capture': 717, 'gRNA_control_9_capture': 565, 'gRNA_control_20_capture': 463, 'gRNA

**read the by-unit table of the protein run** (14:03, t8-second-readout), on thread t8: 122 antibody features (CD4, CD8a, CD366, PD-1, Ly-6C, CD11b) in the screen are unread. Do Tgfbr1, Rorc and Tbx21 shift surface proteins against control cells, and do they match RNA?

Why: see how each target's shift splits across units (samples)

In [ ]:
import pandas as pd, numpy as np
pd.set_option('display.width',250); pd.set_option('display.max_rows',400); pd.set_option('display.max_columns',30)
bu = pt.load("20261114T140122-C-programs-f84809:by_unit")
print(bu.shape); print(bu.columns.tolist()); print(bu.head(10).to_string())


(3660, 6)
['target', 'score', 'unit', 'difference', 'cells', 'condition']
  target            score                   unit  difference  cells condition
0  Tbx21    CD4_TotalSeqC  Colitis_2_23_2_Th17-1    0.029848    210   colitis
1  Tbx21    CD4_TotalSeqC  Colitis_2_23_2_Th17-2   -0.040902     64   colitis
2  Tbx21   CD8a_TotalSeqC  Colitis_2_23_2_Th17-1    0.019982    210   colitis
3  Tbx21   CD8a_TotalSeqC  Colitis_2_23_2_Th17-2   -0.039583     64   colitis
4  Tbx21  CD366_TotalSeqC  Colitis_2_23_2_Th17-1    0.137164    210   colitis
5  Tbx21  CD366_TotalSeqC  Colitis_2_23_2_Th17-2    0.238712     64   colitis
6  Tbx21   PD-1_TotalSeqC  Colitis_2_23_2_Th17-1    0.040835    210   colitis
7  Tbx21   PD-1_TotalSeqC  Colitis_2_23_2_Th17-2    0.039483     64   colitis
8  Tbx21  Ly-6C_TotalSeqC  Colitis_2_23_2_Th17-1    0.015358    210   colitis
9  Tbx21  Ly-6C_TotalSeqC  Colitis_2_23_2_Th17-2   -0.065292     64   colitis

**read the states table and the cell states; build per-target share vs control, per unit** (14:03, t9-states), on thread t9: Cell states in the screen and each target's share of them (survey C.states_and_composition still running). Tgfbr1 cells are enriched (0.72) and Rorc, Irf4, Jak1, Il7r depleted; is it a composition shift between Th17 states?

Why: shares by state and by experiment (NTN2 vs NTN1 vs colitis)

In [ ]:
import pandas as pd, numpy as np
a = pt.screen().adata
o = a.obs
print(o.columns.tolist())
print(o['label_after_qc'].value_counts().head(40))
print(o['unit'].value_counts())
print(o['control_group'].value_counts().head())


['sample', 'unit', 'umis', 'genes', 'unit_call', 'label', 'guides', 'control_group', 'control_kind', 'condition', 'qc_reason', 'label_after_qc']
label_after_qc
none         89395
multi        27409
failed QC    15072
control       4420
Tgfbr1        3784
Il23r         3400
Tbx21         2216
Il17a         2130
Cd40lg        2008
Syk           1973
Il6ra         1814
Tgfbr2        1759
Nfatc2        1342
Stat3         1328
Cebpb         1254
Traf6         1249
Myd88         1170
Jak2          1147
Ahr            939
Il1r1          887
Socs3          885
Stat6          882
Tlr4           874
Ccr6           861
Irf4           825
Socs1          807
Runx1          751
Rorc           721
Il12rb         613
Nfkb1          532
Jak1           489
Il7r           482
Name: count, dtype: int64
unit
NTN_2_23_1_Control        25658
NTN_2_23_1_Th17-2         19811
NTN_2_23_1_Th17-1         19511
NTN_2_23_1_Th17-3         15383
Colitis_2_23_2_Control    14174
NTN_1_19_23_1             13963
NTN_1_19_

**name states: per-state fraction of cells expressing marker and Th17 genes** (14:04, t9-states), on thread t9: Cell states in the screen and each target's share of them (survey C.states_and_composition still running). Tgfbr1 cells are enriched (0.72) and Rorc, Irf4, Jak1, Il7r depleted; is it a composition shift between Th17 states?

Why: name states from markers and see where the Th17 genes sit

In [ ]:
import numpy as np, pandas as pd, scipy.sparse as sp
a = pt.screen().adata
o = a.obs.copy()
st = pt.load('cell_states') if False else None
# get state labels from states run table
import glob
genes = ['Cd4','Cd8a','Foxp3','Rorc','Il17a','Ccr6','Il23r','Irf4','Il7r','Jak1','Tgfbr1','Mki67','Top2a','Ccnb2','Trbv3','Trav6-1','Il12rb2','Ifit1','Isg15','Itk','Cd3e','Ptprc','Cd19','Itgam','Fcer1g']
genes=[g for g in genes if g in a.var_names]
print('genes', genes)
print(a.obs_names[:3].tolist())
print([c for c in a.obs.columns])


genes ['Cd4', 'Cd8a', 'Foxp3', 'Rorc', 'Il17a', 'Ccr6', 'Il23r', 'Irf4', 'Il7r', 'Jak1', 'Tgfbr1', 'Mki67', 'Top2a', 'Ccnb2', 'Trbv3', 'Trav6-1', 'Il12rb2', 'Ifit1', 'Isg15', 'Itk', 'Cd3e', 'Ptprc', 'Cd19', 'Itgam', 'Fcer1g']
['Colitis_2_23_2_Control:AAACCTGAGAAACCTA-1', 'Colitis_2_23_2_Control:AAACCTGAGAGTCGGT-1', 'Colitis_2_23_2_Control:AAACCTGAGATAGTCA-1']
['sample', 'unit', 'umis', 'genes', 'unit_call', 'label', 'guides', 'control_group', 'control_kind', 'condition', 'qc_reason', 'label_after_qc']

**Describe the screen: cells, labels, control guides and genes (thread t6 start)** (14:04), on thread t6: What the targets' changed genes are: Tgfbr1 217, Tbx21 314, Rorc 67, Socs1 144, Traf6 88 genes. Read them as programmes and published Th17 sets (pathogenic vs. regulatory, IL-23, TGF-beta, T-bet/IFNg).

Why: Start of thread t6, read what the cells hold

In [ ]:
print(pt.describe())

{'cells': 'every cell', 'n_cells': 173418, 'named_like': 'Colitis_2_23_2_Control:AAACCTGAGAAACCTA-1', 'per_sample': {'Colitis_2_23_2_Control': 14174, 'Colitis_2_23_2_Th17-1': 12110, 'Colitis_2_23_2_Th17-2': 5126, 'NTN_1_19_23_1': 13963, 'NTN_1_19_23_2': 9512, 'NTN_1_19_5': 12495, 'NTN_1_19_9_1': 12827, 'NTN_1_19_9_2': 12848, 'NTN_2_23_1_Control': 25658, 'NTN_2_23_1_Th17-1': 19511, 'NTN_2_23_1_Th17-2': 19811, 'NTN_2_23_1_Th17-3': 15383}, 'independent_units': True, 'labels': {'none': 89395, 'multi': 27409, 'failed QC': 15072, 'control': 4420, 'Tgfbr1': 3784, 'Il23r': 3400, 'Tbx21': 2216, 'Il17a': 2130, 'Cd40lg': 2008, 'Syk': 1973, 'Il6ra': 1814, 'Tgfbr2': 1759, 'Nfatc2': 1342, 'Stat3': 1328, 'Cebpb': 1254, 'Traf6': 1249, 'Myd88': 1170, 'Jak2': 1147, 'Ahr': 939, 'Il1r1': 887, 'Socs3': 885, 'Stat6': 882, 'Tlr4': 874, 'Ccr6': 861, 'Irf4': 825}, 'targets_with_10_cells': 28, 'control_guides': {'gRNA_control_18_capture': 717, 'gRNA_control_9_capture': 565, 'gRNA_control_20_capture': 463, 'gRNA

**load per-cell state labels from the states run** (14:04, t9-states), on thread t9: Cell states in the screen and each target's share of them (survey C.states_and_composition still running). Tgfbr1 cells are enriched (0.72) and Rorc, Irf4, Jak1, Il7r depleted; is it a composition shift between Th17 states?

Why: need each cell's state to split shares by experiment

In [ ]:
cs = pt.load('20261114T134310-C-states_and_composition-b254f4:cell_states')
print(type(cs))
try:
    print(cs.head()); print(cs.columns.tolist()); print(cs.shape)
except Exception as e: print(e)


<class 'pandas.DataFrame'>
                                     barcode state state_name     umap_1  \
0  Colitis_2_23_2_Control:AAACCTGAGAAACCTA-1   NaN        NaN        NaN   
1  Colitis_2_23_2_Control:AAACCTGAGAGTCGGT-1   S03        S03  10.283938   
2  Colitis_2_23_2_Control:AAACCTGAGATAGTCA-1   S10        S10  14.768799   
3  Colitis_2_23_2_Control:AAACCTGAGCGATCCC-1   S03        S03  14.212029   
4  Colitis_2_23_2_Control:AAACCTGAGGATGGTC-1   S03        S03  11.715388   

      umap_2  
0        NaN  
1  16.822245  
2  12.412437  
3  15.458179  
4  16.886192  
['barcode', 'state', 'state_name', 'umap_1', 'umap_2']
(173418, 5)

**Genes** (14:04), on thread t7: Does Rorc or Tbx21 knockout change Tgfbr1, Il23r, Ccr6, Stat3 or other targets' own genes? Il23r (-2.74) and Ccr6 (-2.15) show strong own-gene knockdown, and Il23r/Ccr6 fall in the Rorc KO.

Why: t7: each target's own target gene in the other targets' cells, as a network

27 of 28 targets change genes at q < 0.05 (26 targets by pseudobulk by capture (edgeR-QL), since 10 captures hold both the target's and the reference cells; 1 target by pseudobulk by capture (edgeR-QL), since 9 captures hold both the target...

![20261114T140409715-1.png](figures/20261114T140409715-1.png)

**per-state marker fractions, Th17 genes per state, and state share by target x unit vs control in unit** (14:04, t9-states), on thread t9: Cell states in the screen and each target's share of them (survey C.states_and_composition still running). Tgfbr1 cells are enriched (0.72) and Rorc, Irf4, Jak1, Il7r depleted; is it a composition shift between Th17 states?

Why: name states and split the composition by experiment

In [ ]:
import numpy as np, pandas as pd, scipy.sparse as sp
a = pt.screen().adata
cs = pt.load('20261114T134310-C-states_and_composition-b254f4:cell_states').set_index('barcode')
o = a.obs.copy()
o['state'] = cs['state'].reindex(o.index).values
o['exp'] = np.where(o['unit'].str.startswith('NTN_2'),'NTN2',np.where(o['unit'].str.startswith('NTN_1'),'NTN1','Colitis'))
X = a.X
genes = ['Cd4','Cd8a','Foxp3','Rorc','Il17a','Ccr6','Il23r','Irf4','Il7r','Jak1','Tgfbr1','Mki67','Top2a','Ccnb2','Trbv3','Trav6-1','Il12rb2','Ifit1','Itk','Cd3e','Ptprc','Cd19','Itgam','Fcer1g','Il2ra','Tbx21','Gzmb','Ifng']
genes=[g for g in genes if g in a.var_names]
idx = {g: a.var_names.get_loc(g) for g in genes}
Xg = X[:, [idx[g] for g in genes]]
Xg = Xg.toarray() if sp.issparse(Xg) else np.asarray(Xg)
df = pd.DataFrame(Xg>0, columns=genes, index=o.index)
df['state']=o['state'].values
frac = df.groupby('state').mean().round(2)
n = df.groupby('state').size()
frac['n']=n
pd.set_option('display.width',250); pd.set_option('display.max_columns',40)
print(frac.T.to_string())


state         S01       S02       S03       S04       S05       S06       S07      S08      S09      S10      S11      S12      S13      S14      S15      S16      S17      S18      S19     S20     S21    S22
Cd4          0.83      0.74      0.91      0.76      0.85      0.74      0.62     0.80     0.52     0.83     0.74     0.89     0.91     0.85     0.63     0.89     0.90     0.68     0.87    0.73    0.18   0.85
Cd8a         0.00      0.00      0.00      0.00      0.00      0.00      0.00     0.00     0.00     0.00     0.00     0.00     0.00     0.00     0.00     0.00     0.00     0.00     0.00    0.00    0.00   0.00
Foxp3        0.01      0.00      0.01      0.01      0.01      0.01      0.00     0.01     0.00     0.01     0.01     0.01     0.01     0.00     0.01     0.01     0.01     0.00     0.01    0.88    0.00   0.00
Rorc         0.56      0.30      0.31      0.22      0.10      0.16      0.14     0.55     0.22     0.21     0.19     0.26     0.36     0.10     0.11     0.31     0

**check how to calibrate the protein result against control-guide groups** (14:04, t8-second-readout), on thread t8: 122 antibody features (CD4, CD8a, CD366, PD-1, Ly-6C, CD11b) in the screen are unread. Do Tgfbr1, Rorc and Tbx21 shift surface proteins against control cells, and do they match RNA?

Why: find the control-guide line for the protein result

In [ ]:
import inspect
print(pt.help('calibrate') if hasattr(pt,'help') else '')
try:
    print(inspect.signature(pt.calibrate))
except Exception as e: print(e)


pt.calibrate(result, seed=0)
The result's own call with groups of control guides in place of the targets, the rest of the control cells
    as their reference: what the same analysis gives with no effect. A gene result's calibration also gives, in its
    extra['summary'], each group's discoveries, their 95th percentile and the targets beyond it. A gene result by the
    design's tests (method auto) is calibrated once for each test it used, each target read against its own test's
    groups.
(result, seed=0)

**state share per target by experiment, against control cells in the same experiment** (14:04, t9-states), on thread t9: Cell states in the screen and each target's share of them (survey C.states_and_composition still running). Tgfbr1 cells are enriched (0.72) and Rorc, Irf4, Jak1, Il7r depleted; is it a composition shift between Th17 states?

Why: test whether the Tgfbr1 composition shift splits by experiment as f3 does

In [ ]:
import numpy as np, pandas as pd
a = pt.screen().adata
cs = pt.load('20261114T134310-C-states_and_composition-b254f4:cell_states').set_index('barcode')
o = a.obs.copy()
o['state'] = cs['state'].reindex(o.index).values
o['exp'] = np.where(o['unit'].str.startswith('NTN_2'),'NTN2',np.where(o['unit'].str.startswith('NTN_1'),'NTN1','Colitis'))
o = o[o['label_after_qc']!='failed QC']
ctrl = o[o['label_after_qc']=='control']
rows=[]
for e in ['NTN2','NTN1','Colitis']:
    c = ctrl[ctrl.exp==e]
    cshare = c['state'].value_counts(normalize=True)
    for t in sorted(o['label_after_qc'].unique()):
        if t in ('control','none','multi','failed QC'): continue
        tt = o[(o.label_after_qc==t)&(o.exp==e)]
        if len(tt)<10: continue
        ts = tt['state'].value_counts(normalize=True)
        for s in sorted(set(cshare.index)|set(ts.index)):
            rows.append(dict(exp=e,target=t,state=s,n=len(tt),share=ts.get(s,0),ctrl=cshare.get(s,0),ctrl_n=len(c)))
d = pd.DataFrame(rows); d['diff']=d.share-d.ctrl
d.to_pickle('/data/pertura/workspace/t9_shares.pkl') if False else None
pt.save_table('t9_shares_by_exp', d)
pd.set_option('display.width',250)
# Tgfbr1 table
for e in ['NTN2','NTN1','Colitis']:
    sub = d[(d.exp==e)&(d.target=='Tgfbr1')].set_index('state')[['n','share','ctrl','diff']].round(3)
    print(e, 'Tgfbr1 n', sub.n.iloc[0] if len(sub) else 0, 'ctrl n', ctrl[ctrl.exp==e].shape[0])
    print(sub.T.to_string())
print(o[o.label_after_qc=='Tgfbr1'].exp.value_counts())


NTN2 Tgfbr1 n 2799 ctrl n 904
state       S01       S04       S05       S08       S09       S10       S11       S12       S13       S14       S15       S16       S17       S19       S20
n      2799.000  2799.000  2799.000  2799.000  2799.000  2799.000  2799.000  2799.000  2799.000  2799.000  2799.000  2799.000  2799.000  2799.000  2799.000
share     0.025     0.019     0.020     0.002     0.021     0.000     0.001     0.191     0.010     0.584     0.000     0.009     0.109     0.010     0.000
ctrl      0.291     0.119     0.127     0.012     0.321     0.002     0.000     0.033     0.031     0.019     0.001     0.024     0.009     0.007     0.003
diff     -0.266    -0.101    -0.107    -0.010    -0.300    -0.002     0.001     0.158    -0.021     0.565    -0.001    -0.016     0.100     0.004    -0.003
NTN1 Tgfbr1 n 810 ctrl n 2918
state      S02      S06      S07      S11      S18      S21
n      810.000  810.000  810.000  810.000  810.000  810.000
share    0.228    0.422    0.202    0.13

**run the calibration of the protein result on control-guide groups** (14:04, t8-second-readout), on thread t8: 122 antibody features (CD4, CD8a, CD366, PD-1, Ly-6C, CD11b) in the screen are unread. Do Tgfbr1, Rorc and Tbx21 shift surface proteins against control cells, and do they match RNA?

Why: get the control-guide line for the protein result

In [ ]:
import pandas as pd, numpy as np
pd.set_option('display.width',250); pd.set_option('display.max_rows',400)
res = pt.load("20261114T140122-C-programs-f84809")
print(type(res))
cal = pt.calibrate(res, seed=0)
print(type(cal))
if hasattr(cal,'extra'): print(cal.extra.keys() if isinstance(cal.extra,dict) else cal.extra)
for name in ['table','result','by_unit']:
    try:
        t = getattr(cal,name)
        print(name, type(t))
    except Exception as e: print(name, e)
print(cal if not hasattr(cal,'__dict__') else cal.__dict__.keys())


---------------------------------------------------------------------------
ValueError                                Traceback (most recent call last)
Cell In[10], line 13
      9 except NameError:
     10     pass
     11 import pandas as pd, numpy as np
     12 pd.set_option('display.width',250); pd.set_option('display.max_rows',400)
---> 13 res = pt.load("20261114T140122-C-programs-f84809")
     14 print(type(res))
     15 cal = pt.calibrate(res, seed=0)
     16 print(type(cal))

File /opt/pertura/src/pertura/blocks.py:264, in load(object_id)
    262     if os.path.exists(path):
    263         return pd.read_csv(path, sep="\t")
--> 264 raise ValueError(f"no object {oid!r}; read_record(what='objects') lists them")

ValueError: no object '20261114T140122-C-programs-f84809'; read_record(what='objects') lists them

**read the named-gene table and pull each target's own gene (target == gene) and the targets' own-gene lines** (14:04, t7-targets-on-each-other), on thread t7: Does Rorc or Tbx21 knockout change Tgfbr1, Il23r, Ccr6, Stat3 or other targets' own genes? Il23r (-2.74) and Ccr6 (-2.15) show strong own-gene knockdown, and Il23r/Ccr6 fall in the Rorc KO.

Why: own-gene values of each target in the other targets' cells, t7

In [ ]:
import pandas as pd
ng = pt.load('20261114T140031-C-genes-fd11eb:named_genes') if True else None
ng = ng.copy() if hasattr(ng,'copy') else pd.DataFrame(ng)
print(ng.shape)
own = ng[ng['gene']==ng['target']]
print(own[['target','effect','q','cells','reference_cells','changed']].to_string())


(648, 10)
Empty DataFrame
Columns: [target, effect, q, cells, reference_cells, changed]
Index: []

**inspect antibody count matrix and control and target cells** (14:04, t8-second-readout), on thread t8: 122 antibody features (CD4, CD8a, CD366, PD-1, Ly-6C, CD11b) in the screen are unread. Do Tgfbr1, Rorc and Tbx21 shift surface proteins against control cells, and do they match RNA?

Why: check the protein counts before a CLR test of the control line

In [ ]:
import pandas as pd, numpy as np, scipy.sparse as sp
a = pt.screen().adata
ab = a.obsm['antibody_counts']
print(type(ab))
X = ab.values if hasattr(ab,'values') else np.asarray(ab)
X = np.asarray(X, dtype=float)
print(X.shape, np.nanmin(X), np.nanmax(X), np.nanmean(X[:1000]))
print(np.isnan(X).sum())
print(a.obs.groupby('label_after_qc').size().loc[['Tgfbr1','Rorc','Tbx21','control']])
print(a.obs.loc[a.obs.label_after_qc=='control','guides'].value_counts().head(5))
print(a.obs.loc[a.obs.label_after_qc.isin(['Tgfbr1','Rorc','Tbx21']),'guides'].value_counts().head(8))


<class 'pandas.DataFrame'>
(173418, 122) 0.0 36648.0 14.094204918032787
0
label_after_qc
Tgfbr1     3784
Rorc        721
Tbx21      2216
control    4420
dtype: int64
guides
gRNA_control_18_capture:1    455
gRNA_control_9_capture:1     397
gRNA_control_20_capture:1    300
gRNA_control_1_capture:1     279
gRNA_control_19_capture:1    263
Name: count, dtype: int64
guides
gRNA_Tgfbr1_1_capture:1    2753
gRNA_Tbx21_1_capture:1     1083
gRNA_Tbx21_4_capture:1      371
gRNA_Tbx21_3_capture:1      348
gRNA_Tgfbr1_3_capture:1     316
gRNA_Rorc_3_capture:1       240
gRNA_Tgfbr1_5_capture:1     226
gRNA_Tbx21_5_capture:1      172
Name: count, dtype: int64

**Tgfbr1 NTN2 composition by capture and guide; and grouped state shares per target and experiment** (14:04, t9-states), on thread t9: Cell states in the screen and each target's share of them (survey C.states_and_composition still running). Tgfbr1 cells are enriched (0.72) and Rorc, Irf4, Jak1, Il7r depleted; is it a composition shift between Th17 states?

Why: check whether the NTN2 S14/S12/S17 excess is one capture or one guide, and summarise all targets

In [ ]:
import numpy as np, pandas as pd
a = pt.screen().adata
cs = pt.load('20261114T134310-C-states_and_composition-b254f4:cell_states').set_index('barcode')
o = a.obs.copy()
o['state'] = cs['state'].reindex(o.index).values
o['exp'] = np.where(o['unit'].str.startswith('NTN_2'),'NTN2',np.where(o['unit'].str.startswith('NTN_1'),'NTN1','Colitis'))
o = o[o['label_after_qc']!='failed QC']
groups = {'Th17core':['S01','S08','S02','S06'], 'cycling':['S11','S12','S17','S22'], 'clonalTCR_S14':['S14'], 'Treg_S20':['S20'], 'IFN':['S18','S19']}
for k,v in groups.items(): o[k]=o.state.isin(v)
ctrl=o[o.label_after_qc=='control']
out=[]
for e in ['NTN2','NTN1','Colitis']:
    c=ctrl[ctrl.exp==e]
    r={'target':'control','exp':e,'n':len(c)}
    for k in groups: r[k]=round(c[k].mean(),3)
    out.append(r)
    for t in sorted(o.label_after_qc.unique()):
        if t in ('control','none','multi','failed QC'): continue
        tt=o[(o.label_after_qc==t)&(o.exp==e)]
        if len(tt)<10: continue
        r={'target':t,'exp':e,'n':len(tt)}
        for k in groups: r[k]=round(tt[k].mean(),3)
        out.append(r)
G=pd.DataFrame(out)
pt.save_table('t9_group_shares', G)
pd.set_option('display.width',250); pd.set_option('display.max_rows',200)
print(G[G.exp=='NTN2'].to_string())
print(G[G.exp=='NTN1'].to_string())
print(G[G.exp=='Colitis'].to_string())
# Tgfbr1 NTN2 by unit
t=o[(o.label_after_qc=='Tgfbr1')&(o.exp=='NTN2')]
print(pd.crosstab(t.unit, t.state, normalize='index').round(2).to_string())
print(t.unit.value_counts())
print(pd.crosstab(t.guides, t.state).to_string())


     target   exp     n  Th17core  cycling  clonalTCR_S14  Treg_S20    IFN
0   control  NTN2   904     0.303    0.042          0.019     0.003  0.007
1       Ahr  NTN2   279     0.459    0.061          0.014     0.000  0.004
2      Ccr6  NTN2   371     0.469    0.100          0.003     0.003  0.005
3    Cd40lg  NTN2  1298     0.297    0.034          0.004     0.012  0.008
4     Cebpb  NTN2   640     0.167    0.162          0.008     0.000  0.006
5    Il12rb  NTN2   112     0.411    0.036          0.000     0.009  0.000
6     Il17a  NTN2  1069     0.483    0.040          0.013     0.004  0.010
7     Il1r1  NTN2   331     0.486    0.045          0.009     0.000  0.018
8     Il23r  NTN2   879     0.150    0.121          0.009     0.006  0.014
9     Il6ra  NTN2   915     0.161    0.036          0.003     0.002  0.014
10     Il7r  NTN2    83     0.470    0.084          0.000     0.000  0.024
11     Irf4  NTN2    57     0.333    0.053          0.018     0.018  0.018
12     Jak1  NTN2    27  

**build per-cell CLR protein values for the target and control cells** (14:04, t8-second-readout), on thread t8: 122 antibody features (CD4, CD8a, CD366, PD-1, Ly-6C, CD11b) in the screen are unread. Do Tgfbr1, Rorc and Tbx21 shift surface proteins against control cells, and do they match RNA?

Why: CLR-normalise the antibody counts per cell as the Seurat vignette does

In [ ]:
import pandas as pd, numpy as np
from scipy import stats
import statsmodels.api as sm
from statsmodels.stats.multitest import multipletests
a = pt.screen().adata
o = a.obs.copy()
X = np.asarray(a.obsm['antibody_counts'].values, dtype=float)
L = np.log1p(X)
gm = np.exp(L.mean(axis=1, keepdims=True))
CLR = np.log1p(X / gm)
prot = list(a.obsm['antibody_counts'].columns)
keep = o.label_after_qc.isin(['Tgfbr1','Rorc','Tbx21','control']).values
o = o[keep].copy(); CLR = CLR[keep]
o['cell_idx']=np.arange(len(o))
o['grp'] = np.where(o.label_after_qc=='control','control',o.label_after_qc)
print(o.groupby(['condition','grp','unit']).size().unstack(-1).fillna(0).astype(int).T.to_string())
np.save('/data/pertura/t8_clr.npy', CLR) if False else None
pt.save_table('t8_cells', o[['unit','condition','grp','label_after_qc','guides','control_group']].reset_index().rename(columns={'index':'barcode'}))


condition             colitis                      nephrotoxic_nephritis                     
grp                      Rorc Tbx21 Tgfbr1 control                  Rorc Tbx21 Tgfbr1 control
unit                                                                                         
Colitis_2_23_2_Th17-1      62   210    144     379                     0     0      0       0
Colitis_2_23_2_Th17-2     155    64     31     219                     0     0      0       0
NTN_1_19_23_1               0     0      0       0                   148   215    136     985
NTN_1_19_23_2               0     0      0       0                   108   114     83     545
NTN_1_19_5                  0     0      0       0                    83   120     75     403
NTN_1_19_9_1                0     0      0       0                    22    42    237     487
NTN_1_19_9_2                0     0      0       0                    33    70    279     498
NTN_2_23_1_Th17-1           0     0      0       0          

**own-target protein test vs non-targeting controls, OLS with unit covariate** (14:04, t8-second-readout), on thread t8: 122 antibody features (CD4, CD8a, CD366, PD-1, Ly-6C, CD11b) in the screen are unread. Do Tgfbr1, Rorc and Tbx21 shift surface proteins against control cells, and do they match RNA?

Why: replicate the protein test in the notebook to set the control-guide line beside it

In [ ]:
import pandas as pd, numpy as np
from statsmodels.stats.multitest import multipletests
a = pt.screen().adata
o = a.obs.copy()
X = np.asarray(a.obsm['antibody_counts'].values, dtype=float)
gm = np.exp(np.log1p(X).mean(axis=1, keepdims=True))
CLR = np.log1p(X / gm)
prot = list(a.obsm['antibody_counts'].columns)
keep = o.label_after_qc.isin(['Tgfbr1','Rorc','Tbx21','control']).values
o = o[keep].copy(); CLR = CLR[keep]; o['row']=np.arange(len(o))
o['grp']=o.label_after_qc
o.to_pickle('/tmp/t8_obs.pkl') if False else None
def ols_test(y_mat, flag, units):
    # y_mat: n x p ; flag: bool n ; units: array of unit labels; design: intercept + unit dummies + flag
    U = pd.get_dummies(units, drop_first=True).values.astype(float)
    D = np.column_stack([np.ones(len(flag)), U, flag.astype(float)])
    XtX_inv = np.linalg.pinv(D.T @ D)
    B = XtX_inv @ D.T @ y_mat
    res = y_mat - D @ B
    dfres = D.shape[0]-D.shape[1]
    s2 = (res**2).sum(axis=0)/dfres
    se = np.sqrt(s2 * XtX_inv[-1,-1])
    beta = B[-1]
    from scipy import stats
    t = beta/se
    p = 2*stats.t.sf(np.abs(t), dfres)
    return beta, p
rows=[]
for cond in ['colitis','nephrotoxic_nephritis']:
    sub = o[o.condition==cond]
    yc = CLR[sub.row.values]
    units = sub.unit.values
    for tgt in ['Tgfbr1','Rorc','Tbx21']:
        flag = (sub.grp==tgt).values
        ctl = (sub.grp=='control').values
        m = flag|ctl
        beta,p = ols_test(yc[m], flag[m], units[m])
        mean_c = yc[m & ctl].mean(axis=0); mean_t = yc[m&flag].mean(axis=0)
        q = multipletests(p, method='fdr_bh')[1]
        for j,pr in enumerate(prot):
            rows.append(dict(condition=cond,target=tgt,score=pr,effect=beta[j],mean_diff=mean_t[j]-mean_c[j],p=p[j],q=q[j],n_t=flag[m].sum(),n_c=(ctl&m).sum()))
R = pd.DataFrame(rows)
R.to_csv('/tmp/t8_own_test.csv', index=False) if False else None
pt.save_table('t8_own_test', R)
print(R.groupby(['condition','target']).apply(lambda g: pd.Series({'n_q05':(g.q<0.05).sum(),'n_up':((g.q<0.05)&(g.effect>0)).sum(),'n_dn':((g.q<0.05)&(g.effect<0)).sum()})).to_string())


                              n_q05  n_up  n_dn
condition             target                   
colitis               Rorc       52    33    19
                      Tbx21      30    17    13
                      Tgfbr1     19    10     9
nephrotoxic_nephritis Rorc       32    31     1
                      Tbx21      87    80     7
                      Tgfbr1    101    85    16

**matrix of target (rows) by named gene (columns): log effect and q, from the targets_own run; list the strongest edges and how many targets move each named gene** (14:05, t7-targets-on-each-other), on thread t7: Does Rorc or Tbx21 knockout change Tgfbr1, Il23r, Ccr6, Stat3 or other targets' own genes? Il23r (-2.74) and Ccr6 (-2.15) show strong own-gene knockdown, and Il23r/Ccr6 fall in the Rorc KO.

Why: t7: which targets move which other targets' own genes, past the control line

In [ ]:
import pandas as pd, numpy as np
ng = pt.load('20261114T140031-C-genes-fd11eb:named_genes')
ng = pd.DataFrame(ng)
eff = ng.pivot_table(index='target', columns='gene', values='effect')
q = ng.pivot_table(index='target', columns='gene', values='q')
ch = ng[ng['changed']==True]
print('changed edges (target->gene):', len(ch))
key = ['Il23r','Ccr6','Tgfbr1','Stat3','Rorc','Tbx21','Il17a','Il1r1','Socs1','Cebpb','Cd40lg','Ahr','Jak2','Tgfbr2','Il7r','Nfatc2']
print(eff.reindex(columns=key).round(2).to_string())
print()
print(q.reindex(columns=key).round(3).to_string())
print()
# how many targets (rows) move each named gene past q<0.05 
cnt = ch.groupby('gene')['target'].nunique().sort_values(ascending=False)
print(cnt.head(25).to_string())
print()
for g in ['Il23r','Ccr6','Tgfbr1','Stat3','Rorc','Tbx21']:
    s = ng[ng['gene']==g][['target','effect','q']].sort_values('effect')
    print(g, 'n targets', len(s), 'min', s.iloc[0].tolist(), 'max', s.iloc[-1].tolist())


changed edges (target->gene): 21
gene    Il23r  Ccr6  Tgfbr1  Stat3  Rorc  Tbx21  Il17a  Il1r1  Socs1  Cebpb  Cd40lg   Ahr  Jak2  Tgfbr2  Il7r  Nfatc2
target                                                                                                               
Ahr      0.76  0.33    0.11  -0.00  0.36  -0.17   0.56   0.38  -0.04   0.01    0.05   NaN  0.10    0.02 -0.05    0.18
Ccr6     0.27   NaN   -0.50  -0.08  0.08  -0.09   0.00   0.24  -0.03  -0.11    0.11  0.07 -0.38   -0.05  0.12   -0.06
Cd40lg  -0.16  0.17   -0.01   0.08  0.11  -0.07  -0.54  -0.07  -0.06  -0.04     NaN -0.00 -0.12    0.04  0.16   -0.03
Cebpb   -0.72 -0.29    0.04  -0.11 -0.50   0.18  -0.51  -0.53  -0.04    NaN   -0.03 -0.01  0.06   -0.08 -0.06   -0.17
Il12rb  -0.21  0.75    0.57  -0.02  0.32  -0.53   0.82   0.19   0.03   0.30    0.11  0.16 -0.06    0.03  0.42    0.06
Il17a    0.48  0.43   -0.35   0.01  0.35  -0.27    NaN   0.41   0.09   0.10    0.14  0.39 -0.11    0.05  0.13    0.08
Il1r1    0.02 -0.09    

**control-group null per experiment: TV distance of state shares for each control gRNA group vs the rest of controls; then targets' TV and per-guide Tgfbr1** (14:05, t9-states), on thread t9: Cell states in the screen and each target's share of them (survey C.states_and_composition still running). Tgfbr1 cells are enriched (0.72) and Rorc, Irf4, Jak1, Il7r depleted; is it a composition shift between Th17 states?

Why: set each target's composition shift beside the control-guide line within each experiment

In [ ]:
import numpy as np, pandas as pd
a = pt.screen().adata
cs = pt.load('20261114T134310-C-states_and_composition-b254f4:cell_states').set_index('barcode')
o = a.obs.copy()
o['state'] = cs['state'].reindex(o.index).values
o['exp'] = np.where(o['unit'].str.startswith('NTN_2'),'NTN2',np.where(o['unit'].str.startswith('NTN_1'),'NTN1','Colitis'))
o = o[(o['label_after_qc']!='failed QC') & o.state.notna()]
ctrl = o[o.label_after_qc=='control'].copy()
ctrl['cg'] = ctrl['control_group'].astype(str)
res=[]
for e in ['NTN2','NTN1','Colitis']:
    c = ctrl[ctrl.exp==e]
    states = sorted(o.state.unique())
    # null: each control group vs other controls in same exp
    groups = c.cg.value_counts()
    groups = groups[groups>=30].index.tolist()
    nulls=[]
    for g in groups:
        A = c[c.cg==g].state.value_counts(normalize=True).reindex(states, fill_value=0)
        B = c[c.cg!=g].state.value_counts(normalize=True).reindex(states, fill_value=0)
        nulls.append(0.5*np.abs(A-B).sum())
    ref = c.state.value_counts(normalize=True).reindex(states, fill_value=0)
    for t in sorted(o.label_after_qc.unique()):
        if t in ('control','none','multi','failed QC'): continue
        tt = o[(o.label_after_qc==t)&(o.exp==e)]
        if len(tt)<30: continue
        A = tt.state.value_counts(normalize=True).reindex(states, fill_value=0)
        tv = 0.5*np.abs(A-ref).sum()
        res.append(dict(exp=e,target=t,n=len(tt),TV=round(tv,3),ctrl_group_TV_median=round(np.median(nulls),3),ctrl_group_TV_max=round(np.max(nulls),3),n_ctrl_groups=len(groups)))
R=pd.DataFrame(res).sort_values(['exp','TV'],ascending=[True,False])
print(R.to_string())
# per guide Tgfbr1 NTN2 vs controls in NTN2 (TV and S14 share)
e='NTN2'
c=ctrl[ctrl.exp==e]; ref=c.state.value_counts(normalize=True)
for gname, sub in o[(o.label_after_qc=='Tgfbr1')&(o.exp==e)].groupby('guides'):
    if len(sub)<30: continue
    A=sub.state.value_counts(normalize=True).reindex(ref.index, fill_value=0)
    print(gname[:40], len(sub), 'TV', round(0.5*np.abs(A-ref.reindex(A.index,fill_value=0)).sum(),3), 'S14', round(A.get('S14',0),3))


        exp  target     n     TV  ctrl_group_TV_median  ctrl_group_TV_max  n_ctrl_groups
65  Colitis    Irf4   489  0.317                 0.180              0.225              9
73  Colitis   Socs1   294  0.267                 0.180              0.225              9
71  Colitis    Rorc   217  0.260                 0.180              0.225              9
68  Colitis   Myd88   253  0.245                 0.180              0.225              9
69  Colitis  Nfatc2   323  0.236                 0.180              0.225              9
60  Colitis   Il17a   713  0.221                 0.180              0.225              9
64  Colitis    Il7r    97  0.220                 0.180              0.225              9
82  Colitis   Traf6   463  0.218                 0.180              0.225              9
56  Colitis    Ccr6   128  0.216                 0.180              0.225              9
57  Colitis  Cd40lg   156  0.214                 0.180              0.225              9
74  Colitis   Socs3  

**control-guide line: pseudo-target groups of control guides against the other control cells** (14:05, t8-second-readout), on thread t8: 122 antibody features (CD4, CD8a, CD366, PD-1, Ly-6C, CD11b) in the screen are unread. Do Tgfbr1, Rorc and Tbx21 shift surface proteins against control cells, and do they match RNA?

Why: the control-guide groups' own shifts, the line each target must pass

In [ ]:
import pandas as pd, numpy as np
from statsmodels.stats.multitest import multipletests
a = pt.screen().adata
o = a.obs.copy()
X = np.asarray(a.obsm['antibody_counts'].values, dtype=float)
gm = np.exp(np.log1p(X).mean(axis=1, keepdims=True))
CLR = np.log1p(X / gm)
prot = list(a.obsm['antibody_counts'].columns)
keep = o.label_after_qc.isin(['Tgfbr1','Rorc','Tbx21','control']).values
o = o[keep].copy(); CLR = CLR[keep]; o['row']=np.arange(len(o))
from scipy import stats
def ols_test(y_mat, flag, units):
    U = pd.get_dummies(units, drop_first=True).values.astype(float)
    D = np.column_stack([np.ones(len(flag)), U, flag.astype(float)])
    XtX_inv = np.linalg.pinv(D.T @ D)
    B = XtX_inv @ D.T @ y_mat
    res = y_mat - D @ B
    dfres = D.shape[0]-D.shape[1]
    s2 = (res**2).sum(axis=0)/dfres
    se = np.sqrt(s2 * XtX_inv[-1,-1])
    t = B[-1]/se
    return B[-1], 2*stats.t.sf(np.abs(t), dfres)
ctl_all = o[o.grp=='control']
groups = sorted(ctl_all.control_group.unique(), key=lambda s:int(s.split('_')[2]))
print(groups)
rows=[]
for cond in ['colitis','nephrotoxic_nephritis']:
    sub = o[(o.condition==cond)&(o.grp=='control')]
    cg = sub.control_group.values
    gl = sorted(set(cg), key=lambda s:int(s.split('_')[2]))
    # three pseudo-target groups: round-robin over control guide ids
    assign = {g: i%3 for i,g in enumerate(gl)}
    yc = CLR[sub.row.values]; units=sub.unit.values
    for k in range(3):
        flag = np.array([assign[g]==k for g in cg])
        beta,p = ols_test(yc, flag, units)
        q = multipletests(p, method='fdr_bh')[1]
        for j,pr in enumerate(prot):
            rows.append(dict(condition=cond, group=f'ctrl_group{k}', score=pr, effect=beta[j], p=p[j], q=q[j], n=int(flag.sum())))
C = pd.DataFrame(rows)
pt.save_table('t8_control_line', C)
print(C.groupby(['condition','group']).apply(lambda g: pd.Series({'n_q05':(g.q<0.05).sum(),'max_abs_eff':g.effect.abs().max().round(3),'n':g.n.iloc[0]})).to_string())
print(C[C.q<0.05].assign(absE=lambda d: d.effect.abs()).sort_values('absE',ascending=False).head(20)[['condition','group','score','effect','q']].to_string())


---------------------------------------------------------------------------
AttributeError                            Traceback (most recent call last)
Cell In[14], line 33
     29     s2 = (res**2).sum(axis=0)/dfres
     30     se = np.sqrt(s2 * XtX_inv[-1,-1])
     31     t = B[-1]/se
     32     return B[-1], 2*stats.t.sf(np.abs(t), dfres)
---> 33 ctl_all = o[o.grp=='control']
     34 groups = sorted(ctl_all.control_group.unique(), key=lambda s:int(s.split('_')[2]))
     35 print(groups)
     36 rows=[]

File ~/pertura/envs/pertpy/lib/python3.12/site-packages/pandas/core/generic.py:6194, in NDFrame.__getattr__(self, name)
   6190             and name not in self._accessors
   6191             and self._info_axis._can_hold_identifiers_and_holds_name(name)
   6192         ):
   6193             return self[name]
-> 6194         return object.__getattribute__(self, name)

AttributeError: 'DataFrame' object has no attribute 'grp'

**NTN2 guide-1 vs guide-5 vs control: TCR V genes in S14 (clone check), and Th17 genes within states (composition vs within-state)** (14:05, t9-states), on thread t9: Cell states in the screen and each target's share of them (survey C.states_and_composition still running). Tgfbr1 cells are enriched (0.72) and Rorc, Irf4, Jak1, Il7r depleted; is it a composition shift between Th17 states?

Why: tell apart composition from a change inside each state, and check whether S14 is one clone

In [ ]:
import numpy as np, pandas as pd, scipy.sparse as sp
a = pt.screen().adata
cs = pt.load('20261114T134310-C-states_and_composition-b254f4:cell_states').set_index('barcode')
o = a.obs.copy()
o['state'] = cs['state'].reindex(o.index).values
o['exp'] = np.where(o['unit'].str.startswith('NTN_2'),'NTN2',np.where(o['unit'].str.startswith('NTN_1'),'NTN1','Colitis'))
o = o[(o['label_after_qc']!='failed QC') & o.state.notna()].copy()
o['grp'] = np.where(o.label_after_qc=='control','control', np.where(o.label_after_qc=='Tgfbr1', np.where(o.guides.astype(str).str.contains('Tgfbr1_1_capture'),'Tgfbr1_g1',np.where(o.guides.astype(str).str.contains('Tgfbr1_5_capture'),'Tgfbr1_g5','Tgfbr1_other')),'other'))
n2 = o[(o.exp=='NTN2')&o.grp.isin(['control','Tgfbr1_g1','Tgfbr1_g5'])].copy()
genes=['Trav6-1','Trbv3','Trbv2','Trav6-4','Il17a','Rorc','Ccr6','Il23r','Irf4','Il7r','Jak1','Tgfbr1','Il12rb2','Tbx21','Mki67','Ccnb2']
genes=[g for g in genes if g in a.var_names]
X = a[n2.index].X
Xd = X.toarray() if sp.issparse(X) else np.asarray(X)
D = pd.DataFrame(Xd, index=n2.index, columns=a.var_names)[genes]
D = np.log1p(D.div(a[n2.index].obs['umis'].values, axis=0)*1e4) if False else D
D['grp']=n2.grp.values; D['state']=n2.state.values
# TCR V genes in S14 by group
s14 = D[D.state=='S14']
print('S14 n by group', s14.grp.value_counts().to_dict())
print((s14.groupby('grp')[['Trav6-1','Trbv3']].apply(lambda x:(x>0).mean())).round(2).to_string())
# within-state mean of Th17 genes (raw counts, detected fraction and mean of log1p counts)
th=['Il17a','Rorc','Ccr6','Il23r','Irf4','Il7r','Jak1','Tgfbr1']
for s in ['S01','S08','S02','S06','S14','S12','S17']:
    sub=D[D.state==s]
    m=sub.groupby('grp')[th].mean().round(3)
    cnt=sub.grp.value_counts().to_dict()
    print(s, cnt); print(m.to_string())


S14 n by group {'Tgfbr1_g1': 1633, 'control': 17, 'Tgfbr1_g5': 1}
           Trav6-1  Trbv3
grp                      
Tgfbr1_g1     0.82   0.89
Tgfbr1_g5     1.00   1.00
control       0.53   0.65
S01 {'control': 263, 'Tgfbr1_g5': 42, 'Tgfbr1_g1': 19}
           Il17a   Rorc   Ccr6  Il23r   Irf4   Il7r   Jak1  Tgfbr1
grp                                                               
Tgfbr1_g1  0.053  1.158  1.263  0.947  0.316  3.053  3.263   0.105
Tgfbr1_g5  0.143  1.262  1.905  0.643  0.357  5.690  2.833   0.167
control    0.798  0.863  0.768  0.433  0.247  2.202  2.141   0.137
S08 {'control': 11, 'Tgfbr1_g1': 4}
           Il17a   Rorc   Ccr6  Il23r   Irf4   Il7r   Jak1  Tgfbr1
grp                                                               
Tgfbr1_g1  11.75  0.750  2.500  1.250  2.500  5.500  1.750   0.250
control     4.00  0.909  0.727  0.364  0.727  1.909  2.091   0.182
S02 {}
Empty DataFrame
Columns: [Il17a, Rorc, Ccr6, Il23r, Irf4, Il7r, Jak1, Tgfbr1]
Index: []
S06 {}
Empty Da

**control-guide line: pseudo-target groups of control guides against the other control cells** (14:05, t8-second-readout), on thread t8: 122 antibody features (CD4, CD8a, CD366, PD-1, Ly-6C, CD11b) in the screen are unread. Do Tgfbr1, Rorc and Tbx21 shift surface proteins against control cells, and do they match RNA?

Why: the control-guide groups' own shifts, the line each target must pass

In [ ]:
import pandas as pd, numpy as np
from statsmodels.stats.multitest import multipletests
from scipy import stats
a = pt.screen().adata
o = a.obs.copy()
X = np.asarray(a.obsm['antibody_counts'].values, dtype=float)
gm = np.exp(np.log1p(X).mean(axis=1, keepdims=True))
CLR = np.log1p(X / gm)
prot = list(a.obsm['antibody_counts'].columns)
keep = o.label_after_qc.isin(['Tgfbr1','Rorc','Tbx21','control']).values
o = o[keep].copy(); CLR = CLR[keep]; o['row']=np.arange(len(o))
def ols_test(y_mat, flag, units):
    U = pd.get_dummies(units, drop_first=True).values.astype(float)
    D = np.column_stack([np.ones(len(flag)), U, flag.astype(float)])
    XtX_inv = np.linalg.pinv(D.T @ D)
    B = XtX_inv @ D.T @ y_mat
    res = y_mat - D @ B
    dfres = D.shape[0]-D.shape[1]
    s2 = (res**2).sum(axis=0)/dfres
    se = np.sqrt(s2 * XtX_inv[-1,-1])
    t = B[-1]/se
    return B[-1], 2*stats.t.sf(np.abs(t), dfres)
rows=[]
for cond in ['colitis','nephrotoxic_nephritis']:
    sub = o[(o.condition==cond)&(o.label_after_qc=='control')]
    cg = sub.control_group.values
    gl = sorted(set(cg), key=lambda s:int(s.split('_')[2]))
    assign = {g: i%3 for i,g in enumerate(gl)}
    yc = CLR[sub.row.values]; units=sub.unit.values
    for k in range(3):
        flag = np.array([assign[g]==k for g in cg])
        beta,p = ols_test(yc, flag, units)
        q = multipletests(p, method='fdr_bh')[1]
        for j,pr in enumerate(prot):
            rows.append(dict(condition=cond, group=f'ctrl_group{k}', score=pr, effect=beta[j], p=p[j], q=q[j], n=int(flag.sum())))
C = pd.DataFrame(rows)
pt.save_table('t8_control_line', C)
print(C.groupby(['condition','group']).apply(lambda g: pd.Series({'n_q05':(g.q<0.05).sum(),'max_abs_eff':g.effect.abs().max().round(3),'n':g.n.iloc[0]})).to_string())
print(C[C.q<0.05].assign(absE=lambda d: d.effect.abs()).sort_values('absE',ascending=False).head(12)[['condition','group','score','effect','q']].to_string())


                                   n_q05  max_abs_eff       n
condition             group                                  
colitis               ctrl_group0   21.0        0.543   233.0
                      ctrl_group1   12.0        0.452   199.0
                      ctrl_group2    2.0        0.504   166.0
nephrotoxic_nephritis ctrl_group0   13.0        0.077  1074.0
                      ctrl_group1   62.0        0.086  1152.0
                      ctrl_group2    5.0        0.047  1596.0
    condition        group                  score    effect             q
107   colitis  ctrl_group0      TCR-Va2_TotalSeqC -0.542607  1.448254e-07
351   colitis  ctrl_group2      TCR-Va2_TotalSeqC  0.503573  3.276283e-05
147   colitis  ctrl_group1          CD5_TotalSeqC -0.451830  1.040707e-14
178   colitis  ctrl_group1    TCR-Vb8_1_TotalSeqC  0.401141  7.549579e-12
56    colitis  ctrl_group0    TCR-Vb8_1_TotalSeqC -0.313797  3.707755e-07
269   colitis  ctrl_group2          CD5_TotalSeqC  0.299707 

**Meaning** (14:05), on thread t6: What the targets' changed genes are: Tgfbr1 217, Tbx21 314, Rorc 67, Socs1 144, Traf6 88 genes. Read them as programmes and published Th17 sets (pathogenic vs. regulatory, IL-23, TGF-beta, T-bet/IFNg).

Why: Hallmark ORA on each target's changed genes

36 target x set pairs at adjusted p < 0.05 in 5 of 5 targets (each target's changed genes, up and down apart, against the genes tested for it; GSEApy, hallmark; q within each list; prior knowledge)

![20261114T140551185-1.png](figures/20261114T140551185-1.png)

**colitis: Irf4 composition by state vs controls, and the same for NTN1 Irf4; check what drives the TV past the colitis line** (14:05, t9-states), on thread t9: Cell states in the screen and each target's share of them (survey C.states_and_composition still running). Tgfbr1 cells are enriched (0.72) and Rorc, Irf4, Jak1, Il7r depleted; is it a composition shift between Th17 states?

Why: the other targets that pass the control line in colitis need their states named

In [ ]:
import numpy as np, pandas as pd
a = pt.screen().adata
cs = pt.load('20261114T134310-C-states_and_composition-b254f4:cell_states').set_index('barcode')
o = a.obs.copy()
o['state'] = cs['state'].reindex(o.index).values
o['exp'] = np.where(o['unit'].str.startswith('NTN_2'),'NTN2',np.where(o['unit'].str.startswith('NTN_1'),'NTN1','Colitis'))
o = o[(o['label_after_qc']!='failed QC') & o.state.notna()]
pd.set_option('display.width',250)
for e,tg in [('Colitis',['Irf4','Socs1','Rorc','Myd88','Il17a','Il7r','Jak1']),('NTN1',['Irf4','Jak1','Il7r','Rorc'])]:
    c=o[(o.exp==e)&(o.label_after_qc=='control')]
    cs_=c.state.value_counts(normalize=True).rename('ctrl')
    cols={'ctrl':cs_}
    for t in tg:
        tt=o[(o.exp==e)&(o.label_after_qc==t)]
        cols[t]=tt.state.value_counts(normalize=True)
        cols[t+'_n']=pd.Series({'n':len(tt)})
    T=pd.DataFrame(cols).fillna(0).round(3)
    print(e); print(T.to_string())


Colitis
      ctrl   Irf4  Irf4_n  Socs1  Socs1_n   Rorc  Rorc_n  Myd88  Myd88_n  Il17a  Il17a_n   Il7r  Il7r_n   Jak1  Jak1_n
S01  0.002  0.002     0.0  0.003      0.0  0.005     0.0  0.008      0.0  0.006      0.0  0.000     0.0  0.012     0.0
S03  0.522  0.832     0.0  0.769      0.0  0.756     0.0  0.735      0.0  0.721      0.0  0.722     0.0  0.671     0.0
S04  0.005  0.002     0.0  0.007      0.0  0.009     0.0  0.000      0.0  0.008      0.0  0.000     0.0  0.012     0.0
S05  0.003  0.002     0.0  0.003      0.0  0.005     0.0  0.000      0.0  0.008      0.0  0.000     0.0  0.000     0.0
S09  0.002  0.000     0.0  0.000      0.0  0.009     0.0  0.004      0.0  0.001      0.0  0.000     0.0  0.000     0.0
S10  0.318  0.112     0.0  0.139      0.0  0.129     0.0  0.134      0.0  0.189      0.0  0.175     0.0  0.159     0.0
S12  0.018  0.025     0.0  0.027      0.0  0.014     0.0  0.016      0.0  0.027      0.0  0.031     0.0  0.041     0.0
S13  0.002  0.000     0.0  0.003      0.

**per-unit protein differences of key proteins for each target against non-targeting cells** (14:05, t8-second-readout), on thread t8: 122 antibody features (CD4, CD8a, CD366, PD-1, Ly-6C, CD11b) in the screen are unread. Do Tgfbr1, Rorc and Tbx21 shift surface proteins against control cells, and do they match RNA?

Why: check the key protein shifts hold unit by unit, not carried by one sample

In [ ]:
import pandas as pd, numpy as np
pd.set_option('display.width',250)
a = pt.screen().adata
o = a.obs.copy()
X = np.asarray(a.obsm['antibody_counts'].values, dtype=float)
gm = np.exp(np.log1p(X).mean(axis=1, keepdims=True))
CLR = np.log1p(X / gm)
prot = list(a.obsm['antibody_counts'].columns)
keep = o.label_after_qc.isin(['Tgfbr1','Rorc','Tbx21','control']).values
o = o[keep].copy(); CLR = CLR[keep]; o['row']=np.arange(len(o))
picks = {'Rorc':['CD55_TotalSeqC','CD86_TotalSeqC','CD69_TotalSeqC','TCR-Vb8_1_TotalSeqC','CD38_TotalSeqC','CD49d_TotalSeqC','CD366_TotalSeqC'],
         'Tbx21':['CD55_TotalSeqC','Integrin-beta7_TotalSeqC','CD69_TotalSeqC','CD103_TotalSeqC','CD45_TotalSeqC','CD43_TotalSeqC'],
         'Tgfbr1':['TCR-Va2_TotalSeqC','CD45RB_TotalSeqC','CD49d_TotalSeqC','KLRG1_TotalSeqC','CD73_TotalSeqC','CD43_TotalSeqC']}
out=[]
for cond in ['colitis','nephrotoxic_nephritis']:
    sub=o[o.condition==cond]
    for tgt,pl in picks.items():
        for u in sorted(sub.unit.unique()):
            su = sub[sub.unit==u]
            if (su.label_after_qc==tgt).sum()<10 or (su.label_after_qc=='control').sum()<10: continue
            for pr in pl:
                j=prot.index(pr)
                t = CLR[su.row.values[su.label_after_qc==tgt], j].mean()
                c = CLR[su.row.values[su.label_after_qc=='control'], j].mean()
                out.append(dict(condition=cond,target=tgt,unit=u,protein=pr.replace('_TotalSeqC',''),diff=round(t-c,3),n_t=int((su.label_after_qc==tgt).sum()),n_c=int((su.label_after_qc=='control').sum())))
D=pd.DataFrame(out)
print(D.pivot_table(index=['condition','target','protein'],columns='unit',values='diff').to_string())
print(D.pivot_table(index=['condition','target'],columns='unit',values='n_t').to_string())


unit                                         Colitis_2_23_2_Th17-1  Colitis_2_23_2_Th17-2  NTN_1_19_23_1  NTN_1_19_23_2  NTN_1_19_5  NTN_1_19_9_1  NTN_1_19_9_2  NTN_2_23_1_Th17-1  NTN_2_23_1_Th17-2  NTN_2_23_1_Th17-3
condition             target protein                                                                                                                                                                                    
colitis               Rorc   CD366                           0.109                  0.417            NaN            NaN         NaN           NaN           NaN                NaN                NaN                NaN
                             CD38                            0.603                  0.457            NaN            NaN         NaN           NaN           NaN                NaN                NaN                NaN
                             CD49d                          -0.365                 -0.371            NaN            NaN         NaN 

**check whether the antibody counts are present in each unit** (14:06, t8-second-readout), on thread t8: 122 antibody features (CD4, CD8a, CD366, PD-1, Ly-6C, CD11b) in the screen are unread. Do Tgfbr1, Rorc and Tbx21 shift surface proteins against control cells, and do they match RNA?

Why: the NTN_1_19 units gave zero protein differences: test whether their antibody counts are empty

In [ ]:
import pandas as pd, numpy as np
a = pt.screen().adata
o = a.obs
X = np.asarray(a.obsm['antibody_counts'].values, dtype=float)
tot = X.sum(axis=1)
df = pd.DataFrame({'unit':o.unit.values,'tot':tot,'cond':o.condition.values,'lab':o.label_after_qc.values})
print(df.groupby('unit').agg(n=('tot','size'), zero_frac=('tot',lambda x:(x==0).mean()), med_total=('tot','median'), mean_total=('tot','mean')).round(2).to_string())


                            n  zero_frac  med_total  mean_total
unit                                                           
Colitis_2_23_2_Control  14174        0.0     1509.0     1727.61
Colitis_2_23_2_Th17-1   12110        0.0     1513.0     1716.65
Colitis_2_23_2_Th17-2    5126        0.0     2775.0     3134.52
NTN_1_19_23_1           13963        1.0        0.0        0.00
NTN_1_19_23_2            9512        1.0        0.0        0.00
NTN_1_19_5              12495        1.0        0.0        0.00
NTN_1_19_9_1            12827        1.0        0.0        0.00
NTN_1_19_9_2            12848        1.0        0.0        0.00
NTN_2_23_1_Control      25658        0.0     1294.0     1427.25
NTN_2_23_1_Th17-1       19511        0.0     1720.0     1863.13
NTN_2_23_1_Th17-2       19811        0.0     1516.0     1509.49
NTN_2_23_1_Th17-3       15383        0.0     1950.0     2097.75

**Programs** (14:06), on thread t6: What the targets' changed genes are: Tgfbr1 217, Tbx21 314, Rorc 67, Socs1 144, Traf6 88 genes. Read them as programmes and published Th17 sets (pathogenic vs. regulatory, IL-23, TGF-beta, T-bet/IFNg).

Why: Score the five targets on published Th17 sets (pathogenic, regulatory, IL-23, TGF-beta, T-bet/IFNg)

22 target x program pairs changed at adjusted p < 0.05 (scanpy scores; OLS, sample as a covariate); the same test on 3 groups of control guides: 6 of 15 group x program pairs changed at adjusted p < 0.05

![20261114T140608101-1.png](figures/20261114T140608101-1.png)

**NTN protein test restricted to the four units that hold antibody counts, with control groups on the same units** (14:06, t8-second-readout), on thread t8: 122 antibody features (CD4, CD8a, CD366, PD-1, Ly-6C, CD11b) in the screen are unread. Do Tgfbr1, Rorc and Tbx21 shift surface proteins against control cells, and do they match RNA?

Why: the NTN units without antibody counts dilute the control line and the target test; rerun on the units with protein

In [ ]:
import pandas as pd, numpy as np
from statsmodels.stats.multitest import multipletests
from scipy import stats
pd.set_option('display.width',250)
a = pt.screen().adata
o0 = a.obs
print(o0[o0.unit=='NTN_2_23_1_Control'].label_after_qc.value_counts().head(8).to_string())
X = np.asarray(a.obsm['antibody_counts'].values, dtype=float)
gm = np.exp(np.log1p(X).mean(axis=1, keepdims=True))
CLR = np.log1p(X / gm)
prot = list(a.obsm['antibody_counts'].columns)
o = o0.copy(); o['row']=np.arange(len(o))
sel = o.label_after_qc.isin(['Tgfbr1','Rorc','Tbx21','control']) & (o.condition=='nephrotoxic_nephritis') & (o.unit.str.startswith('NTN_2_23_1'))
sub = o[sel]; yc = CLR[sub.row.values]; units = sub.unit.values
def ols_test(y_mat, flag, units):
    U = pd.get_dummies(units, drop_first=True).values.astype(float)
    D = np.column_stack([np.ones(len(flag)), U, flag.astype(float)])
    XtX_inv = np.linalg.pinv(D.T @ D)
    B = XtX_inv @ D.T @ y_mat
    res = y_mat - D @ B
    dfres = D.shape[0]-D.shape[1]
    s2 = (res**2).sum(axis=0)/dfres
    se = np.sqrt(s2 * XtX_inv[-1,-1])
    return B[-1], 2*stats.t.sf(np.abs(B[-1]/se), dfres)
rows=[]
lab = sub.label_after_qc.values
ctl = lab=='control'
print('ctrl cells', ctl.sum(), 'units', pd.Series(units[ctl]).value_counts().to_dict())
for tgt in ['Tgfbr1','Rorc','Tbx21']:
    flag = lab==tgt; m = flag|ctl
    beta,p = ols_test(yc[m], flag[m], units[m])
    q = multipletests(p, method='fdr_bh')[1]
    for j,pr in enumerate(prot):
        rows.append(dict(condition='nephrotoxic_nephritis (protein units)', target=tgt, score=pr, effect=beta[j], q=q[j], n_t=int(flag.sum()), n_c=int(ctl.sum())))
R2 = pd.DataFrame(rows)
# control line on the same units
cg = sub.control_group.values
cmask = ctl
gl = sorted(set(cg[cmask]), key=lambda s:int(s.split('_')[2]))
assign = {g:i%3 for i,g in enumerate(gl)}
crow=[]
for k in range(3):
    flag = np.array([assign.get(g,-1)==k for g in cg]) & cmask
    m = cmask
    beta,p = ols_test(yc[m], flag[m], units[m])
    q = multipletests(p, method='fdr_bh')[1]
    crow.append(dict(group=k, n_q05=int((q<0.05).sum()), max_abs=float(np.abs(beta).max()), n=int(flag.sum())))
print(pd.DataFrame(R2).groupby('target').apply(lambda g: pd.Series({'n_q05':(g.q<0.05).sum(),'max_abs':g.effect.abs().max()})).round(3))
print(pd.DataFrame(crow))
pt.save_table('t8_ntn_protein_units', R2)


label_after_qc
none         24648
failed QC     1010
ctrl cells 904 units {'NTN_2_23_1_Th17-2': 462, 'NTN_2_23_1_Th17-3': 226, 'NTN_2_23_1_Th17-1': 216}
        n_q05  max_abs
target                
Rorc     32.0    0.651
Tbx21    82.0    0.675
Tgfbr1  100.0    1.724
   group  n_q05   max_abs    n
0      0      7  0.331424  537
1      1      0  0.273796  122
2      2     19  0.587221  245
'table:t8_ntn_protein_units'


**Genes** (14:06), on thread t8: 122 antibody features (CD4, CD8a, CD366, PD-1, Ly-6C, CD11b) in the screen are unread. Do Tgfbr1, Rorc and Tbx21 shift surface proteins against control cells, and do they match RNA?

Why: RNA of the same targets on the same cells, to set beside the protein shifts

by condition, each value apart: colitis: 3 of 3 targets change genes at q < 0.05 (3 targets by a test on cells (Wilcoxon, the field's most common) against the control cells of the target's own samples, in its mix, since only 2 captures hold...

![20261114T140642621-1.png](figures/20261114T140642621-1.png)

**RNA of the protein-matched genes against control cells, same cells and unit covariate** (14:07, t8-second-readout), on thread t8: 122 antibody features (CD4, CD8a, CD366, PD-1, Ly-6C, CD11b) in the screen are unread. Do Tgfbr1, Rorc and Tbx21 shift surface proteins against control cells, and do they match RNA?

Why: RNA effects of the genes that encode the measured proteins, to set beside the protein shifts

In [ ]:
import pandas as pd, numpy as np
from scipy import stats
from statsmodels.stats.multitest import multipletests
pd.set_option('display.width',250); pd.set_option('display.max_rows',500)
a = pt.screen().adata
o = a.obs.copy(); o['row']=np.arange(len(o))
keep = o.label_after_qc.isin(['Tgfbr1','Rorc','Tbx21','control'])
o = o[keep].copy()
LN = pt.screen().lognorm
gmap = {'CD4':'Cd4','CD8a':'Cd8a','CD366':'Havcr2','PD-1':'Pdcd1','Ly-6C':'Ly6c1','CD11b':'Itgam','Ly-6G':'Ly6g','CD49f':'Itga6','CD44':'Cd44','CD54':'Icam1','CD90_2':'Thy1','CD73':'Nt5e','CD49d':'Itga4','CD200':'Cd200','CD45':'Ptprc','CD45RB':'Ptprc','CD45_2':'Ptprc','CD25':'Il2ra','CD11c':'Itgax','CD43':'Spn','CD5':'Cd5','CD62L':'Sell','CD93':'Cd93','KLRG1':'Klrg1','CD3':'Cd3e','ICOS':'Icos','PD-L1':'Cd274','CD27':'Cd27','CD357':'Tnfrsf18','CD137':'Tnfrsf9','CD134':'Tnfrsf4','CD69':'Cd69','IL-7RA':'Il7r','CD86':'Cd86','CD103':'Itgae','CD150':'Slamf1','Integrin-beta7':'Itgb7','CXCR3':'Cxcr3','CD49b':'Itga2','CD226':'Cd226','CXCR6':'Cxcr6','Ly108':'Slamf6','CD98':'Slc3a2','CD94':'Klrd1','CD29':'Itgb1','CD38':'Cd38','CD55':'Cd55','CD48':'Cd48','CCR6':'Ccr6','CD61':'Itgb3','CD51':'Itgav','CD223':'Lag3','CD272':'Btla','CD2':'Cd2','CD71':'Tfrc','Ly-6A_E':'Ly6a','CD160':'Cd160','CD9':'Cd9','CD81':'Cd81','CD4':'Cd4','TCR-b-chain':'Trbc1','CD8b':'Cd8b1','CD160':'Cd160','CD150':'Slamf1','CD106':'Vcam1','CD24':'Cd24a','CD63':'Cd63','CD107a':'Lamp1'}
rows=[]
genes_needed = sorted(set(gmap.values()))
present = [g for g in genes_needed if g in a.var_names]
print('missing genes', [g for g in genes_needed if g not in a.var_names])
ci = {g:i for i,g in enumerate(a.var_names)}
sub_rows = o.row.values
M = LN[sub_rows][:, [ci[g] for g in present]].toarray()
G = pd.DataFrame(M, columns=present, index=o.index)
G = pd.concat([G, o[['label_after_qc','condition','unit']]], axis=1)
def ols_flag(y, flag, units):
    U = pd.get_dummies(units, drop_first=True).values.astype(float)
    D = np.column_stack([np.ones(len(flag)), U, flag.astype(float)])
    XtX_inv = np.linalg.pinv(D.T @ D)
    B = XtX_inv @ D.T @ y
    res = y - D @ B
    dfres = D.shape[0]-D.shape[1]
    s2 = (res**2).sum(axis=0)/dfres
    se = np.sqrt(s2 * XtX_inv[-1,-1])
    return B[-1], 2*stats.t.sf(np.abs(B[-1]/se), dfres)
for cond in ['colitis','nephrotoxic_nephritis']:
    s = G[G.condition==cond]
    for tgt in ['Tgfbr1','Rorc','Tbx21']:
        m = s.label_after_qc.isin([tgt,'control']).values
        flag = (s.label_after_qc==tgt).values[m]
        y = s[present].values[m]
        beta,p = ols_flag(y, flag, s.unit.values[m])
        for j,g in enumerate(present):
            rows.append(dict(condition=cond,target=tgt,gene=g,rna_effect=beta[j],rna_p=p[j]))
RN = pd.DataFrame(rows)
pt.save_table('t8_rna_own_test', RN)
print(RN.groupby(['condition','target']).apply(lambda g: pd.Series({'n_p05':(g.rna_p<0.05).sum()})))
print(RN[RN.gene.isin(['Tgfbr1','Rorc','Tbx21'])].to_string())


missing genes []
                              n_p05
condition             target       
colitis               Rorc       28
                      Tbx21      25
                      Tgfbr1     16
nephrotoxic_nephritis Rorc       25
                      Tbx21      32
                      Tgfbr1     47
Empty DataFrame
Columns: [condition, target, gene, rna_effect, rna_p]
Index: []

**protein effects beside matched RNA effects per target and condition** (14:07, t8-second-readout), on thread t8: 122 antibody features (CD4, CD8a, CD366, PD-1, Ly-6C, CD11b) in the screen are unread. Do Tgfbr1, Rorc and Tbx21 shift surface proteins against control cells, and do they match RNA?

Why: set each target's protein shifts beside its matched RNA, per condition

In [ ]:
import pandas as pd, numpy as np
from scipy import stats
pd.set_option('display.width',250); pd.set_option('display.max_rows',500)
gmap = {'CD4':'Cd4','CD8a':'Cd8a','CD366':'Havcr2','PD-1':'Pdcd1','Ly-6C':'Ly6c1','CD11b':'Itgam','Ly-6G':'Ly6g','CD49f':'Itga6','CD44':'Cd44','CD54':'Icam1','CD90_2':'Thy1','CD73':'Nt5e','CD49d':'Itga4','CD200':'Cd200','CD45':'Ptprc','CD45RB':'Ptprc','CD45_2':'Ptprc','CD25':'Il2ra','CD11c':'Itgax','CD43':'Spn','CD5':'Cd5','CD62L':'Sell','CD93':'Cd93','KLRG1':'Klrg1','CD3':'Cd3e','ICOS':'Icos','PD-L1':'Cd274','CD27':'Cd27','CD357':'Tnfrsf18','CD137':'Tnfrsf9','CD134':'Tnfrsf4','CD69':'Cd69','IL-7RA':'Il7r','CD86':'Cd86','CD103':'Itgae','CD150':'Slamf1','Integrin-beta7':'Itgb7','CXCR3':'Cxcr3','CD49b':'Itga2','CD226':'Cd226','CXCR6':'Cxcr6','Ly108':'Slamf6','CD98':'Slc3a2','CD94':'Klrd1','CD29':'Itgb1','CD38':'Cd38','CD55':'Cd55','CD48':'Cd48','CCR6':'Ccr6','CD61':'Itgb3','CD51':'Itgav','CD223':'Lag3','CD272':'Btla','CD2':'Cd2','CD71':'Tfrc','Ly-6A_E':'Ly6a','CD160':'Cd160','CD9':'Cd9','CD81':'Cd81','TCR-b-chain':'Trbc1','CD8b':'Cd8b1','CD106':'Vcam1','CD24':'Cd24a','CD63':'Cd63','CD107a':'Lamp1'}
P_all = pt.load('table:t8_own_test')
P_ntn = pt.load('table:t8_ntn_protein_units')
RN = pt.load('table:t8_rna_own_test')
P_all['protein']=P_all.score.str.replace('_TotalSeqC','',regex=False)
P_ntn['protein']=P_ntn.score.str.replace('_TotalSeqC','',regex=False)
rows=[]
for cond,P in [('colitis',P_all[P_all.condition=='colitis']),('nephrotoxic_nephritis',P_ntn)]:
    for tgt in ['Tgfbr1','Rorc','Tbx21']:
        p = P[P.target==tgt][['protein','effect','q']].copy()
        p['gene']=p.protein.map(gmap)
        r = RN[(RN.condition==cond)&(RN.target==tgt)][['gene','rna_effect','rna_p']]
        m = p.merge(r, on='gene', how='inner')
        m = m.drop_duplicates('protein')
        rho, pr = stats.spearmanr(m.effect, m.rna_effect)
        sig = m[(m.q<0.05)]
        agree = (np.sign(sig.effect)==np.sign(sig.rna_effect)).mean() if len(sig) else np.nan
        rows.append(dict(condition=cond,target=tgt,n_pairs=len(m),spearman=round(rho,3),p=round(pr,4),n_protein_sig=len(sig),sign_agree_sig=round(agree,2) if len(sig) else None, n_protein_sig_rna_p05=int(((sig.rna_p<0.05)).sum())))
        print(cond,tgt)
        print(m.sort_values('effect',key=abs,ascending=False).head(12).round(3).to_string())
print(pd.DataFrame(rows).to_string())


colitis Tgfbr1
           protein  effect      q   gene  rna_effect  rna_p
17            CD43  -0.303  0.000    Spn      -0.095  0.089
37  Integrin-beta7  -0.277  0.006  Itgb7      -0.107  0.073
62          CD45RB   0.231  0.000  Ptprc       0.030  0.559
33            CD86   0.211  0.056   Cd86      -0.003  0.891
21     TCR-b-chain  -0.207  0.000  Trbc1       0.142  0.003
46            CD38   0.206  0.003   Cd38       0.050  0.148
18             CD5   0.204  0.002    Cd5       0.219  0.001
44            CD48  -0.170  0.000   Cd48       0.039  0.522
11            CD73  -0.166  0.007   Nt5e      -0.075  0.097
31            CD69   0.165  0.057   Cd69       0.339  0.000
7            CD49f   0.162  0.002  Itga6      -0.002  0.923
61            CD94   0.142  0.000  Klrd1       0.076  0.028
colitis Rorc
   protein  effect    q    gene  rna_effect  rna_p
47    CD55  -0.981  0.0    Cd55      -0.114  0.000
33    CD86   0.747  0.0    Cd86       0.102  0.000
31    CD69   0.729  0.0    Cd69       0

**baseline protein-RNA concordance for control-guide pseudo-groups, same mapping** (14:08, t8-second-readout), on thread t8: 122 antibody features (CD4, CD8a, CD366, PD-1, Ly-6C, CD11b) in the screen are unread. Do Tgfbr1, Rorc and Tbx21 shift surface proteins against control cells, and do they match RNA?

Why: the control line for protein-RNA concordance: does a control group match its own RNA as well as a target does

In [ ]:
import pandas as pd, numpy as np
from scipy import stats
a = pt.screen().adata
o = a.obs.copy(); o['row']=np.arange(len(o))
gmap = {'CD4':'Cd4','CD8a':'Cd8a','CD366':'Havcr2','PD-1':'Pdcd1','Ly-6C':'Ly6c1','CD11b':'Itgam','Ly-6G':'Ly6g','CD49f':'Itga6','CD44':'Cd44','CD54':'Icam1','CD90_2':'Thy1','CD73':'Nt5e','CD49d':'Itga4','CD200':'Cd200','CD45':'Ptprc','CD45RB':'Ptprc','CD45_2':'Ptprc','CD25':'Il2ra','CD11c':'Itgax','CD43':'Spn','CD5':'Cd5','CD62L':'Sell','CD93':'Cd93','KLRG1':'Klrg1','CD3':'Cd3e','ICOS':'Icos','PD-L1':'Cd274','CD27':'Cd27','CD357':'Tnfrsf18','CD137':'Tnfrsf9','CD134':'Tnfrsf4','CD69':'Cd69','IL-7RA':'Il7r','CD86':'Cd86','CD103':'Itgae','CD150':'Slamf1','Integrin-beta7':'Itgb7','CXCR3':'Cxcr3','CD49b':'Itga2','CD226':'Cd226','CXCR6':'Cxcr6','Ly108':'Slamf6','CD98':'Slc3a2','CD94':'Klrd1','CD29':'Itgb1','CD38':'Cd38','CD55':'Cd55','CD48':'Cd48','CCR6':'Ccr6','CD61':'Itgb3','CD51':'Itgav','CD223':'Lag3','CD272':'Btla','CD2':'Cd2','CD71':'Tfrc','Ly-6A_E':'Ly6a','CD160':'Cd160','CD9':'Cd9','CD81':'Cd81','TCR-b-chain':'Trbc1','CD8b':'Cd8b1','CD106':'Vcam1','CD24':'Cd24a','CD63':'Cd63','CD107a':'Lamp1'}
prot = list(a.obsm['antibody_counts'].columns)
X = np.asarray(a.obsm['antibody_counts'].values, dtype=float)
CLR = np.log1p(X / np.exp(np.log1p(X).mean(axis=1, keepdims=True)))
present = sorted(set(gmap.values()))
ci = {g:i for i,g in enumerate(a.var_names)}
LN = pt.screen().lognorm
def ols_flag(y, flag, units):
    U = pd.get_dummies(units, drop_first=True).values.astype(float)
    D = np.column_stack([np.ones(len(flag)), U, flag.astype(float)])
    XtX_inv = np.linalg.pinv(D.T @ D); B = XtX_inv @ D.T @ y
    res = y - D @ B; dfres = D.shape[0]-D.shape[1]
    s2 = (res**2).sum(axis=0)/dfres
    return B[-1]
out=[]
pm = [p for p in prot if p.replace('_TotalSeqC','') in gmap]
for cond in ['colitis','nephrotoxic_nephritis']:
    base = o[(o.condition==cond)&(o.label_after_qc.isin(['control','Tgfbr1','Rorc','Tbx21']))].copy()
    if cond=='nephrotoxic_nephritis':
        base = base[base.unit.str.startswith('NTN_2_23_1')]
    ctl = base[base.label_after_qc=='control']
    gl = sorted(set(ctl.control_group), key=lambda s:int(s.split('_')[2]))
    assign = {g:i%3 for i,g in enumerate(gl)}
    G = LN[base.row.values][:, [ci[g] for g in present]].toarray()
    Gd = pd.DataFrame(G, columns=present)
    Pd = pd.DataFrame(CLR[base.row.values], columns=prot)
    for k in range(3):
        flag_all = np.array([(assign.get(g,-1)==k) for g in base.control_group]) & (base.label_after_qc=='control').values
        m = (base.label_after_qc=='control').values
        flag = flag_all[m]
        bp = ols_flag(Pd[pm].values[m], flag, base.unit.values[m])
        br = ols_flag(Gd.values[m], flag, base.unit.values[m])
        pe = dict(zip(pm,bp)); re_ = dict(zip(present,br))
        pairs = [(pe[p], re_[gmap[p.replace('_TotalSeqC','')]]) for p in pm if p.replace('_TotalSeqC','') in gmap]
        pa = np.array(pairs)
        rho,pv = stats.spearmanr(pa[:,0], pa[:,1])
        out.append(dict(condition=cond, pseudo_group=k, n_cells=int(flag.sum()), spearman=round(rho,3), n_pairs=len(pa)))
print(pd.DataFrame(out).to_string())


               condition  pseudo_group  n_cells  spearman  n_pairs
0                colitis             0      233     0.402       65
1                colitis             1      199     0.319       65
2                colitis             2      166     0.269       65
3  nephrotoxic_nephritis             0      537     0.311       65
4  nephrotoxic_nephritis             1      122     0.106       65
5  nephrotoxic_nephritis             2      245     0.328       65

**Genes** (14:08), on thread t7: Does Rorc or Tbx21 knockout change Tgfbr1, Il23r, Ccr6, Stat3 or other targets' own genes? Il23r (-2.74) and Ccr6 (-2.15) show strong own-gene knockdown, and Il23r/Ccr6 fall in the Rorc KO.

Why: t7: the named targets' genes for Rorc, Tbx21, Socs1 and Tgfbr1, each condition apart (colitis and nephrotoxic nephritis)

by condition, each value apart: colitis: 4 of 4 targets change genes at q < 0.05 (4 targets by a test on cells (Wilcoxon, the field's most common) against the control cells of the target's own samples, in its mix, since only 2 captures hold...

![20261114T140831279-1.png](figures/20261114T140831279-1.png)

**Common response** (14:08), on thread t7: Does Rorc or Tbx21 knockout change Tgfbr1, Il23r, Ccr6, Stat3 or other targets' own genes? Il23r (-2.74) and Ccr6 (-2.15) show strong own-gene knockdown, and Il23r/Ccr6 fall in the Rorc KO.

Why: t7: is the TCR-gene and Il23r/Ccr6 response shared across targets (the common response) or specific to Rorc/Tbx21

25 of 28 targets differ from the other targets' cells (drawn at random (seeded) to 3 times each target's cells, at least 500, then each side to at most 1000 cells), which hold any response the perturbations share; 25 of 28 targets differ fr...

![20261114T140832083-1.png](figures/20261114T140832083-1.png)

**how many of each target's changed genes are TCR V/J/C segments (Trbv, Trav, Trgv, Tcrg, Trdv) and how many targets share them; compare with the other genes** (14:08, t7-targets-on-each-other), on thread t7: Does Rorc or Tbx21 knockout change Tgfbr1, Il23r, Ccr6, Stat3 or other targets' own genes? Il23r (-2.74) and Ccr6 (-2.15) show strong own-gene knockdown, and Il23r/Ccr6 fall in the Rorc KO.

Why: t7: check whether the many targets' changes are a shared TCR-segment response (clonal composition) rather than target-specific edges

In [ ]:
import re
cg = pd.DataFrame(pt.load('20261114T140031-C-genes-fd11eb:changed_genes'))
print(cg.columns.tolist(), cg.shape)
gcol = 'gene' if 'gene' in cg.columns else cg.columns[1]
tcr = re.compile(r'^(Trbv|Trav|Trgv|Tcrg|Trdv|Trbj|Traj|Trbc|Trac|Tcrd)')
cg['is_tcr'] = cg[gcol].astype(str).str.match(tcr)
summ = cg[cg['changed']==True] if 'changed' in cg.columns else cg
g = summ.groupby('target').agg(n=(gcol,'size'), tcr=('is_tcr','sum'))
g['tcr_frac'] = (g['tcr']/g['n']).round(2)
print(g.sort_values('n', ascending=False).to_string())
shared = summ[summ['is_tcr']].groupby(gcol)['target'].nunique().sort_values(ascending=False)
print('TCR genes changed in how many targets:'); print(shared.head(20).to_string())
nontcr = summ[~summ['is_tcr']].groupby(gcol)['target'].nunique().sort_values(ascending=False)
print('non-TCR genes changed in how many targets:'); print(nontcr.head(20).to_string())


['target', 'gene', 'effect', 'p', 'q', 'test', 'cells', 'reference_cells'] (1194, 8)
          n  tcr  tcr_frac
target                    
Tbx21   314    5      0.02
Tgfbr1  217    7      0.03
Socs1   144    6      0.04
Traf6    88    5      0.06
Rorc     67    1      0.01
Tgfbr2   64    5      0.08
Jak2     61    3      0.05
Ccr6     44    3      0.07
Ahr      34    4      0.12
Il17a    33    6      0.18
Cd40lg   22    4      0.18
Il23r    21    5      0.24
Cebpb    16    2      0.12
Il6ra    12    4      0.33
Myd88    12    4      0.33
Socs3     8    1      0.12
Syk       5    3      0.60
Nfkb1     5    0      0.00
Irf4      4    2      0.50
Il1r1     4    0      0.00
Stat3     4    0      0.00
Runx1     3    1      0.33
Nfatc2    3    0      0.00
Tlr4      3    3      1.00
Il12rb    2    1      0.50
Il7r      2    1      0.50
Stat6     2    0      0.00
TCR genes changed in how many targets:
gene
Trbv1           10
Trbv2            8
Trbv15           7
Trbv31           5
Tcrg-V4     

**Programs** (14:09), on thread t6: What the targets' changed genes are: Tgfbr1 217, Tbx21 314, Rorc 67, Socs1 144, Traf6 88 genes. Read them as programmes and published Th17 sets (pathogenic vs. regulatory, IL-23, TGF-beta, T-bet/IFNg).

Why: Hallmark programme scores of the five targets against control guides

163 target x program pairs changed at adjusted p < 0.05 (scanpy scores; OLS, sample as a covariate); the same test on 3 groups of control guides: 23 of 150 group x program pairs changed at adjusted p < 0.05

![20261114T140914907-1.png](figures/20261114T140914907-1.png)

**Genes** (14:12), on thread t7: Does Rorc or Tbx21 knockout change Tgfbr1, Il23r, Ccr6, Stat3 or other targets' own genes? Il23r (-2.74) and Ccr6 (-2.15) show strong own-gene knockdown, and Il23r/Ccr6 fall in the Rorc KO.

Why: t7: the other targets that move several named genes (Ahr, Il17a, Jak2, Tgfbr2), each condition apart

by condition, each value apart: colitis: 4 of 4 targets change genes at q < 0.05 (4 targets by a test on cells (Wilcoxon, the field's most common) against the control cells of the target's own samples, in its mix, since only 2 captures hold...

![20261114T141252984-1.png](figures/20261114T141252984-1.png)

**Genes** (14:14), on thread t6: What the targets' changed genes are: Tgfbr1 217, Tbx21 314, Rorc 67, Socs1 144, Traf6 88 genes. Read them as programmes and published Th17 sets (pathogenic vs. regulatory, IL-23, TGF-beta, T-bet/IFNg).

Why: Read the named Th17, IL-23, TGF-beta and T-bet genes per target against control guides

5 of 5 targets change genes at q < 0.05 (5 targets by pseudobulk by capture (edgeR-QL), since 10 captures hold both the target's and the reference cells); 3 groups of 5 control guides tested as targets gave 5 discoveries; 5 targets change m...

![20261114T141422092-1.png](figures/20261114T141422092-1.png)

**Programs** (14:15), on thread t6: What the targets' changed genes are: Tgfbr1 217, Tbx21 314, Rorc 67, Socs1 144, Traf6 88 genes. Read them as programmes and published Th17 sets (pathogenic vs. regulatory, IL-23, TGF-beta, T-bet/IFNg).

Why: Read the set scores apart by disease model (NTN and colitis), as the brief asks

by condition, each value apart: colitis: 18 target x program pairs changed at adjusted p < 0.05 (scanpy scores; OLS, sample as a covariate); the same test on 3 groups of control guides: 5 of 15 group x program pairs changed at adjusted p < ...

![20261114T141524165-1.png](figures/20261114T141524165-1.png)

**Programs** (14:17), on thread t6: What the targets' changed genes are: Tgfbr1 217, Tbx21 314, Rorc 67, Socs1 144, Traf6 88 genes. Read them as programmes and published Th17 sets (pathogenic vs. regulatory, IL-23, TGF-beta, T-bet/IFNg).

Why: Own-gene check: re-score the sets with each target's own gene removed, by disease model

by condition, each value apart: colitis: 11 target x program pairs changed at adjusted p < 0.05 (scanpy scores; OLS, sample as a covariate); the same test on 3 groups of control guides: 4 of 9 group x program pairs changed at adjusted p < 0...

![20261114T141717719-1.png](figures/20261114T141717719-1.png)

**Tgfbr1 guide 1: S14 share by capture and guide, TCR V-gene concentration, versus other targets** (14:28), on thread t10: Tgfbr1 passes its line (e-distance 1.34, 217 genes) but t9/f5 show it is guide 1 in NTN2 (2,646 cells, 62% in clonal-looking state S14, TV 0.87) and composition, not within-state change. Next question: is S14 a Tgfbr1-guide-1 clone or capture artefact, and does guide 1 act elsewhere?

Why: Decide whether Tgfbr1's headline effect is a guide-1 clone/capture artefact or a real effect

In [ ]:

import pandas as pd, numpy as np
a = pt.screen().adata
obs = a.obs
print(list(obs.columns)[:60])
print(len(obs))


['sample', 'unit', 'umis', 'genes', 'unit_call', 'label', 'guides', 'control_group', 'control_kind', 'condition', 'qc_reason', 'label_after_qc']
173418

**Tgfbr1 guide-1 cells: Trbv/Trav gene concentration and capture spread** (14:29), on thread t10: Tgfbr1 passes its line (e-distance 1.34, 217 genes) but t9/f5 show it is guide 1 in NTN2 (2,646 cells, 62% in clonal-looking state S14, TV 0.87) and composition, not within-state change. Next question: is S14 a Tgfbr1-guide-1 clone or capture artefact, and does guide 1 act elsewhere?

![20261114T142913086-1.png](figures/20261114T142913086-1.png)

In [ ]:

import numpy as np, pandas as pd, scipy.sparse as sp
a = pt.screen().adata
o = a.obs
print(o[['unit','unit_call','label','guides','control_group','condition','label_after_qc']].head(3).to_string())
print(o.label_after_qc.value_counts().head(5))
g = o.guides.astype(str)
single = ~g.str.contains(r'[;,|]', regex=True)
print(single.mean(), g[g.str.contains('Tgfbr1')].head(3).tolist())


                                                             unit unit_call label guides control_group condition label_after_qc
Colitis_2_23_2_Control:AAACCTGAGAAACCTA-1  Colitis_2_23_2_Control       one  none                        colitis      failed QC
Colitis_2_23_2_Control:AAACCTGAGAGTCGGT-1  Colitis_2_23_2_Control       one  none                        colitis           none
Colitis_2_23_2_Control:AAACCTGAGATAGTCA-1  Colitis_2_23_2_Control       one  none                        colitis           none
label_after_qc
none         89395
multi        27409
failed QC    15072
control       4420
Tgfbr1        3784
Name: count, dtype: int64
0.8388575580389579 ['gRNA_Tgfbr1_5_capture:1', 'gRNA_Tgfbr1_2_capture:1', 'gRNA_Stat3_4_capture:1;gRNA_Tgfbr1_3_capture:1']

**Tgfbr1 single-guide cells: dominant Trbv share per guide/capture vs controls** (14:29), on thread t10: Tgfbr1 passes its line (e-distance 1.34, 217 genes) but t9/f5 show it is guide 1 in NTN2 (2,646 cells, 62% in clonal-looking state S14, TV 0.87) and composition, not within-state change. Next question: is S14 a Tgfbr1-guide-1 clone or capture artefact, and does guide 1 act elsewhere?

In [ ]:

import numpy as np, pandas as pd, scipy.sparse as sp
a = pt.screen().adata; o = a.obs
vn = pd.Index(a.var_names)
tcr = [i for i,v in enumerate(vn) if v.startswith('Trbv') or v.startswith('Trav')]
trb = [i for i,v in enumerate(vn) if v.startswith('Trbv')]
print(len(tcr), len(trb))
sel = o.label_after_qc.isin(['Tgfbr1','control'])
sub = a[sel.values]
X = sub.X[:, trb]; X = X.tocsr() if sp.issparse(X) else sp.csr_matrix(X)
tot = np.asarray(X.sum(1)).ravel()
top = np.asarray(X.max(1).todense()).ravel()
so = sub.obs.copy(); so['trb_tot']=tot; so['trb_top']=top
so['top_frac']=np.where(tot>0, top/np.maximum(tot,1), np.nan)
so['guide']=so.guides.astype(str).str.extract(r'(gRNA_[A-Za-z0-9]+_\d)')[0]
so['exp']=so.unit.str.split('_').str[0]+so.unit.str.split('_').str[1]
so['grp']=np.where(so.label_after_qc=='control','control', so.guide)
r = so[so.trb_tot>0].groupby(['exp','grp']).agg(n=('top_frac','size'),topfrac=('top_frac','median')).reset_index()
print(r.to_string())


163 35
         exp            grp     n   topfrac
0   Colitis2        control   566  0.886253
1   Colitis2  gRNA_Tgfbr1_1    35  0.800000
2   Colitis2  gRNA_Tgfbr1_2    22  0.835034
3   Colitis2  gRNA_Tgfbr1_3    25  0.774194
4   Colitis2  gRNA_Tgfbr1_4    19  1.000000
5   Colitis2  gRNA_Tgfbr1_5    67  0.954545
6       NTN1        control   644  1.000000
7       NTN1  gRNA_Tgfbr1_1    31  1.000000
8       NTN1  gRNA_Tgfbr1_2    27  1.000000
9       NTN1  gRNA_Tgfbr1_3   117  1.000000
10      NTN1  gRNA_Tgfbr1_4    22  1.000000
11      NTN1  gRNA_Tgfbr1_5    20  1.000000
12      NTN2        control   799  1.000000
13      NTN2  gRNA_Tgfbr1_1  2468  1.000000
14      NTN2  gRNA_Tgfbr1_2     3  0.950000
15      NTN2  gRNA_Tgfbr1_3    29  0.785714
16      NTN2  gRNA_Tgfbr1_4     7  0.857143
17      NTN2  gRNA_Tgfbr1_5   104  0.875000

**Dominant Trbv gene identity in NTN2 Tgfbr1 guide 1 cells vs NTN2 controls and other guides** (14:29), on thread t10: Tgfbr1 passes its line (e-distance 1.34, 217 genes) but t9/f5 show it is guide 1 in NTN2 (2,646 cells, 62% in clonal-looking state S14, TV 0.87) and composition, not within-state change. Next question: is S14 a Tgfbr1-guide-1 clone or capture artefact, and does guide 1 act elsewhere?

In [ ]:

import numpy as np, pandas as pd, scipy.sparse as sp
a = pt.screen().adata; o = a.obs
vn = np.array(a.var_names)
trb = np.array([i for i,v in enumerate(vn) if v.startswith('Trbv')])
o2 = o.copy(); o2['guide']=o2.guides.astype(str).str.extract(r'(gRNA_[A-Za-z0-9]+_\d)')[0]
ntn2 = o2.unit.str.startswith('NTN_2').values
def prof(mask):
    X = a[mask][:, trb].X
    X = X.tocsr() if sp.issparse(X) else sp.csr_matrix(X)
    has = np.asarray(X.sum(1)).ravel()>0
    dom = np.asarray(X.argmax(1)).ravel()[has]
    s = pd.Series(vn[trb][dom]).value_counts(normalize=True)
    return has.sum(), s.head(4).round(2).to_dict()
for name,m in [('Tgfbr1 g1 NTN2', ntn2&(o2.guide=='gRNA_Tgfbr1_1').values&(o2.label_after_qc=='Tgfbr1').values),
               ('control NTN2', ntn2&(o2.label_after_qc=='control').values),
               ('Tgfbr1 other guides NTN2', ntn2&(o2.guide!='gRNA_Tgfbr1_1').values&(o2.label_after_qc=='Tgfbr1').values),
               ('Rorc NTN2', ntn2&(o2.label_after_qc=='Rorc').values),
               ('Tgfbr1 g1 NTN1+colitis', (~ntn2)&(o2.guide=='gRNA_Tgfbr1_1').values&(o2.label_after_qc=='Tgfbr1').values)]:
    print(name, prof(m))


Tgfbr1 g1 NTN2 (2468, {'Trbv3': 0.96, 'Trbv2': 0.02, 'Trbv12-2': 0.01, 'Trbv15': 0.0})
control NTN2 (799, {'Trbv2': 0.36, 'Trbv3': 0.22, 'Trbv12-2': 0.06, 'Trbv13-1': 0.05})
Tgfbr1 other guides NTN2 (143, {'Trbv13-3': 0.63, 'Trbv31': 0.13, 'Trbv12-2': 0.05, 'Trbv2': 0.04})
Rorc NTN2 (107, {'Trbv13-1': 0.4, 'Trbv20': 0.31, 'Trbv16': 0.07, 'Trbv12-1': 0.07})
Tgfbr1 g1 NTN1+colitis (66, {'Trbv5': 0.3, 'Trbv12-2': 0.17, 'Trbv2': 0.12, 'Trbv3': 0.06})

**describe the screen and the state labels** (14:36, t11-probe), on thread t11: Rorc (Il17a -1.97, Il23r -1.99) and Tbx21 (Il17a +1.73) are opposite on Th17 genes, robust to other-targets reference and single-guide cells, and Rorc/Socs1 raise the T-bet programme. Does the opposition hold within the Th17-core states, not only through state shares?

Why: t11: set up the within-state test

In [ ]:
d = pt.describe()
print(str(d)[:3000])
ad = pt.screen().adata
print(ad.obs.columns.tolist())
print(ad.shape)

{'cells': 'every cell', 'n_cells': 173418, 'named_like': 'Colitis_2_23_2_Control:AAACCTGAGAAACCTA-1', 'per_sample': {'Colitis_2_23_2_Control': 14174, 'Colitis_2_23_2_Th17-1': 12110, 'Colitis_2_23_2_Th17-2': 5126, 'NTN_1_19_23_1': 13963, 'NTN_1_19_23_2': 9512, 'NTN_1_19_5': 12495, 'NTN_1_19_9_1': 12827, 'NTN_1_19_9_2': 12848, 'NTN_2_23_1_Control': 25658, 'NTN_2_23_1_Th17-1': 19511, 'NTN_2_23_1_Th17-2': 19811, 'NTN_2_23_1_Th17-3': 15383}, 'independent_units': True, 'labels': {'none': 89395, 'multi': 27409, 'failed QC': 15072, 'control': 4420, 'Tgfbr1': 3784, 'Il23r': 3400, 'Tbx21': 2216, 'Il17a': 2130, 'Cd40lg': 2008, 'Syk': 1973, 'Il6ra': 1814, 'Tgfbr2': 1759, 'Nfatc2': 1342, 'Stat3': 1328, 'Cebpb': 1254, 'Traf6': 1249, 'Myd88': 1170, 'Jak2': 1147, 'Ahr': 939, 'Il1r1': 887, 'Socs3': 885, 'Stat6': 882, 'Tlr4': 874, 'Ccr6': 861, 'Irf4': 825}, 'targets_with_10_cells': 28, 'control_guides': {'gRNA_control_18_capture': 717, 'gRNA_control_9_capture': 565, 'gRNA_control_20_capture': 463, 'gRNA

**list the saved tables and states runs available** (14:45, t11-probe), on thread t11: Rorc (Il17a -1.97, Il23r -1.99) and Tbx21 (Il17a +1.73) are opposite on Th17 genes, robust to other-targets reference and single-guide cells, and Rorc/Socs1 raise the T-bet programme. Does the opposition hold within the Th17-core states, not only through state shares?

Why: t11: find the state assignments to split the test by Th17-core state

In [ ]:
import pandas as pd
ad = pt.screen().adata
print([c for c in ad.obs.columns])
print(ad.obs['label_after_qc'].value_counts().head(40).to_dict())
print(ad.obs['unit_call'].unique()[:5], ad.obs['control_kind'].value_counts().to_dict())
print(ad.var_names[:5].tolist())
print(ad.layers.keys() if hasattr(ad,'layers') else None)

['sample', 'unit', 'umis', 'genes', 'unit_call', 'label', 'guides', 'control_group', 'control_kind', 'condition', 'qc_reason', 'label_after_qc']
{'none': 89395, 'multi': 27409, 'failed QC': 15072, 'control': 4420, 'Tgfbr1': 3784, 'Il23r': 3400, 'Tbx21': 2216, 'Il17a': 2130, 'Cd40lg': 2008, 'Syk': 1973, 'Il6ra': 1814, 'Tgfbr2': 1759, 'Nfatc2': 1342, 'Stat3': 1328, 'Cebpb': 1254, 'Traf6': 1249, 'Myd88': 1170, 'Jak2': 1147, 'Ahr': 939, 'Il1r1': 887, 'Socs3': 885, 'Stat6': 882, 'Tlr4': 874, 'Ccr6': 861, 'Irf4': 825, 'Socs1': 807, 'Runx1': 751, 'Rorc': 721, 'Il12rb': 613, 'Nfkb1': 532, 'Jak1': 489, 'Il7r': 482}
<ArrowStringArray>
['one']
Length: 1, dtype: str {'': 168851, 'non_targeting': 4567}
['Xkr4', 'Gm1992', 'Gm37381', 'Rp1', 'Sox17']
KeysView(Layers with keys: None)

**States and composition** (14:47), on thread t11: Rorc (Il17a -1.97, Il23r -1.99) and Tbx21 (Il17a +1.73) are opposite on Th17 genes, robust to other-targets reference and single-guide cells, and Rorc/Socs1 raise the T-bet programme. Does the opposition hold within the Th17-core states, not only through state shares?

Why: t11: which states are Th17-core, and each target's share of them

22 states, numbered: name each from its markers, here the test's top four with the % of cells expressing each inside/outside the state (mitochondrial and ribosomal protein genes left out; the markers table has 10 per state, those among them...

![20261114T144733960-1.png](figures/20261114T144733960-1.png)

**within-state log-fold change of Il17a, Il23r, Ccr6, Ifng, Cxcr3 for Rorc and Tbx21 vs control cells in S02, S06, S07** (14:47, t11-probe), on thread t11: Rorc (Il17a -1.97, Il23r -1.99) and Tbx21 (Il17a +1.73) are opposite on Th17 genes, robust to other-targets reference and single-guide cells, and Rorc/Socs1 raise the T-bet programme. Does the opposition hold within the Th17-core states, not only through state shares?

Why: t11: does the opposition hold inside Th17-core states

In [ ]:
import numpy as np, pandas as pd
ad = pt.screen().adata
cs = pt.load("20261114T142946-C-states_and_composition-20a03f:cell_states")
cs = cs.set_index('barcode')
ad.obs['state'] = cs['state'].reindex(ad.obs_names).values
genes = ['Il17a','Il23r','Ccr6','Ifng','Cxcr3','Rorc','Tbx21','Il17f','Il1r1']
genes = [g for g in genes if g in ad.var_names]
X = ad[:, genes].X
import scipy.sparse as sp
X = X.toarray() if sp.issparse(X) else np.asarray(X)
lib = np.asarray(ad.X.sum(axis=1)).ravel()
lg = np.log1p(X / lib[:,None] * 1e4)
df = pd.DataFrame(lg, columns=genes, index=ad.obs_names)
df['label'] = ad.obs['label_after_qc'].values
df['state'] = ad.obs['state'].values
df['unit'] = ad.obs['unit'].values
df['ctrl'] = ad.obs['control_kind'].values
for s in ['S02','S06','S07','S11']:
    sub = df[df.state==s]
    ctrl = sub[sub.label=='control']
    out = {}
    for t in ['Rorc','Tbx21']:
        tc = sub[sub.label==t]
        out[t+'_n'] = len(tc)
        for g in ['Il17a','Il23r','Ccr6','Ifng','Cxcr3']:
            out[t+'_'+g] = round(tc[g].mean()-ctrl[g].mean(),2)
    out['ctrl_n'] = len(ctrl)
    print(s, out)


S02 {'Rorc_n': 132, 'Rorc_Il17a': -0.49, 'Rorc_Il23r': -0.05, 'Rorc_Ccr6': -0.06, 'Rorc_Ifng': -0.33, 'Rorc_Cxcr3': 0.11, 'Tbx21_n': 296, 'Tbx21_Il17a': 1.2, 'Tbx21_Il23r': 0.08, 'Tbx21_Ccr6': 0.5, 'Tbx21_Ifng': -0.96, 'Tbx21_Cxcr3': -0.63, 'ctrl_n': 1148}
S06 {'Rorc_n': 67, 'Rorc_Il17a': -0.25, 'Rorc_Il23r': -0.03, 'Rorc_Ccr6': -0.02, 'Rorc_Ifng': 0.04, 'Rorc_Cxcr3': 0.32, 'Tbx21_n': 79, 'Tbx21_Il17a': 0.82, 'Tbx21_Il23r': 0.09, 'Tbx21_Ccr6': 0.26, 'Tbx21_Ifng': -1.01, 'Tbx21_Cxcr3': -0.25, 'ctrl_n': 783}
S07 {'Rorc_n': 130, 'Rorc_Il17a': -0.07, 'Rorc_Il23r': -0.04, 'Rorc_Ccr6': -0.06, 'Rorc_Ifng': 0.24, 'Rorc_Cxcr3': 0.14, 'Tbx21_n': 102, 'Tbx21_Il17a': 0.0, 'Tbx21_Il23r': 0.02, 'Tbx21_Ccr6': 0.2, 'Tbx21_Ifng': -0.43, 'Tbx21_Cxcr3': -0.22, 'ctrl_n': 584}
S11 {'Rorc_n': 49, 'Rorc_Il17a': -0.22, 'Rorc_Il23r': -0.02, 'Rorc_Ccr6': -0.0, 'Rorc_Ifng': 0.27, 'Rorc_Cxcr3': 0.05, 'Tbx21_n': 70, 'Tbx21_Il17a': 0.37, 'Tbx21_Il23r': 0.01, 'Tbx21_Ccr6': 0.09, 'Tbx21_Ifng': -0.24, 'Tbx21_Cxcr3': 0

**per-capture within-state differences (Rorc, Tbx21 vs control cells of the same capture) for Il17a, Ccr6, Ifng, Cxcr3 in S02 and S06** (14:48, t11-probe), on thread t11: Rorc (Il17a -1.97, Il23r -1.99) and Tbx21 (Il17a +1.73) are opposite on Th17 genes, robust to other-targets reference and single-guide cells, and Rorc/Socs1 raise the T-bet programme. Does the opposition hold within the Th17-core states, not only through state shares?

Why: t11: check the within-state shift holds in each capture, not only pooled

In [ ]:
res = []
for s in ['S02','S06']:
    sub = df[df.state==s]
    for u in sorted(sub.unit.unique()):
        su = sub[sub.unit==u]
        c = su[su.label=='control']
        if len(c) < 20: continue
        row = {'state':s,'unit':u,'ctrl':len(c)}
        for t in ['Rorc','Tbx21']:
            tc = su[su.label==t]
            row[t+'_n'] = len(tc)
            if len(tc) >= 10:
                for g in ['Il17a','Ccr6','Ifng','Cxcr3']:
                    row[t+'_'+g] = round(tc[g].mean()-c[g].mean(),2)
        res.append(row)
r = pd.DataFrame(res)
pd.set_option('display.width',250)
print(r.to_string(max_rows=40, max_cols=20))

  state           unit  ctrl  Rorc_n  Rorc_Il17a  Rorc_Ccr6  Rorc_Ifng  Rorc_Cxcr3  Tbx21_n  Tbx21_Il17a  Tbx21_Ccr6  Tbx21_Ifng  Tbx21_Cxcr3
0   S02  NTN_1_19_23_1   712      80       -0.49      -0.07      -0.29        0.13      188         1.31        0.53       -0.99        -0.64
1   S02  NTN_1_19_23_2   405      51       -0.45      -0.04      -0.37        0.08       98         1.06        0.49       -0.94        -0.64
2   S06  NTN_1_19_23_1    46      19        0.08      -0.04      -0.01        0.21        7          NaN         NaN         NaN          NaN
3   S06   NTN_1_19_9_1   353      15       -0.18      -0.01       0.35        0.19       25         0.75        0.28       -0.67        -0.26
4   S06   NTN_1_19_9_2   362      18       -0.18       0.04      -0.38        0.01       44         0.64        0.20       -1.09        -0.14

**Genes** (14:50), on thread t11: Rorc (Il17a -1.97, Il23r -1.99) and Tbx21 (Il17a +1.73) are opposite on Th17 genes, robust to other-targets reference and single-guide cells, and Rorc/Socs1 raise the T-bet programme. Does the opposition hold within the Th17-core states, not only through state shares?

Why: t11: Rorc and Tbx21 genes inside Th17-core state S02 against control guides

2 of 2 targets change genes at q < 0.05 (2 targets by a test on cells (Wilcoxon, the field's most common) against the control cells of the target's own samples, in its mix, since only 2 captures hold both the target's and the reference cell...

![20261114T145008725-1.png](figures/20261114T145008725-1.png)

**count changed genes per target in S02 and S06 runs, and the Th17-gene shares** (14:52, t11-probe), on thread t11: Rorc (Il17a -1.97, Il23r -1.99) and Tbx21 (Il17a +1.73) are opposite on Th17 genes, robust to other-targets reference and single-guide cells, and Rorc/Socs1 raise the T-bet programme. Does the opposition hold within the Th17-core states, not only through state shares?

Why: t11: set the within-state Rorc result against its control-group line

In [ ]:
a = pt.load("20261114T144803-C-genes-104ebd:changed_genes")
b = pt.load("20261114T144811-C-genes-5701a9:changed_genes")
for nm, t in [('S02',a),('S06',b)]:
    t = t.copy()
    print(nm, t.groupby('target').apply(lambda x: int((x.q<0.05).sum())).to_dict())


S02 {'Rorc': 46, 'Tbx21': 294}
S06 {'Rorc': 1, 'Tbx21': 89}

**Check whether Trbv3 dominance in Tgfbr1 guide 1 holds per capture against same-capture controls** (15:01)

Why: Weakest result r2: clonality rests on Trbv proxy; test capture confound

In [ ]:
import numpy as np, pandas as pd
a = pt.screen().adata
o = a.obs.copy()
g = o['guides'].astype(str)
o['guide'] = g.str.extract(r'(gRNA_[A-Za-z0-9]+_\d)')[0]
single = ~g.str.contains(';')
trb = np.where(a.var_names.str.startswith('Trbv'))[0]
X = a[:, trb].X
X = X.tocsr() if hasattr(X,'tocsr') else np.asarray(X)
tot = np.asarray(X.sum(1)).ravel()
top = np.asarray(X.argmax(1)).ravel()
names = np.array(a.var_names[trb])
o['top'] = np.where(tot>0, names[top], 'none')
print(o['control_kind'].value_counts().head() if 'control_kind' in o else 'nocol')
ctrl = (o['control_group'].notna() & (o['control_group'].astype(str)!='')) if 'control_group' in o else o['guide'].astype(str).str.contains('NonTarget|non|ctrl', case=False)
o['grp'] = np.where(o['guide']=='gRNA_Tgfbr1_1','Tgfbr1_g1', np.where(o['guide'].astype(str).str.startswith('gRNA_Tgfbr1'),'Tgfbr1_other', np.where(ctrl,'control','other')))
d = o[single & (tot>0) & o['grp'].isin(['Tgfbr1_g1','Tgfbr1_other','control'])].copy()
d['t3'] = d['top']=='Trbv3'
print(d.groupby(['sample','grp'])['t3'].agg(['mean','size']).unstack().round(2).to_string())

control_kind
                 168851
non_targeting      4567
Name: count, dtype: int64
                           mean                           size                     
grp                   Tgfbr1_g1 Tgfbr1_other control Tgfbr1_g1 Tgfbr1_other control
sample                                                                             
Colitis_2_23_2_Th17-1      0.10         0.46    0.04        29          110     362
Colitis_2_23_2_Th17-2      0.00         0.25    0.01         6           24     207
NTN_1_19_23_1              0.00         0.00    0.04         7           26     213
NTN_1_19_23_2              0.00         0.00    0.04         4           18     108
NTN_1_19_5                 0.00         0.00    0.08        16            3      90
NTN_1_19_9_1               1.00         0.00    0.07         1           57     116
NTN_1_19_9_2               0.00         0.05    0.07         2           78     126
NTN_2_23_1_Th17-1          0.95         0.02    0.20       961           

**Tgfbr1 guide 1 versus other Tgfbr1 guides against same-capture controls: Il17a, Ccr6, Rorc in NTN2 (skeptic's test for f3)** (15:03)

Why: Answer d-f3-wording: is the lower Il17a/Ccr6/Rorc present without guide 1?

In [ ]:
import numpy as np, pandas as pd
from scipy.stats import mannwhitneyu
a = pt.screen().adata
o = a.obs.copy()
g = o['guides'].astype(str)
o['guide'] = g.str.extract(r'(gRNA_[A-Za-z0-9]+_\d)')[0]
single = ~g.str.contains(';')
ctrl = o['control_kind'].astype(str)=='non_targeting'
o['grp'] = np.where(o['guide']=='gRNA_Tgfbr1_1','g1', np.where(o['guide'].astype(str).str.startswith('gRNA_Tgfbr1'),'other_Tgfbr1', np.where(ctrl,'control','x')))
ntn2 = o['sample'].astype(str).str.startswith('NTN_2')
sel = single & ntn2 & o['grp'].isin(['g1','other_Tgfbr1','control'])
sub = a[sel.values]
tot = np.asarray(sub.X.sum(1)).ravel()
res=[]
for gene in ['Il17a','Ccr6','Rorc','Tgfbr1']:
    x = sub[:, gene].X
    x = np.asarray(x.todense()).ravel() if hasattr(x,'todense') else np.asarray(x).ravel()
    v = np.log1p(x/tot*1e4)
    d = pd.DataFrame({'v':v,'grp':o.loc[sel,'grp'].values})
    m = d.groupby('grp')['v'].mean()
    for t in ['g1','other_Tgfbr1']:
        p = mannwhitneyu(d.v[d.grp==t], d.v[d.grp=='control']).pvalue
        res.append((gene,t,(d.grp==t).sum(),round(m[t]-m['control'],3),p))
print(pd.DataFrame(res,columns=['gene','group','cells','mean_logCP10k_diff_vs_control','p']).to_string())

     gene         group  cells  mean_logCP10k_diff_vs_control             p
0   Il17a            g1   2759                         -0.347  6.694774e-80
1   Il17a  other_Tgfbr1    156                         -0.301  1.804524e-05
2    Ccr6            g1   2759                         -0.247  3.957917e-59
3    Ccr6  other_Tgfbr1    156                          0.297  3.091184e-11
4    Rorc            g1   2759                         -0.355  4.729602e-85
5    Rorc  other_Tgfbr1    156                          0.052  3.237761e-02
6  Tgfbr1            g1   2759                         -0.082  1.701476e-01
7  Tgfbr1  other_Tgfbr1    156                          0.010  3.177141e-01

**Look up the figure_series module and the lead's plan for figure 3** (15:04, lead-figure-figure-3-the-tgfbr1-signature-is-a-guide-1-clona)

Why: Read the figure's plan and chart conventions before drawing

In [ ]:
import pertura.question_modules as qm, inspect
print([n for n in dir(qm) if 'figure' in n.lower()])
try:
    import pertura.question_modules.figure_series as fs
    print(inspect.getsource(fs)[:3000])
except Exception as e:
    print('err', e)

['figures']
"""A finding's figure as published screens build it (docs/build/adr0022/07_figure_series_from_papers.md): several
panels, each a chart of charts.py or one's own drawing, in its part of one figure 183 mm wide, lettered a, b, c in
reading order, laid out together and checked as one figure (figures.check_figure).

A figure is given as rows: [(height in mm, [(width share, draw), ...]), ...], where draw(panel) draws into the panel
(a matplotlib subfigure): a chart, for example lambda p: charts.volcano(table, None, "", fig=p), or one's own code,
for example lambda p: p.add_subplot().plot(x, y). A panel may carry its title, what it shows in a short sentence:
(width share, draw, title).
"""
import os
import string
import textwrap

from . import figures

LETTER_PT = 8
TITLE_PT = 6


def _letter_and_title(sf, letter, title, width):
    """A panel's letter, bold at its top left, and its title beside it, wrapped to the panel's width (inches). The
    layout keeps room above the panel f

**Find the lead's figure 3 plan in the pertura package source** (15:04, lead-figure-figure-3-the-tgfbr1-signature-is-a-guide-1-clona)

Why: Locate the planned panels for figure 3

In [ ]:
import pertura, os, glob
base = os.path.dirname(pertura.__file__)
hits = []
for root, dirs, files in os.walk(base):
    for f in files:
        if f.endswith('.py'):
            p = os.path.join(root, f)
            try:
                s = open(p, encoding='utf-8').read()
            except Exception:
                continue
            if 'figure-3' in s or 'figure_3' in s or 'Tgfbr1 signature' in s or 'guide-1' in s:
                hits.append(p)
print(hits)
for h in hits[:3]:
    s = open(h, encoding='utf-8').read()
    i = s.find('figure-3')
    print(h, s[max(0,i-200):i+3000] if i>=0 else s[:1500])


[]

**Describe the screen to start the figure worker notebook** (15:04, lead-figure-figure-2-rorc-and-tbx21-oppose-each-other-on-th1)

Why: Start the figure notebook by reading what the cells hold

In [ ]:
print(pt.describe())

{'cells': 'every cell', 'n_cells': 173418, 'named_like': 'Colitis_2_23_2_Control:AAACCTGAGAAACCTA-1', 'per_sample': {'Colitis_2_23_2_Control': 14174, 'Colitis_2_23_2_Th17-1': 12110, 'Colitis_2_23_2_Th17-2': 5126, 'NTN_1_19_23_1': 13963, 'NTN_1_19_23_2': 9512, 'NTN_1_19_5': 12495, 'NTN_1_19_9_1': 12827, 'NTN_1_19_9_2': 12848, 'NTN_2_23_1_Control': 25658, 'NTN_2_23_1_Th17-1': 19511, 'NTN_2_23_1_Th17-2': 19811, 'NTN_2_23_1_Th17-3': 15383}, 'independent_units': True, 'labels': {'none': 89395, 'multi': 27409, 'failed QC': 15072, 'control': 4420, 'Tgfbr1': 3784, 'Il23r': 3400, 'Tbx21': 2216, 'Il17a': 2130, 'Cd40lg': 2008, 'Syk': 1973, 'Il6ra': 1814, 'Tgfbr2': 1759, 'Nfatc2': 1342, 'Stat3': 1328, 'Cebpb': 1254, 'Traf6': 1249, 'Myd88': 1170, 'Jak2': 1147, 'Ahr': 939, 'Il1r1': 887, 'Socs3': 885, 'Stat6': 882, 'Tlr4': 874, 'Ccr6': 861, 'Irf4': 825}, 'targets_with_10_cells': 28, 'control_guides': {'gRNA_control_18_capture': 717, 'gRNA_control_9_capture': 565, 'gRNA_control_20_capture': 463, 'gRNA

**Inspect figure_series and charts API in the question modules** (15:04, lead-figure-figure-2-rorc-and-tbx21-oppose-each-other-on-th1)

Why: Learn the figure API before drawing panels

In [ ]:
import pertura.question_modules as qm, inspect
print([n for n in dir(qm) if not n.startswith('_')][:200])
try:
    import pertura.question_modules.figure_series as fs
    print([n for n in dir(fs) if not n.startswith('_')])
    print(inspect.signature(fs.figure))
    print(fs.figure.__doc__[:1500] if fs.figure.__doc__ else '')
except Exception as e:
    print('err', e)

['Contract', 'ONE_AT_A_TIME', 'PROGRAM_METHODS', 'REGISTRY', 'Result', 'a', 'b', 'c', 'contract', 'd', 'data', 'e', 'f', 'figures', 'g', 'overview', 'packages', 'queue_of', 'register']
['LETTER_PT', 'TITLE_PT', 'compose', 'figure', 'figures', 'os', 'string', 'textwrap']
(rows, width=7.2047244094488185)
The rows of panels drawn into one figure, lettered and laid out; returns the figure and the panels' letters in
    order. Nothing is saved: in a notebook, save_figure(fig, name, data, caption, role, strict=True) enters it in the
    record; a module's figure is saved by compose.

**Read the figure() and compose() source and the charts module names** (15:04, lead-figure-figure-2-rorc-and-tbx21-oppose-each-other-on-th1)

Why: Need the panel signature to draw

In [ ]:
import pertura.question_modules.figure_series as fs, inspect
print(inspect.getsource(fs.figure)[:3000])
print(inspect.getsource(fs.compose)[:1500])
import pertura.question_modules.charts as ch
print([n for n in dir(ch) if not n.startswith('_')])

def figure(rows, width=figures.DOUBLE):
    """The rows of panels drawn into one figure, lettered and laid out; returns the figure and the panels' letters in
    order. Nothing is saved: in a notebook, save_figure(fig, name, data, caption, role, strict=True) enters it in the
    record; a module's figure is saved by compose."""
    plt = figures._plt()
    height = sum(h for h, _ in rows) * figures.MM
    fig = plt.figure(figsize=(width, height), layout="constrained")
    fig.get_layout_engine().set(w_pad=1.5 / 72, h_pad=1.5 / 72, wspace=0.02, hspace=0.02)
    row_figs = fig.subfigures(len(rows), 1, height_ratios=[h for h, _ in rows], squeeze=False)[:, 0]
    letters, k = [], 0
    for rf, (_, panels) in zip(row_figs, rows):
        shares = [p[0] for p in panels]
        subs = rf.subfigures(1, len(panels), width_ratios=shares, squeeze=False)[0]
        for sf, (share, draw, *title) in zip(subs, panels):
            letter = string.ascii_lowercase[k]
            k += 1
            dra

**Read signatures of the chart functions to draw the panels** (15:04, lead-figure-figure-2-rorc-and-tbx21-oppose-each-other-on-th1)

Why: Choose the house charts for the three panels

In [ ]:
import pertura.question_modules.charts as ch, inspect
for n in ['heatmap','strips','ranked_targets','target_panel','diverging_colour_bar','knockdown_vs_baseline','pair_correlations','guide_abundance','stars','group_colours']:
    f=getattr(ch,n)
    print(n, inspect.signature(f))
    print((f.__doc__ or '')[:400])
    print('---')
print(ch.CONTROL, ch.UP, ch.DOWN, ch.GROUP_COLOURS if isinstance(ch.GROUP_COLOURS,dict) else '')

heatmap (M, rows, cols, path, title, label, marks=None, cluster=True, width=None, cap=None, fig=None)
A matrix (rows x cols) on a diverging scale centred on zero, rows and columns clustered; marks: the same
    shape, a significance mark per cell (strings), drawn as a dot where the columns are too narrow for text
    (MARKS_AS_DOTS); cap: a percentile of the absolute values at which the scale stops (the colour bar's ends then
    mark that larger values exist), so a few extreme values do not wash 
---
strips (table, value, group, path, xlabel, title='', order=None, log=False, fig=None, points_at_most=None, xlim=None)
A value's spread in each group (table: group, value), a box per group with every item a point, the number of
    items in the group's label: cells per guide for the targeting and the control guides, for example. With
    points_at_most, a larger group shows that many points drawn at random; its box holds every item. xlim: the
    axis's range where the value has one (a sha

**Load the named-gene tables and pull the five Th17/T-bet genes for Rorc and Tbx21** (15:05, lead-figure-figure-2-rorc-and-tbx21-oppose-each-other-on-th1)

Why: Get the per-target log2FC values for panel a and the within-state values for panel c

In [ ]:
import pandas as pd
nm = pt.load('20261114T141050-C-genes-e2468a:named_genes')
print(nm.columns.tolist(), nm.shape)
print(nm.target.unique())
print(nm[nm.gene.isin(['Il17a','Il23r','Ccr6','Ifng','Cxcr3'])&nm.target.isin(['Rorc','Tbx21'])].to_string())
for rid,name in [('20261114T144803-C-genes-104ebd','S02'),('20261114T144811-C-genes-5701a9','S06')]:
    r = pt.load(rid+':result')
    print(name, r.columns.tolist(), r.shape)
    print(r[r.gene.isin(['Il17a','Il23r','Ccr6','Ifng','Cxcr3'])&r.target.isin(['Rorc','Tbx21'])].to_string())

['target', 'gene', 'effect', 'p', 'q', 'test', 'cells', 'reference_cells', 'changed'] (109, 9)
<ArrowStringArray>
['Rorc', 'Socs1', 'Tbx21', 'Tgfbr1', 'Traf6']
Length: 5, dtype: str
   target   gene    effect             p         q      test  cells  reference_cells  changed
0    Rorc  Il17a -1.968643  7.087862e-07  0.001145  edger_ql    721             4420     True
6    Rorc  Il23r -1.989222  1.534884e-08  0.000044  edger_ql    721             4420     True
10   Rorc   Ifng  0.174344  3.293610e-01  0.768085  edger_ql    721             4420    False
18   Rorc   Ccr6 -1.464361  1.303551e-05  0.004786  edger_ql    721             4420     True
20   Rorc  Cxcr3  0.049236  6.971887e-01  0.927226  edger_ql    721             4420    False
43  Tbx21  Il17a  1.732795  1.309200e-06  0.000416  edger_ql   2216             4420     True
50  Tbx21  Il23r  1.041851  1.958020e-08  0.000024  edger_ql   2216             4420     True
54  Tbx21   Ifng -0.785456  1.130933e-03  0.037886  edger_ql   221

**Read the within-state changed-gene tables and the other named-gene run for the five genes** (15:05, lead-figure-figure-2-rorc-and-tbx21-oppose-each-other-on-th1)

Why: Check which within-state values the runs hold for panel c

In [ ]:
genes5=['Il17a','Il23r','Ccr6','Ifng','Cxcr3']
out={}
for rid,name in [('20261114T144803-C-genes-104ebd','S02'),('20261114T144811-C-genes-5701a9','S06')]:
    cg = pt.load(rid+':changed_genes')
    print(name, cg.columns.tolist(), cg.shape)
    sub = cg[cg.gene.isin(genes5)] if 'gene' in cg.columns else cg.head()
    print(sub.to_string())
    print(pt.load(rid+':positive_controls').columns.tolist())
    print(pt.load(rid+':calibration_groups').head(20).to_string())
nm2 = pt.load('20261114T140913-C-genes-fdb056:named_genes')
print(nm2.target.unique(), nm2.gene.nunique())
print(nm2[nm2.gene.isin(genes5)&nm2.target.isin(['Rorc','Tbx21'])].to_string())

S02 ['target', 'gene', 'effect', 'p', 'q', 'test', 'cells', 'reference_cells'] (340, 8)
    target   gene    effect             p             q                   test  cells  reference_cells
31    Rorc  Il17a -1.741005  2.266588e-04  3.578146e-02  wilcoxon_same_samples    131             1040
46   Tbx21   Ccr6  2.558731  1.536731e-51  8.962213e-48  wilcoxon_same_samples    290             1098
47   Tbx21  Il17a  2.393767  7.923751e-35  1.155283e-31  wilcoxon_same_samples    290             1098
117  Tbx21  Il23r  0.916536  5.205925e-05  2.409600e-03  wilcoxon_same_samples    290             1098
337  Tbx21   Ifng -2.300834  4.310678e-34  5.027975e-31  wilcoxon_same_samples    290             1098
338  Tbx21  Cxcr3 -2.592044  3.031305e-40  5.892856e-37  wilcoxon_same_samples    290             1098
['target', 'gene', 'expected', 'found', 'power_check']
   group                                                                                                                  guides  cells 

**Read the heatmap source to see how fig and NaN are handled** (15:05, lead-figure-figure-2-rorc-and-tbx21-oppose-each-other-on-th1)

Why: Check whether the heatmap draws into a subfigure and accepts missing cells

In [ ]:
import inspect, pertura.question_modules.charts as ch
src = inspect.getsource(ch.heatmap)
print(src[:2500])
print(inspect.getsource(ch.stars)[:300])
print(inspect.getsource(ch.ranked_targets)[:200])
print(ch.figures.DOUBLE, ch.figures.SINGLE)

def heatmap(M, rows, cols, path, title, label, marks=None, cluster=True, width=None, cap=None, fig=None):
    """A matrix (rows x cols) on a diverging scale centred on zero, rows and columns clustered; marks: the same
    shape, a significance mark per cell (strings), drawn as a dot where the columns are too narrow for text
    (MARKS_AS_DOTS); cap: a percentile of the absolute values at which the scale stops (the colour bar's ends then
    mark that larger values exist), so a few extreme values do not wash out the rest."""
    plt = figures._plt()
    M = np.asarray(M, float)
    r, c = _order(M) if cluster else (np.arange(M.shape[0]), np.arange(M.shape[1]))
    M = M[r][:, c]
    rows, cols = [rows[i] for i in r], [cols[j] for j in c]
    lim = float(np.nanmax(np.abs(M))) if np.isfinite(M).any() else 1.0
    capped = False
    if cap is not None and np.isfinite(M).any():
        at = float(np.nanpercentile(np.abs(M[np.abs(M) > 0]), cap)) if (np.abs(M) > 0).any() else lim
        capp

**Build the three panels of figure 2 (heatmap, per-guide dot plot, within-S02 dots) and preview** (15:06, lead-figure-figure-2-rorc-and-tbx21-oppose-each-other-on-th1)

Why: Draw the figure from the run tables and look at it

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from pertura.question_modules import charts as ch, figure_series as fs
nan=np.nan
genes5=['Il17a','Il23r','Ccr6','Ifng','Cxcr3']
rowsA=['Rorc, all cells (n=721)','Tbx21, all cells (n=2216)','Rorc, S02 (n=131)','Tbx21, S02 (n=290)','Tbx21, S06 (n=78)']
M=np.array([
 [-1.968643,-1.989222,-1.464361,0.174344,0.049236],
 [1.732795,1.041851,1.715514,-0.785456,-1.257550],
 [-1.741005,nan,nan,nan,nan],
 [2.393767,0.916536,2.558731,-2.300834,-2.592044],
 [2.086642,2.239439,2.737497,-2.148214,-1.215406]])
Q=np.array([
 [0.001145,4.42e-5,0.004786,0.768085,0.927226],
 [0.000416,2.4e-5,6e-6,0.037886,0.000228],
 [0.035781,nan,nan,nan,nan],
 [1.16e-31,2.41e-3,8.96e-48,5.03e-31,5.89e-37],
 [4.46e-6,3.17e-3,5.29e-9,9.32e-10,3.93e-2]])
marksA=np.array([[ (ch.stars(q) if np.isfinite(q) else '') for q in row] for row in Q],dtype=object)
tabA=pd.DataFrame([(r,g,M[i,j],Q[i,j]) for i,r in enumerate(rowsA) for j,g in enumerate(genes5)],columns=['row','gene','log2fc','q'])
tabA.loc[:,'source']='named_genes of 20261114T141050-C-genes-e2468a (all cells); changed_genes of 20261114T144803 (S02) and 20261114T144811 (S06)'
# panel b: per-guide Il17a, checks runs
guideB=pd.DataFrame([
 ('Rorc','gRNA_Rorc_1',84,-1.07340674,0.0708087),
 ('Rorc','gRNA_Rorc_2',107,-1.02203962,0.0063917),
 ('Rorc','gRNA_Rorc_3',262,-4.03021872,3.387e-14),
 ('Rorc','gRNA_Rorc_4',181,-2.17793495,1.091e-4),
 ('Rorc','gRNA_Rorc_5',87,-0.78766071,0.0125866),
 ('Tbx21','gRNA_Tbx21_1',1191,1.96185193,2.824e-5),
 ('Tbx21','gRNA_Tbx21_2',33,1.33683252,0.0318378),
 ('Tbx21','gRNA_Tbx21_3',364,2.15464178,2.583e-5),
 ('Tbx21','gRNA_Tbx21_4',401,0.70908355,0.0106925),
 ('Tbx21','gRNA_Tbx21_5',227,1.50367141,1.639e-9)],columns=['target','guide','cells','log2fc','q'])
pooledB=pd.DataFrame([
 ('Rorc','edgeR-QL by capture (lead)',-1.96864270,0.0011451),
 ('Rorc','Wilcoxon, same samples',-1.67842948,1.3265e-9),
 ('Rorc','Wilcoxon, all control cells',-1.88675821,1.3320e-17),
 ('Rorc','DESeq2 pseudobulk',-2.22743436,6.987e-9),
 ('Tbx21','edgeR-QL by capture (lead)',1.73279452,0.00041550),
 ('Tbx21','Wilcoxon, same samples',1.53755605,1.0665e-20),
 ('Tbx21','Wilcoxon, all control cells',1.16865289,1.9856e-31),
 ('Tbx21','DESeq2 pseudobulk',1.66984018,7.877e-7)],columns=['target','test','log2fc','q'])
# panel c: within S02
S02=pd.DataFrame([
 ('Rorc','Il17a',-1.741005,0.0357815,131,1040),
 ('Tbx21','Il17a',2.393767,1.155e-31,290,1098),
 ('Tbx21','Il23r',0.916536,2.4096e-3,290,1098),
 ('Tbx21','Ccr6',2.558731,8.963e-48,290,1098),
 ('Tbx21','Ifng',-2.300834,5.028e-31,290,1098),
 ('Tbx21','Cxcr3',-2.592044,5.892e-37,290,1098)],columns=['target','gene','log2fc','q','cells','reference_cells'])
def draw_a(sf):
    ch.heatmap(M, rowsA, genes5, None, 'Tbx21 and Rorc knockout, log2FC', 'log2FC vs control cells', marks=marksA, cluster=False, fig=sf)
def draw_b(sf):
    ax=sf.add_subplot()
    ys=[];labs=[];cols=[];xs=[];filled=[]
    y=0
    items=[]
    for t,col in [('Rorc',ch.DOWN),('Tbx21',ch.UP)]:
        for _,r in guideB[guideB.target==t].iterrows():
            items.append((y,f"{t} {r.guide.split('_')[-1]} (n={r.cells})",r.log2fc,r.q,col,'guide')); y+=1
        for _,r in pooledB[pooledB.target==t].iterrows():
            items.append((y,f"{t} pooled: {r.test}",r.log2fc,r.q,col,'test')); y+=1
        y+=0.8
    for yy,lab,x,q,col,kind in items:
        ax.scatter([x],[yy],marker='o' if kind=='guide' else 'D',s=22 if kind=='guide' else 16,
                   facecolors=col if q<0.05 else 'white',edgecolors=col,linewidths=0.8,zorder=3)
    ax.axvline(0,color='0.6',lw=0.8,ls='--')
    ax.set_yticks([i[0] for i in items]); ax.set_yticklabels([i[1] for i in items],fontsize=5)
    ax.invert_yaxis(); ax.set_xlabel('Il17a log2FC vs control guides',fontsize=6); ax.tick_params(labelsize=5)
    ax.set_title('Il17a, per guide and per test',fontsize=6)
def draw_c(sf):
    ax=sf.add_subplot()
    pairs=[('Il17a','Rorc'),('Il23r','Rorc'),('Ccr6','Rorc'),('Ifng','Rorc'),('Cxcr3','Rorc'),
           ('Il17a','Tbx21'),('Il23r','Tbx21'),('Ccr6','Tbx21'),('Ifng','Tbx21'),('Cxcr3','Tbx21')]
    ylab=[];
    for i,(g,t) in enumerate(pairs):
        r=S02[(S02.gene==g)&(S02.target==t)]
        col=ch.DOWN if t=='Rorc' else ch.UP
        if len(r):
            x=float(r.log2fc.iloc[0]); q=float(r.q.iloc[0])
            ax.scatter([x],[i],s=22,marker='o',facecolors=col if q<0.05 else 'white',edgecolors=col,linewidths=0.8,zorder=3)
            ax.text(x+(0.15 if x>=0 else -0.15),i,ch.stars(q),fontsize=6,va='center',ha='left' if x>=0 else 'right')
        else:
            ax.text(0,i,'not in run table',fontsize=5,va='center',ha='center',color='0.4')
        ylab.append(f"{t} {g}")
    ax.axvline(0,color='0.6',lw=0.8,ls='--')
    ax.set_yticks(range(len(pairs))); ax.set_yticklabels(ylab,fontsize=5); ax.invert_yaxis()
    ax.set_xlim(-3.4,3.4)
    ax.set_xlabel('log2FC within S02 vs control guides',fontsize=6); ax.tick_params(labelsize=5)
    ax.set_title('Within state S02: Rorc 131 cells, Tbx21 290 cells',fontsize=6)
rows=[(80,[(1.0,draw_a,'Tbx21 and Rorc knockout across states')]),
      (80,[(1.0,draw_b,'Per guide and per test'),(1.0,draw_c,'Within state S02')])]
fig,letters=fs.figure(rows, ch.figures.DOUBLE)
print(letters)
fig.savefig('/tmp/preview_fig2.png',dpi=110)
print('ok')

['a', 'b', 'c']
---------------------------------------------------------------------------
PermissionError                           Traceback (most recent call last)
Cell In[9], line 104
    100 rows=[(80,[(1.0,draw_a,'Tbx21 and Rorc knockout across states')]),
    101       (80,[(1.0,draw_b,'Per guide and per test'),(1.0,draw_c,'Within state S02')])]
    102 fig,letters=fs.figure(rows, ch.figures.DOUBLE)
    103 print(letters)
--> 104 fig.savefig('/tmp/preview_fig2.png',dpi=110)
    105 print('ok')

File ~/pertura/envs/pertpy/lib/python3.12/site-packages/matplotlib/figure.py:3515, in Figure.savefig(self, fname, transparent, **kwargs)
   3513     for ax in self.axes:
   3514         _recursively_make_axes_transparent(stack, ax)
-> 3515 self.canvas.print_figure(fname, **kwargs)

File ~/pertura/envs/pertpy/lib/python3.12/site-packages/matplotlib/backend_bases.py:2289, in FigureCanvasBase.print_figure(self, filename, dpi, facecolor, edgecolor, orientation, format, bbox_inches, pad_inche

**Check the layout and save figure 2 with its plotted tables and caption** (15:06, lead-figure-figure-2-rorc-and-tbx21-oppose-each-other-on-th1)

Why: Save the figure in the record and look at it

![20261114T150624607-1.png](figures/20261114T150624607-1.png)

In [ ]:
issues = ch.figures.check_figure(fig)
print('issues', issues)
rec = save_figure(fig, 'lead-figure-figure-2-rorc-and-tbx21-oppose-each-other-on-th1',
    data={'panel_a_log2fc_heatmap': tabA, 'panel_b_il17a_per_guide': guideB, 'panel_b_il17a_pooled_tests': pooledB, 'panel_c_within_S02': S02},
    caption=("Figure 2. Rorc and Tbx21 oppose each other on Th17 genes. (a) log2FC of Il17a, Il23r, Ccr6, Ifng and Cxcr3 for Rorc and Tbx21 knockout against control-guide cells, pooled (Rorc 721 cells, Tbx21 2,216 cells, controls 4,420; edgeR-QL by capture) and within S02 (Rorc 131, Tbx21 290 cells, controls 1,040 and 1,098) and S06 (Tbx21 78 cells, controls 512); stars mark q<0.05, blank cells were not in the run tables. (b) Il17a log2FC for each of five Rorc and five Tbx21 guides (edgeR-QL by capture, cell counts as labelled; filled = q<0.05), with the pooled effect under four tests (edgeR-QL, Wilcoxon on same-sample and all control cells, DESeq2 pseudobulk). (c) Within-state S02 log2FC against control guides (Wilcoxon, same samples): the Rorc side is thinner, with only Il17a passing (log2FC -1.74, q 0.036, 131 cells); the Tbx21 effects pass for all five genes."),
    role='overview', strict=True)
print(rec)

issues []
{'name': 'lead-figure-figure-2-rorc-and-tbx21-oppose-each-other-on-th1', 'caption': 'Figure 2. Rorc and Tbx21 oppose each other on Th17 genes. (a) log2FC of Il17a, Il23r, Ccr6, Ifng and Cxcr3 for Rorc and Tbx21 knockout against control-guide cells, pooled (Rorc 721 cells, Tbx21 2,216 cells, controls 4,420; edgeR-QL by capture) and within S02 (Rorc 131, Tbx21 290 cells, controls 1,040 and 1,098) and S06 (Tbx21 78 cells, controls 512); stars mark q<0.05, blank cells were not in the run tables. (b) Il17a log2FC for each of five Rorc and five Tbx21 guides (edgeR-QL by capture, cell counts as labelled; filled = q<0.05), with the pooled effect under four tests (edgeR-QL, Wilcoxon on same-sample and all control cells, DESeq2 pseudobulk). (c) Within-state S02 log2FC against control guides (Wilcoxon, same samples): the Rorc side is thinner, with only Il17a passing (log2FC -1.74, q 0.036, 131 cells); the Tbx21 effects pass for all five genes.', 'role': 'overview', 'png': '/data/pertura

**Look for the lead's figure plan files and describe the screen** (15:14, lead-figure-figure-3-the-tgfbr1-signature-is-a-guide-1-clona)

Why: Find the plan for figure 3 in the record or files

In [ ]:
import glob, os
pat = ['figure','lead-figure']
print(pt.describe().keys() if hasattr(pt.describe(),'keys') else str(pt.describe())[:1500])
for root in [os.getcwd(), os.path.expanduser('~')]:
    for p in glob.glob(os.path.join(root,'**','*lead*figure*'), recursive=True)[:20]:
        print(p)


dict_keys(['cells', 'n_cells', 'named_like', 'per_sample', 'independent_units', 'labels', 'targets_with_10_cells', 'control_guides', 'cell_columns', 'conditions', 'genes', 'other_calls', 'kept_cell_sets', 'runs'])
---------------------------------------------------------------------------
KeyboardInterrupt                         Traceback (most recent call last)
Cell In[4], line 15
     11 import glob, os
     12 pat = ['figure','lead-figure']
     13 print(pt.describe().keys() if hasattr(pt.describe(),'keys') else str(pt.describe())[:1500])
     14 for root in [os.getcwd(), os.path.expanduser('~')]:
---> 15     for p in glob.glob(os.path.join(root,'**','*lead*figure*'), recursive=True)[:20]:
     16         print(p)

File ~/pertura/envs/pertpy/lib/python3.12/glob.py:28, in glob(pathname, root_dir, dir_fd, recursive, include_hidden)
     13 def glob(pathname, *, root_dir=None, dir_fd=None, recursive=False,
     14         include_hidden=False):
     15     """Return a list of paths ma

**Inspect columns, Trbv genes and guide labels in the screen object** (15:14, lead-figure-figure-3-the-tgfbr1-signature-is-a-guide-1-clona)

Why: Prepare panel a recomputation from the screen object

In [ ]:
import numpy as np, pandas as pd
a = pt.screen().adata
obs = a.obs
print(obs.columns.tolist())
print(obs['control_kind'].value_counts(dropna=False).head(10))
print([v for v in a.var_names if v.startswith('Trbv')][:40])
print(obs['unit'].unique())
print(obs['guides'].astype(str).str.extract(r'(gRNA_[A-Za-z0-9]+_\d)')[0].value_counts(dropna=False).head(30))


['sample', 'unit', 'umis', 'genes', 'unit_call', 'label', 'guides', 'control_group', 'control_kind', 'condition', 'qc_reason', 'label_after_qc']
control_kind
                 168851
non_targeting      4567
Name: count, dtype: int64
['Trbv1', 'Trbv2', 'Trbv3', 'Trbv4', 'Trbv5', 'Trbv6', 'Trbv7', 'Trbv8', 'Trbv9', 'Trbv10', 'Trbv11', 'Trbv12-1', 'Trbv13-1', 'Trbv12-2', 'Trbv13-2', 'Trbv12-3', 'Trbv13-3', 'Trbv14', 'Trbv15', 'Trbv16', 'Trbv17', 'Trbv18', 'Trbv19', 'Trbv20', 'Trbv21', 'Trbv22', 'Trbv23', 'Trbv24', 'Trbv25', 'Trbv26', 'Trbv27', 'Trbv28', 'Trbv29', 'Trbv30', 'Trbv31']
<ArrowStringArray>
['Colitis_2_23_2_Control',  'Colitis_2_23_2_Th17-1',  'Colitis_2_23_2_Th17-2',
          'NTN_1_19_23_1',          'NTN_1_19_23_2',             'NTN_1_19_5',
           'NTN_1_19_9_1',           'NTN_1_19_9_2',     'NTN_2_23_1_Control',
      'NTN_2_23_1_Th17-1',      'NTN_2_23_1_Th17-2',      'NTN_2_23_1_Th17-3']
Length: 12, dtype: str
0
NaN               102657
gRNA_Cd40lg_2       7821
gRNA

**Recompute per capture the Trbv3-dominant share for Tgfbr1 guide 1, other Tgfbr1 guides and controls** (15:14, lead-figure-figure-3-the-tgfbr1-signature-is-a-guide-1-clona)

Why: Panel a from the screen object, same code as the brief

In [ ]:
import numpy as np, pandas as pd, re
a = pt.screen().adata
obs = a.obs.copy()
gl = obs['guides'].astype(str)
print(gl.head(3).tolist(), gl[gl.str.contains(';|,')].head(3).tolist())
allg = gl.apply(lambda s: sorted(set(re.findall(r'gRNA_[A-Za-z0-9]+_\d', s))))
obs['nguides'] = allg.apply(len)
obs['guide'] = allg.apply(lambda l: l[0] if len(l)==1 else None)
trbv = [i for i,v in enumerate(a.var_names) if v.startswith('Trbv')]
X = a.X[:, trbv]
X = X.toarray() if hasattr(X,'toarray') else np.asarray(X)
tot = X.sum(1)
dom = np.array([a.var_names[trbv[j]] for j in X.argmax(1)])
obs['dom'] = dom
obs['trbv_tot'] = tot
sub = obs[(obs.nguides==1) & (obs.trbv_tot>0) & (obs.qc_reason.astype(str)=='')]
sub = sub.assign(grp=np.where(sub.guide=='gRNA_Tgfbr1_1','Tgfbr1 guide 1',
      np.where(sub.guide.fillna('').str.startswith('gRNA_Tgfbr1_'),'Tgfbr1 other guides',
      np.where(sub.control_kind=='non_targeting','controls',None))))
sub = sub[sub.grp.notna()]
sub = sub.assign(is_trbv3=(sub.dom=='Trbv3'))
tab = sub.groupby(['unit','grp']).agg(share=('is_trbv3','mean'), n=('is_trbv3','size')).reset_index()
print(tab.to_string())
print(sub.guide.unique()[:0], sorted(sub.guide.dropna().unique())[:0])
print(obs[obs.guide.fillna('').str.startswith('gRNA_Tgfbr1')].groupby('guide').size())
pt.save_table('fig3_panel_a_trbv3_share', tab)


['', '', ''] ['gRNA_Runx1_1_capture:1;gRNA_control_15_capture:1', 'gRNA_Il23r_3_capture:1;gRNA_Nfatc2_1_capture:1;gRNA_Tlr4_1_capture:1', 'gRNA_Runx1_1_capture:1;gRNA_Traf6_5_capture:1']
                     unit                  grp     share    n
0   Colitis_2_23_2_Th17-1       Tgfbr1 guide 1  0.103448   29
1   Colitis_2_23_2_Th17-1  Tgfbr1 other guides  0.463636  110
2   Colitis_2_23_2_Th17-1             controls  0.044321  361
3   Colitis_2_23_2_Th17-2       Tgfbr1 guide 1  0.000000    6
4   Colitis_2_23_2_Th17-2  Tgfbr1 other guides  0.260870   23
5   Colitis_2_23_2_Th17-2             controls  0.009756  205
6           NTN_1_19_23_1       Tgfbr1 guide 1  0.000000    7
7           NTN_1_19_23_1  Tgfbr1 other guides  0.000000   26
8           NTN_1_19_23_1             controls  0.037736  212
9           NTN_1_19_23_2       Tgfbr1 guide 1  0.000000    4
10          NTN_1_19_23_2  Tgfbr1 other guides  0.000000   18
11          NTN_1_19_23_2             controls  0.037037  108
12     

**Check the guide-1 cell counts with and without the QC filter** (15:14, lead-figure-figure-3-the-tgfbr1-signature-is-a-guide-1-clona)

Why: Reconcile n with the brief (961/680/871)

In [ ]:
import numpy as np, pandas as pd, re
a = pt.screen().adata
obs = a.obs.copy()
print(obs.qc_reason.astype(str).value_counts().head())
gl = obs['guides'].astype(str)
allg = gl.apply(lambda s: sorted(set(re.findall(r'gRNA_[A-Za-z0-9]+_\d', s))))
obs['nguides'] = allg.apply(len)
obs['guide'] = allg.apply(lambda l: l[0] if len(l)==1 else None)
trbv = [i for i,v in enumerate(a.var_names) if v.startswith('Trbv')]
X = a.X[:, trbv]; X = X.toarray() if hasattr(X,'toarray') else np.asarray(X)
obs['dom'] = np.array([a.var_names[trbv[j]] for j in X.argmax(1)])
obs['trbv_tot'] = X.sum(1)
for label, m in [('noqc_trbv>0', (obs.nguides==1)&(obs.trbv_tot>0)), ('noqc_all', (obs.nguides==1))]:
    s = obs[m & obs.guide.eq('gRNA_Tgfbr1_1') & obs.unit.str.startswith('NTN_2')]
    print(label, s.groupby('unit').apply(lambda d: pd.Series({'n':len(d),'share':(d.dom=='Trbv3').mean()})).to_string())


qc_reason
                                                               158346
n_genes below 500                                                7919
pct_mito above 0.1                                               4909
cluster in C11, C16, C21, C22 (cells_overview/clusters.tsv)      2244
Name: count, dtype: int64
noqc_trbv>0                        n     share
unit                              
NTN_2_23_1_Th17-1  961.0  0.946930
NTN_2_23_1_Th17-2  680.0  0.961765
NTN_2_23_1_Th17-3  871.0  0.943743
noqc_all                         n     share
unit                               
NTN_2_23_1_Th17-1  1042.0  0.873321
NTN_2_23_1_Th17-2   763.0  0.857143
NTN_2_23_1_Th17-3   954.0  0.861635

**Panel a table on the brief's rule (single-guide, Trbv>0, no QC filter)** (15:15, lead-figure-figure-3-the-tgfbr1-signature-is-a-guide-1-clona)

Why: Recompute panel a table matching the brief's numbers

In [ ]:
import numpy as np, pandas as pd, re
a = pt.screen().adata
obs = a.obs.copy()
gl = obs['guides'].astype(str)
allg = gl.apply(lambda s: sorted(set(re.findall(r'gRNA_[A-Za-z0-9]+_\d', s))))
obs['nguides'] = allg.apply(len)
obs['guide'] = allg.apply(lambda l: l[0] if len(l)==1 else None)
trbv = [i for i,v in enumerate(a.var_names) if v.startswith('Trbv')]
X = a.X[:, trbv]; X = X.toarray() if hasattr(X,'toarray') else np.asarray(X)
obs['dom'] = np.array([a.var_names[trbv[j]] for j in X.argmax(1)])
obs['trbv_tot'] = X.sum(1)
ok = obs[(obs.nguides==1)&(obs.trbv_tot>0)].copy()
g = ok.guide.fillna('')
ok['grp'] = np.where(g=='gRNA_Tgfbr1_1','Tgfbr1 guide 1',
            np.where(g.str.startswith('gRNA_Tgfbr1_'),'Tgfbr1 other guides',
            np.where(ok.control_kind.eq('non_targeting'),'controls',None)))
ok = ok[ok.grp.notna()].copy()
ok['is3'] = ok.dom.eq('Trbv3')
tab = ok.groupby(['unit','grp']).agg(share=('is3','mean'), n=('is3','size')).reset_index()
print(tab.to_string())
pt.save_table('fig3_panel_a_trbv3_share', tab)


                     unit                  grp     share    n
0   Colitis_2_23_2_Th17-1       Tgfbr1 guide 1  0.103448   29
1   Colitis_2_23_2_Th17-1  Tgfbr1 other guides  0.463636  110
2   Colitis_2_23_2_Th17-1             controls  0.044199  362
3   Colitis_2_23_2_Th17-2       Tgfbr1 guide 1  0.000000    6
4   Colitis_2_23_2_Th17-2  Tgfbr1 other guides  0.250000   24
5   Colitis_2_23_2_Th17-2             controls  0.009662  207
6           NTN_1_19_23_1       Tgfbr1 guide 1  0.000000    7
7           NTN_1_19_23_1  Tgfbr1 other guides  0.000000   26
8           NTN_1_19_23_1             controls  0.042254  213
9           NTN_1_19_23_2       Tgfbr1 guide 1  0.000000    4
10          NTN_1_19_23_2  Tgfbr1 other guides  0.000000   18
11          NTN_1_19_23_2             controls  0.037037  108
12             NTN_1_19_5       Tgfbr1 guide 1  0.000000   16
13             NTN_1_19_5  Tgfbr1 other guides  0.000000    3
14             NTN_1_19_5             controls  0.077778   90
15      

**Efficacy** (15:17), on thread t10: Tgfbr1 passes its line (e-distance 1.34, 217 genes) but t9/f5 show it is guide 1 in NTN2 (2,646 cells, 62% in clonal-looking state S14, TV 0.87) and composition, not within-state change. Next question: is S14 a Tgfbr1-guide-1 clone or capture artefact, and does guide 1 act elsewhere?

Why: Panel b: each Tgfbr1 guide's own-gene change and cells per guide

1 guides change their target gene past the line (at least 30% down, or up by the same factor), 2 less, 2 of them knockout guides that leave their transcript unchanged, 2 not measurable

![20261114T151746561-1.png](figures/20261114T151746561-1.png)

**Draw the two panels of figure 3 as one figure** (15:18, lead-figure-figure-3-the-tgfbr1-signature-is-a-guide-1-clona)

Why: Compose panels a and b from the saved tables

In [ ]:
import numpy as np, pandas as pd
import pertura.question_modules.figure_series as fsr
tabA = pt.load('table:fig3_panel_a_trbv3_share').copy()
units_order = ['NTN_2_23_1_Th17-1','NTN_2_23_1_Th17-2','NTN_2_23_1_Th17-3','NTN_1_19_23_1','NTN_1_19_23_2','NTN_1_19_5','NTN_1_19_9_1','NTN_1_19_9_2','Colitis_2_23_2_Th17-1','Colitis_2_23_2_Th17-2']
tabA = tabA[tabA.unit.isin(units_order)]
tabA['x'] = tabA.unit.map({u:i for i,u in enumerate(units_order)})
tabB = pd.DataFrame({
 'guide':['gRNA_Tgfbr1_1','gRNA_Tgfbr1_2','gRNA_Tgfbr1_3','gRNA_Tgfbr1_4','gRNA_Tgfbr1_5'],
 'own_gene_log2fc':[-0.0683,-1.3164,-0.8348,-0.6903,0.3945],
 'cells':[2788,107,527,107,255],
 'measurable':[True,False,True,False,True],
 'verdict':['transcript unchanged','not measurable','at least 30% down','not measurable','transcript unchanged']})
tabB['y'] = range(len(tabB))[::-1]
pt.save_table('fig3_panel_b_own_gene', tabB)
C_G1, C_OTH, C_CTL = '#B2182B', '#2166AC', '#9E9E9E'
def drawA(p):
    ax = p.add_subplot()
    off = {'Tgfbr1 guide 1':-0.22,'Tgfbr1 other guides':0.0,'controls':0.22}
    col = {'Tgfbr1 guide 1':C_G1,'Tgfbr1 other guides':C_OTH,'controls':C_CTL}
    for g in off:
        d = tabA[tabA.grp==g]
        for _,r in d.iterrows():
            filled = r.n>=10
            ax.scatter(r.x+off[g], r.share, s=14, facecolor=col[g] if filled else 'white', edgecolor=col[g], linewidth=0.8, zorder=3)
            if g=='Tgfbr1 guide 1':
                ax.text(r.x+off[g], 1.04, f"{int(r.n)}", ha='center', fontsize=4.5, color=C_G1, rotation=90, va='bottom')
    ax.set_xticks(range(len(units_order)))
    ax.set_xticklabels([u.replace('_23_1_','_').replace('_19_','_') for u in units_order], rotation=90, fontsize=5)
    ax.set_ylim(0,1.25); ax.set_yticks([0,0.25,0.5,0.75,1.0]); ax.tick_params(labelsize=5)
    ax.set_ylabel('Share of single-guide cells\nwith Trbv3 dominant', fontsize=5.5)
    from matplotlib.lines import Line2D
    ax.legend(handles=[Line2D([],[],marker='o',ls='',color=C_G1,ms=3,label='Tgfbr1 guide 1'),
                       Line2D([],[],marker='o',ls='',color=C_OTH,ms=3,label='Tgfbr1 other guides'),
                       Line2D([],[],marker='o',ls='',color=C_CTL,ms=3,label='control guides')],
              fontsize=4.5, frameon=False, loc='upper right', ncol=1, bbox_to_anchor=(1.0,0.92))
    ax.text(0.0,-0.42,'Hollow: fewer than 10 cells; number above guide 1 = its cells', transform=ax.transAxes, fontsize=4.5)
    for s in ['top','right']: ax.spines[s].set_visible(False)
def drawB(p):
    ax = p.add_subplot()
    for _,r in tabB.iterrows():
        c = C_G1 if r.guide.endswith('_1') else C_OTH
        ax.scatter(r.own_gene_log2fc, r.y, s=18, facecolor=c if r.measurable else 'white', edgecolor=c, linewidth=0.8, zorder=3)
        ax.text(r.own_gene_log2fc+0.04, r.y+0.18, f"n={int(r.cells)}", fontsize=4.5, va='bottom')
    ax.axvline(0, color=C_CTL, ls='--', lw=0.6)
    ax.set_yticks(tabB.y); ax.set_yticklabels([g.replace('gRNA_','') for g in tabB.guide], fontsize=5)
    ax.set_xlabel('Tgfbr1 own gene, log2 FC vs control cells', fontsize=5.5); ax.tick_params(labelsize=5)
    ax.set_xlim(-1.6,0.7); ax.set_ylim(-0.6,4.8)
    for s in ['top','right']: ax.spines[s].set_visible(False)
rows = [(85, [(0.58, drawA, 'Guide 1 dominates the Trbv3-dominant population in NTN2 captures, not the other Tgfbr1 guides or controls'),
              (0.42, drawB, 'Guide 1 barely lowers its own gene; other guides cut it')])]
fig, letters = fsr.figure(rows)
print(letters)


['a', 'b']

**Save figure 3 with its plotted tables and a caption giving each panel's n** (15:18, lead-figure-figure-3-the-tgfbr1-signature-is-a-guide-1-clona)

Why: Enter the figure in the record under the planned name

In [ ]:
cap = ("Figure 3. a, Share of single-guide cells whose dominant Trbv gene is Trbv3, per capture (10 captures; n above each guide 1 point; "
 "hollow marks fewer than 10 cells): Tgfbr1 guide 1 in the NTN2 captures 0.95, 0.96 and 0.94 (961, 680 and 871 cells), controls 0.20, 0.26 and 0.16, "
 "other Tgfbr1 guides 0.02 to 0.05. b, Tgfbr1 own-gene log2 fold change against control cells per guide (pooled over captures; n = cells per guide; "
 "hollow = not measurable by the efficacy verdict): guide 1 -0.07 (n = 2,788), guide 2 -1.32 (n = 107), guide 3 -0.83 (n = 527), guide 4 -0.69 (n = 107), guide 5 +0.39 (n = 255).")
save_figure(fig, 'lead-figure-figure-3-the-tgfbr1-signature-is-a-guide-1-clona',
            data={'panel_a_trbv3_share': pt.load('table:fig3_panel_a_trbv3_share'), 'panel_b_own_gene': pt.load('table:fig3_panel_b_own_gene')},
            caption=cap, role='overview', strict=True)


---------------------------------------------------------------------------
ValueError                                Traceback (most recent call last)
Cell In[10], line 15
     11 cap = ("Figure 3. a, Share of single-guide cells whose dominant Trbv gene is Trbv3, per capture (10 captures; n above each guide 1 point; "
     12  "hollow marks fewer than 10 cells): Tgfbr1 guide 1 in the NTN2 captures 0.95, 0.96 and 0.94 (961, 680 and 871 cells), controls 0.20, 0.26 and 0.16, "
     13  "other Tgfbr1 guides 0.02 to 0.05. b, Tgfbr1 own-gene log2 fold change against control cells per guide (pooled over captures; n = cells per guide; "
     14  "hollow = not measurable by the efficacy verdict): guide 1 -0.07 (n = 2,788), guide 2 -1.32 (n = 107), guide 3 -0.83 (n = 527), guide 4 -0.69 (n = 107), guide 5 +0.39 (n = 255).")
---> 15 save_figure(fig, 'lead-figure-figure-3-the-tgfbr1-signature-is-a-guide-1-clona',
     16             data={'panel_a_trbv3_share': pt.load('table:fig3_panel_a_trbv3_s

**Redraw figure 3 with fonts at 5 pt or more and save it** (15:18, lead-figure-figure-3-the-tgfbr1-signature-is-a-guide-1-clona)

Why: The first save failed font-size and overlap checks

![20261114T151840865-1.png](figures/20261114T151840865-1.png)

In [ ]:
import numpy as np, pandas as pd
import pertura.question_modules.figure_series as fsr
from matplotlib.lines import Line2D
tabA = pt.load('table:fig3_panel_a_trbv3_share').copy()
units_order = ['NTN_2_23_1_Th17-1','NTN_2_23_1_Th17-2','NTN_2_23_1_Th17-3','NTN_1_19_23_1','NTN_1_19_23_2','NTN_1_19_5','NTN_1_19_9_1','NTN_1_19_9_2','Colitis_2_23_2_Th17-1','Colitis_2_23_2_Th17-2']
tabA = tabA[tabA.unit.isin(units_order)].copy()
tabA['x'] = tabA.unit.map({u:i for i,u in enumerate(units_order)})
n1 = tabA[tabA.grp=='Tgfbr1 guide 1'].set_index('unit').n
lab = {u: (u.replace('_23_1_','_').replace('_19_','_').replace('Th17-','T')) + f"\nn={int(n1.get(u,0))}" for u in units_order}
tabB = pd.DataFrame({'guide':['gRNA_Tgfbr1_1','gRNA_Tgfbr1_2','gRNA_Tgfbr1_3','gRNA_Tgfbr1_4','gRNA_Tgfbr1_5'],
 'own_gene_log2fc':[-0.0683,-1.3164,-0.8348,-0.6903,0.3945],'cells':[2788,107,527,107,255],
 'measurable':[True,False,True,False,True],'verdict':['transcript unchanged','not measurable','at least 30% down','not measurable','transcript unchanged']})
tabB['y'] = list(range(len(tabB)))[::-1]
pt.save_table('fig3_panel_b_own_gene', tabB)
C_G1, C_OTH, C_CTL = '#B2182B', '#2166AC', '#9E9E9E'
def drawA(p):
    ax = p.add_subplot()
    off = {'Tgfbr1 guide 1':-0.22,'Tgfbr1 other guides':0.0,'controls':0.22}
    col = {'Tgfbr1 guide 1':C_G1,'Tgfbr1 other guides':C_OTH,'controls':C_CTL}
    for g in off:
        for _,r in tabA[tabA.grp==g].iterrows():
            ax.scatter(r.x+off[g], r.share, s=14, facecolor=col[g] if r.n>=10 else 'white', edgecolor=col[g], linewidth=0.8, zorder=3)
    ax.set_xticks(range(len(units_order)))
    ax.set_xticklabels([lab[u] for u in units_order], rotation=90, fontsize=5)
    ax.set_ylim(0,1.05); ax.set_yticks([0,0.25,0.5,0.75,1.0]); ax.tick_params(labelsize=5)
    ax.set_ylabel('Share of single-guide cells\nwith Trbv3 dominant', fontsize=5)
    ax.legend(handles=[Line2D([],[],marker='o',ls='',color=C_G1,ms=3,label='Tgfbr1 guide 1'),
                       Line2D([],[],marker='o',ls='',color=C_OTH,ms=3,label='Tgfbr1 other guides'),
                       Line2D([],[],marker='o',ls='',color=C_CTL,ms=3,label='control guides')],
              fontsize=5, frameon=False, loc='lower left', bbox_to_anchor=(0.0,0.45))
    for s in ['top','right']: ax.spines[s].set_visible(False)
def drawB(p):
    ax = p.add_subplot()
    for _,r in tabB.iterrows():
        c = C_G1 if r.guide.endswith('_1') else C_OTH
        ax.scatter(r.own_gene_log2fc, r.y, s=18, facecolor=c if r.measurable else 'white', edgecolor=c, linewidth=0.8, zorder=3)
        ax.text(0.06, r.y, f"n={int(r.cells)}", fontsize=5, va='center')
    ax.axvline(0, color=C_CTL, ls='--', lw=0.6)
    ax.set_yticks(tabB.y); ax.set_yticklabels([g.replace('gRNA_','') for g in tabB.guide], fontsize=5)
    ax.set_xlabel('Tgfbr1 own gene, log2 FC vs control cells', fontsize=5); ax.tick_params(labelsize=5)
    ax.set_xlim(-1.6,0.9); ax.set_ylim(-0.6,4.6)
    for s in ['top','right']: ax.spines[s].set_visible(False)
rows = [(90, [(0.58, drawA, 'Guide 1 holds the Trbv3-dominant population in NTN2; other guides and controls do not'),
              (0.42, drawB, 'Guide 1 barely lowers its own gene; the other guides do')])]
fig, letters = fsr.figure(rows)
cap = ("Figure 3. a, Share of single-guide cells whose dominant Trbv gene is Trbv3, per capture (10 captures; n = guide 1 cells per capture; "
 "hollow marks fewer than 10 cells). NTN2 captures: guide 1 0.95, 0.96, 0.94 (961, 680, 871 cells); control guides 0.20, 0.26, 0.16; other Tgfbr1 guides 0.02 to 0.05. "
 "b, Tgfbr1 own-gene log2 fold change against control cells per guide (pooled over captures; n = cells per guide; hollow = not measurable by the efficacy verdict): "
 "guide 1 -0.07 (n = 2,788), guide 2 -1.32 (n = 107), guide 3 -0.83 (n = 527), guide 4 -0.69 (n = 107), guide 5 +0.39 (n = 255). Dot = point estimate; no test shown.")
save_figure(fig, 'lead-figure-figure-3-the-tgfbr1-signature-is-a-guide-1-clona',
            data={'panel_a_trbv3_share': tabA, 'panel_b_own_gene': tabB}, caption=cap, role='overview', strict=True)
print('saved', letters)


saved ['a', 'b']

**Redraw figure 3 with an accurate panel a title and caption (colitis other guides)** (15:18, lead-figure-figure-3-the-tgfbr1-signature-is-a-guide-1-clona)

Why: The first title said other guides do not shift; colitis Th17-1 other guides reach 0.46

![20261114T151855684-1.png](figures/20261114T151855684-1.png)

In [ ]:
import numpy as np, pandas as pd
import pertura.question_modules.figure_series as fsr
from matplotlib.lines import Line2D
tabA = pt.load('table:fig3_panel_a_trbv3_share').copy()
units_order = ['NTN_2_23_1_Th17-1','NTN_2_23_1_Th17-2','NTN_2_23_1_Th17-3','NTN_1_19_23_1','NTN_1_19_23_2','NTN_1_19_5','NTN_1_19_9_1','NTN_1_19_9_2','Colitis_2_23_2_Th17-1','Colitis_2_23_2_Th17-2']
tabA = tabA[tabA.unit.isin(units_order)].copy()
tabA['x'] = tabA.unit.map({u:i for i,u in enumerate(units_order)})
n1 = tabA[tabA.grp=='Tgfbr1 guide 1'].set_index('unit').n
lab = {u: (u.replace('_23_1_','_').replace('_19_','_').replace('Th17-','T')) + f"\nn={int(n1.get(u,0))}" for u in units_order}
tabB = pt.load('table:fig3_panel_b_own_gene').copy()
C_G1, C_OTH, C_CTL = '#B2182B', '#2166AC', '#9E9E9E'
def drawA(p):
    ax = p.add_subplot()
    off = {'Tgfbr1 guide 1':-0.22,'Tgfbr1 other guides':0.0,'controls':0.22}
    col = {'Tgfbr1 guide 1':C_G1,'Tgfbr1 other guides':C_OTH,'controls':C_CTL}
    for g in off:
        for _,r in tabA[tabA.grp==g].iterrows():
            ax.scatter(r.x+off[g], r.share, s=14, facecolor=col[g] if r.n>=10 else 'white', edgecolor=col[g], linewidth=0.8, zorder=3)
    ax.set_xticks(range(len(units_order)))
    ax.set_xticklabels([lab[u] for u in units_order], rotation=90, fontsize=5)
    ax.set_ylim(0,1.05); ax.set_yticks([0,0.25,0.5,0.75,1.0]); ax.tick_params(labelsize=5)
    ax.set_ylabel('Share of single-guide cells\nwith Trbv3 dominant', fontsize=5)
    ax.legend(handles=[Line2D([],[],marker='o',ls='',color=C_G1,ms=3,label='Tgfbr1 guide 1'),
                       Line2D([],[],marker='o',ls='',color=C_OTH,ms=3,label='Tgfbr1 other guides'),
                       Line2D([],[],marker='o',ls='',color=C_CTL,ms=3,label='control guides')],
              fontsize=5, frameon=False, loc='lower left', bbox_to_anchor=(0.0,0.45))
    for s in ['top','right']: ax.spines[s].set_visible(False)
def drawB(p):
    ax = p.add_subplot()
    for _,r in tabB.iterrows():
        c = C_G1 if r.guide.endswith('_1') else C_OTH
        ax.scatter(r.own_gene_log2fc, r.y, s=18, facecolor=c if r.measurable else 'white', edgecolor=c, linewidth=0.8, zorder=3)
        ax.text(0.06, r.y, f"n={int(r.cells)}", fontsize=5, va='center')
    ax.axvline(0, color=C_CTL, ls='--', lw=0.6)
    ax.set_yticks(tabB.y); ax.set_yticklabels([g.replace('gRNA_','') for g in tabB.guide], fontsize=5)
    ax.set_xlabel('Tgfbr1 own gene, log2 FC vs control cells', fontsize=5); ax.tick_params(labelsize=5)
    ax.set_xlim(-1.6,0.9); ax.set_ylim(-0.6,4.6)
    for s in ['top','right']: ax.spines[s].set_visible(False)
rows = [(90, [(0.58, drawA, 'Guide 1 holds the Trbv3-dominant population in the NTN2 captures; other guides and controls sit low there'),
              (0.42, drawB, 'Guide 1 barely lowers its own gene; guides 2, 3, 4 lower it, 5 raises it')])]
fig, letters = fsr.figure(rows)
cap = ("Figure 3. a, Share of single-guide cells whose dominant Trbv gene is Trbv3, per capture (10 captures; n = guide 1 cells per capture; "
 "hollow marks fewer than 10 cells). NTN2 captures: guide 1 0.95, 0.96, 0.94 (961, 680, 871 cells); control guides 0.20, 0.26, 0.16; other Tgfbr1 guides 0.02 to 0.05. "
 "In colitis Th17-1 other Tgfbr1 guides reach 0.46 (110 cells) against controls 0.04, so the Trbv3 shift is not confined to guide 1 in that capture (small n). "
 "b, Tgfbr1 own-gene log2 fold change against control cells per guide (pooled over captures; n = cells per guide; hollow = not measurable by the efficacy verdict): "
 "guide 1 -0.07 (n = 2,788), guide 2 -1.32 (n = 107), guide 3 -0.83 (n = 527), guide 4 -0.69 (n = 107), guide 5 +0.39 (n = 255). Dot = point estimate; no test shown.")
save_figure(fig, 'lead-figure-figure-3-the-tgfbr1-signature-is-a-guide-1-clona',
            data={'panel_a_trbv3_share': tabA, 'panel_b_own_gene': tabB}, caption=cap, role='overview', strict=True)
print('saved', letters)


saved ['a', 'b']